# Modelo Bias-Aware para detección de contenido político en Twitter

## Preparación del entorno `tweetsCYG`
### Instalación Python 3.12.14 (recomendado)
Ejecutar desde la carpeta del proyecto.

```powershell
# 1) Ver las versiones actuales. Python 3.14 puede permanecer instalado.
python --version
py --list 2>$null

# 2) Instalar uv si todavía no está instalado.
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"

# Cierre y abra PowerShell nuevamente si 'uv' todavía no aparece en PATH.
uv --version

# 3) Instalar Python 3.12.14 en paralelo, sin desinstalar ninguna otra versión.
uv python install 3.12.14

# 4) Verificar y localizar exactamente Python 3.12.14.
uv python list
$PY312 = uv python find 3.12.14
& $PY312 --version

# 5) Crear el entorno virtual del proyecto con Python 3.12.14.
#    El entorno virtual y el kernel se llamarán tweetsCYG.
& $PY312 -m venv tweetsCYG

# 6) Activar el entorno.
Set-ExecutionPolicy -Scope Process -ExecutionPolicy Bypass 
.\tweetsCYG\Scripts\Activate.ps1

# 7) Confirmar que el entorno usa Python 3.12.14.
python --version
python -c "import sys; print(sys.executable)"

# 8) Actualizar herramientas base.
python -m pip install --upgrade pip setuptools wheel

# 9) Instalar el stack científico y de notebook.
python -m pip install `
  numpy `
  pandas `
  scipy `
  scikit-learn `
  gensim `
  bokeh `
  openpyxl `
  transformers `
  sentencepiece `
  psutil `
  tqdm `
  ipykernel `
  jupyterlab `
  spacy `
  ipywidgets

# Luego instala 
python -m spacy download es_core_news_sm

# 10) Instalar el modelo lingüístico de español utilizado para la limpieza.
python -m spacy download es_core_news_sm
python -m spacy validate

# 11) Instalar PyTorch.
#     Si existe una GPU NVIDIA, se instala una build CUDA.
#     Si no existe, se instala la build CPU.
if (Get-Command nvidia-smi -ErrorAction SilentlyContinue) {
    nvidia-smi
    python -m pip install torch==2.11.0 --index-url https://download.pytorch.org/whl/cu126
} else {
    python -m pip install torch==2.11.0 --index-url https://download.pytorch.org/whl/cpu
}

# 12) Registrar el entorno como kernel de Jupyter con nombre tweetsCYG.
python -m ipykernel install --user --name tweetsCYG --display-name "tweetsCYG"

# 13) Guardar las versiones realmente instaladas para reproducibilidad.
python -m pip freeze > requirements_tweetsCYG.txt
```

# 1. Librerías, reproducibilidad y hardware

In [286]:
from __future__ import annotations

import gc
import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import os
import platform
import random
import re
import sys
import time
import warnings
from collections import Counter
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import numpy as np
import pandas as pd
import psutil
import spacy
from gensim.models import FastText, Word2Vec
from IPython.display import display
from scipy.stats import binomtest, t as student_t, wilcoxon
from spacy.language import Language
from spacy.tokens import Doc
from sklearn.cluster import HDBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    calinski_harabasz_score,
    cohen_kappa_score,
    confusion_matrix,
    davies_bouldin_score,
    f1_score,
    fbeta_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    silhouette_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

from bokeh.io import output_notebook, show
from bokeh.layouts import column, row
from bokeh.models import (
    ColumnDataSource,
    DataTable,
    Div,
    HoverTool,
    PanTool,
    WheelZoomTool,
    BoxZoomTool,
    ResetTool,
    SaveTool,
    HelpTool,
    TabPanel,
    TableColumn,
    Tabs,
    LinearColorMapper,
    ColorBar,
    BasicTicker,
    LabelSet,
)
from bokeh.palettes import Category10, Category20, Turbo256, Blues9
from bokeh.plotting import figure, show
from bokeh.transform import dodge

import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

output_notebook(hide_banner=True)

In [2]:
warnings.filterwarnings("ignore")
random.seed(5)
np.random.seed(5)
torch.manual_seed(5)

In [3]:
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(5)
    torch.set_float32_matmul_precision("high")

In [4]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

In [5]:
def mostrar_tabla_bokeh(df: pd.DataFrame, titulo: str, ancho: int = 950, alto: int = 280, max_filas: int = 25) -> None:
    """
    Muestra un DataFrame como tabla interactiva de Bokeh dentro del notebook.

    Convierte las columnas del DataFrame en una fuente de datos Bokeh y crea una
    tabla desplazable. Se limita el número de filas visibles para mantener el notebook
    legible sin alterar el DataFrame original.

    Args:
    df (pd.DataFrame): Tabla que se desea visualizar.
    titulo (str): Título que se mostrará antes de la tabla.
    ancho (int): Ancho de la tabla en píxeles.
    *Valor por defecto: 950.*
    alto (int): Alto de la tabla en píxeles.
    *Valor por defecto: 280.*
    max_filas (int): Número máximo de filas que se muestran.
    *Valor por defecto: 25.*

    Returns:
    None:
    La función muestra componentes Bokeh en la salida de la celda.

    Example:
    >>> mostrar_tabla_bokeh(pd.DataFrame({"A": [1, 2]}), "Ejemplo")
    """
    vista = df.head(max_filas).copy()
    for col in vista.columns:
        if pd.api.types.is_object_dtype(vista[col]) or pd.api.types.is_string_dtype(vista[col]):
            vista[col] = vista[col].astype(str)
    source = ColumnDataSource(vista)
    columnas = [TableColumn(field=str(col), title=str(col)) for col in vista.columns]
    tabla = DataTable(source=source, columns=columnas, width=ancho, height=alto, index_position=None)
    show(column(Div(text=f"<h3>{titulo}</h3>"), tabla))

In [6]:
def auditar_librerias_reproducibilidad() -> pd.DataFrame:
    """
    Audita las versiones de Python y de las dependencias críticas del notebook.

    La función consulta las distribuciones instaladas y construye una tabla reproducible
    con la versión recomendada para este protocolo, la versión realmente instalada y
    un estado básico de disponibilidad.

    Args:
    No recibe parámetros.

    Returns:
    pd.DataFrame:
    Tabla con Python, librerías requeridas, versiones recomendadas e instaladas.

    Example:
    >>> tabla = auditar_librerias_reproducibilidad()
    >>> "Instalada" in tabla.columns
    True
    """
    requeridas = {
        "Python": ("python", "3.12.14"),
        "NumPy": ("numpy", "1.26.4"),
        "pandas": ("pandas", "2.2.3"),
        "SciPy": ("scipy", "1.13.1"),
        "scikit-learn": ("scikit-learn", "1.6.1"),
        "Gensim": ("gensim", "4.3.3"),
        "Bokeh": ("bokeh", "3.6.3"),
        "openpyxl": ("openpyxl", "3.1.5"),
        "PyTorch": ("torch", "2.11.0"),
        "Transformers": ("transformers", "4.49.0"),
        "SentencePiece": ("sentencepiece", "0.2.0"),
        "psutil": ("psutil", "7.0.0"),
        "tqdm": ("tqdm", "4.67.1"),
        "ipykernel": ("ipykernel", "6.29.5"),
        "spaCy": ("spacy", "3.8.7"),
        "spaCy español": ("es_core_news_sm", "3.8.0"),
    }

    filas: list[dict[str, str]] = []
    for nombre, (distribucion, recomendada) in requeridas.items():
        if nombre == "Python":
            instalada = platform.python_version()
        else:
            try:
                instalada = importlib_metadata.version(distribucion)
            except importlib_metadata.PackageNotFoundError:
                instalada = "NO INSTALADA"
        estado = "OK" if instalada != "NO INSTALADA" else "FALTA"
        if nombre == "Python" and instalada != recomendada:
            estado = "REVISAR"
        filas.append({
            "Componente": nombre,
            "Distribucion": distribucion,
            "Recomendada": recomendada,
            "Instalada": instalada,
            "Estado": estado,
        })
    return pd.DataFrame(filas)

In [7]:
def auditar_hardware_reproducibilidad() -> pd.DataFrame:
    """
    Audita CPU, memoria y aceleradores disponibles para el pipeline.

    Detecta CUDA, Apple MPS y CPU. Esta información permite documentar el hardware
    empleado y decidir automáticamente dónde ejecutar los modelos Transformer.

    Args:
    No recibe parámetros.

    Returns:
    pd.DataFrame:
    Tabla con sistema operativo, CPU, RAM, dispositivo PyTorch, GPU y versión CUDA.

    Example:
    >>> hardware = auditar_hardware_reproducibilidad()
    >>> "Valor" in hardware.columns
    True
    """
    if torch.cuda.is_available():
        dispositivo = "cuda"
        gpu = torch.cuda.get_device_name(0)
        cuda = str(torch.version.cuda)
        memoria_gpu = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        memoria_gpu_txt = f"{memoria_gpu:.2f} GB"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        dispositivo = "mps"
        gpu = "Apple Metal Performance Shaders"
        cuda = "No aplica"
        memoria_gpu_txt = "Memoria unificada"
    else:
        dispositivo = "cpu"
        gpu = "No disponible"
        cuda = "No aplica"
        memoria_gpu_txt = "No aplica"

    filas = [
        {"Indicador": "Sistema operativo", "Valor": platform.platform()},
        {"Indicador": "Python", "Valor": platform.python_version()},
        {"Indicador": "Ejecutable Python", "Valor": sys.executable},
        {"Indicador": "CPU lógicos", "Valor": str(os.cpu_count() or 1)},
        {"Indicador": "RAM total", "Valor": f"{psutil.virtual_memory().total / (1024 ** 3):.2f} GB"},
        {"Indicador": "Dispositivo PyTorch", "Valor": dispositivo},
        {"Indicador": "GPU", "Valor": gpu},
        {"Indicador": "Memoria GPU", "Valor": memoria_gpu_txt},
        {"Indicador": "CUDA de PyTorch", "Valor": cuda},
        {"Indicador": "PyTorch", "Valor": torch.__version__},
    ]
    return pd.DataFrame(filas)

In [8]:
def seleccionar_dispositivo_torch() -> torch.device:
    """
    Selecciona automáticamente el mejor dispositivo disponible para Transformers.

    Prioriza CUDA cuando existe una GPU NVIDIA compatible, luego Apple MPS y finalmente
    CPU. El resto del pipeline clásico continúa ejecutándose con NumPy/scikit-learn.

    Args:
    No recibe parámetros.

    Returns:
    torch.device:
    Dispositivo que se utilizará para inferencia y fine-tuning Transformer.

    Example:
    >>> dispositivo = seleccionar_dispositivo_torch()
    >>> dispositivo.type in {"cuda", "mps", "cpu"}
    True
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

In [9]:
def guardar_requirements_runtime(tabla_librerias: pd.DataFrame, ruta: Path) -> Path:
    """
    Guarda un archivo de versiones efectivamente observadas durante la ejecución.

    El archivo complementa `pip freeze` y permite asociar las versiones críticas
    directamente con los resultados generados por este notebook.

    Args:
    tabla_librerias (pd.DataFrame): Auditoría producida por `auditar_librerias_reproducibilidad`.
    ruta (Path): Archivo de texto de salida.

    Returns:
    Path:
    Ruta del archivo generado.

    Example:
    >>> ruta = guardar_requirements_runtime(tabla_librerias, Path("requirements_runtime.txt"))
    """
    lineas: list[str] = []
    for _, fila in tabla_librerias.iterrows():
        if fila["Componente"] == "Python":
            lineas.append(f"# Python=={fila['Instalada']}")
        elif fila["Instalada"] != "NO INSTALADA":
            lineas.append(f"{fila['Distribucion']}=={fila['Instalada']}")
    ruta.write_text("\n".join(lineas) + "\n", encoding="utf-8")
    return ruta

In [10]:
AUDITORIA_LIBRERIAS = auditar_librerias_reproducibilidad()
AUDITORIA_HARDWARE = auditar_hardware_reproducibilidad()
DISPOSITIVO_TORCH = seleccionar_dispositivo_torch()

In [11]:
mostrar_tabla_bokeh(
    AUDITORIA_LIBRERIAS,
    "Auditoría de librerías y versiones para reproducibilidad",
    ancho=1000,
    alto=460,
    max_filas=30,
)

In [12]:
mostrar_tabla_bokeh(
    AUDITORIA_HARDWARE,
    "Auditoría de hardware y dispositivo de cómputo",
    ancho=1050,
    alto=300,
    max_filas=20,
)

In [13]:
print(f"Dispositivo seleccionado para Transformers: {DISPOSITIVO_TORCH}")
if platform.python_version() != "3.12.14":
    print(
        "ADVERTENCIA: este protocolo fue preparado para Python 3.12.14."
        f"Versión actual: {platform.python_version()}."
    )

Dispositivo seleccionado para Transformers: cpu


# 2. Configuración experimental

In [14]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs_bias_aware"
REVISION_OUTPUT_DIR = OUTPUT_DIR / "revisores_final"

In [15]:
ARCHIVOS = {
    "tweets": "tweets_consolidado_limpio_final.xlsx",
    "trends": "trends_limpio_final.xlsx",
    "resumen": "resumen_preprocesamiento.xlsx",
    "paro": "paro_nacional_limpio_final.xlsx",
    "muerte": "muerte_cruzada_limpio_final.xlsx",
}

In [16]:
RANDOM_STATE = 5
DIMENSIONES = (50, 100, 200)
TEST_SIZE = 0.20

In [17]:
W2V_WINDOW = 5
W2V_MIN_COUNT = 1
W2V_EPOCHS = 15
W2V_WORKERS = 1

In [18]:
K_CANDIDATOS = tuple(range(2, 11))
KNN_VECINOS = 5

In [19]:
EXPANSION_TOP_N_FINAL = 100
EXPANSION_TOP_N_CANDIDATOS = (25, 50, 100, 200)

In [20]:
HDBSCAN_MIN_CLUSTER_SIZE = 20
HDBSCAN_MIN_SAMPLES = 5

In [21]:
CV_FOLDS = 5
CV_REPEATS = 2
CV_EPOCHS = 10
METRICAS_ESTADISTICAS = (
    "Exactitud",
    "Precisión",
    "Recall",
    "F1-score",
    "Valor F2",
    "Valor F0.5",
)

In [22]:
N_ANOTACION_MANUAL = 600
RUTA_ANOTACION_MANUAL = DATA_DIR / "anotacion_manual_politica.xlsx"

In [23]:
N_MANUAL_DESACUERDOS_HASHTAG = 250
RUTA_ANOTACION_DESACUERDOS_HASHTAG = DATA_DIR / "anotacion_desacuerdos_hashtag.xlsx"

In [24]:
DIMENSION_BENCHMARK = 100
FASTTEXT_EPOCHS = 15
TRANSFORMER_MAX_LENGTH = 128
TRANSFORMER_BATCH_SIZE = 32 if DISPOSITIVO_TORCH.type == "cuda" else 8
TRANSFORMER_FINE_TUNE_EPOCHS = 2
TRANSFORMER_FINE_TUNE_LR = 2e-5
EJECUTAR_FINE_TUNING_TRANSFORMER = DISPOSITIVO_TORCH.type in {"cuda", "mps"}

In [25]:
MODELOS_TRANSFORMER = {
    "BETO": "dccuchile/bert-base-spanish-wwm-uncased",
    "XLM-RoBERTa": "xlm-roberta-base",
}

In [26]:
MODELO_SPACY_ES = "es_core_news_sm"
SPACY_BATCH_SIZE = 512

LONGITUD_MINIMA_TOKEN = 3
PALABRAS_INCLUIDAS = {"ec"}
PALABRAS_EXCLUIDAS = {"jajaja"}
POS_EXCLUIDOS = {"VERB", "AUX", "DET"}

CONSERVAR_HASHTAGS_MODELO = True
CONSERVAR_MENCIONES_MODELO = True

VERSION_PROTOCOLO_ANOTACION = "2026-09-limpieza-anotaciones-v1"

In [27]:
MAX_NGRAM_TREND = 4

In [28]:
SEMILLAS_POLITICAS_EXTRA = [
    "politica", "politico", "presidente", "gobierno", "asamblea",
    "elecciones", "paro", "nacional", "muerte", "cruzada", "lasso",
    "guillermo", "correa", "iza", "protesta", "@asambleaecuador",
    "#paronacional", "ecuador",
]

In [29]:
HASHTAGS_POLITICOS_ESTRICTOS = {
    "#paronacionalec", "#paronacional", "#paronacionalecuador", "#paroecuador",
    "#muertecruzada", "#muertecruzadaya", "#muertecruzadaec",
    "#juiciopolitico", "#juiciopoliticoalasso", "#asambleanacional",
    "#lassoseva", "#guillermolasso", "#fueralassofuera", "#lassodestruyoecuador",
    "#juiciosinpruebas", "#conaie", "#izaterrorista", "#eleccionesec", "#cne",
    "#lassorenuncia", "#lassoasesino", "#ecuadorquierepaz", "#leonidasiza",
}

In [30]:
HASHTAGS_NO_POLITICOS_ESTRICTOS = {
    "#championsleague", "#emelec", "#ligapro", "#ligaprobet", "#copaecuadorecuabet",
    "#premierleague", "#realmadrid", "#laacademia", "#domingodeacademia",
    "#diadelpadre", "#felizdiadelpadre", "#tiktok", "#seguros", "#vida",
    "#portuseguridad", "#diadelamujermaritima", "#webinar", "#ecdfmundialista",
}

In [31]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REVISION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [253]:
RUTA_REQUIREMENTS_RUNTIME = guardar_requirements_runtime(
    AUDITORIA_LIBRERIAS,
    REVISION_OUTPUT_DIR / "requirements_runtime.txt",
)

In [254]:
print(f"Datos esperados en: {DATA_DIR}")
print(f"Resultados en: {REVISION_OUTPUT_DIR}")
print(f"Requirements runtime: {RUTA_REQUIREMENTS_RUNTIME}")
print(f"Dispositivo Transformer: {DISPOSITIVO_TORCH}")
print(f"Fine-tuning Transformer habilitado: {EJECUTAR_FINE_TUNING_TRANSFORMER}")

Datos esperados en: C:\Users\LABIA\Documents\Tesis cleo EPN\Codigo\data
Resultados en: C:\Users\LABIA\Documents\Tesis cleo EPN\Codigo\outputs_bias_aware\revisores_final
Requirements runtime: C:\Users\LABIA\Documents\Tesis cleo EPN\Codigo\outputs_bias_aware\revisores_final\requirements_runtime.txt
Dispositivo Transformer: cpu
Fine-tuning Transformer habilitado: False


# 3. Carga, limpieza y auditoría del conjunto de datos

In [34]:
def resolver_ruta_dato(nombre_archivo: str, data_dir: Path) -> Path:
    """
    Resuelve la ruta de un archivo de datos requerido por el notebook.

    La función busca primero el archivo dentro de la carpeta `data`, que es la
    estructura esperada para este proyecto. Como respaldo, también comprueba el
    directorio actual para facilitar pruebas controladas sin modificar el código.

    Args:
    nombre_archivo (str): Nombre exacto del archivo que se desea localizar.
    data_dir (Path): Carpeta principal donde deben encontrarse los datos.

    Returns:
    Path:
    Ruta existente del archivo solicitado.

    Example:
    >>> resolver_ruta_dato("tweets.xlsx", Path("data"))
    Path('data/tweets.xlsx')
    """
    candidatos = [data_dir / nombre_archivo, Path.cwd() / nombre_archivo]
    for ruta in candidatos:
        if ruta.exists():
            return ruta
    raise FileNotFoundError(
        f"No se encontró '{nombre_archivo}'. Se buscó en: "
        + ", ".join(str(r) for r in candidatos)
    )

In [35]:
def verificar_archivos(archivos: Mapping[str, str], data_dir: Path) -> dict[str, Path]:
    """
    Verifica que todos los archivos requeridos estén disponibles.

    Recorre el diccionario de archivos del proyecto, resuelve cada ruta y devuelve
    un diccionario listo para ser utilizado por las funciones de carga. Si falta
    algún archivo, la ejecución se detiene con un mensaje explícito.

    Args:
    archivos (Mapping[str, str]): Claves lógicas y nombres de los archivos requeridos.
    data_dir (Path): Carpeta donde se espera encontrar los archivos.

    Returns:
    dict[str, Path]:
    Diccionario con las mismas claves y las rutas validadas de cada archivo.

    Example:
    >>> verificar_archivos({"tweets": "tweets.xlsx"}, Path("data"))
    {'tweets': Path('data/tweets.xlsx')}
    """
    rutas: dict[str, Path] = {}
    for clave, nombre in archivos.items():
        rutas[clave] = resolver_ruta_dato(nombre, data_dir)
    return rutas

In [36]:
def cargar_datasets(rutas: Mapping[str, Path]) -> dict[str, Any]:
    """
    Carga todos los datasets limpios y el resumen de preprocesamiento.

    Lee los cuatro archivos de tweets/trends y todas las hojas del archivo de
    resumen. El resultado centraliza los DataFrames para que el resto del notebook
    trabaje con una única estructura de datos.

    Args:
    rutas (Mapping[str, Path]): Diccionario con las rutas validadas de los archivos.

    Returns:
    dict[str, Any]:
    Diccionario con DataFrames de tweets, trends, eventos y hojas del resumen.

    Example:
    >>> datos = cargar_datasets(rutas)
    >>> datos["tweets"].shape[0] > 0
    True
    """
    resumen = pd.read_excel(rutas["resumen"], sheet_name=None)
    return {
        "tweets": pd.read_excel(rutas["tweets"]),
        "trends": pd.read_excel(rutas["trends"]),
        "paro": pd.read_excel(rutas["paro"]),
        "muerte": pd.read_excel(rutas["muerte"]),
        "resumen": resumen,
    }

In [37]:
def convertir_tokens(valor: Any) -> list[str]:
    """
    Convierte una representación preprocesada de tokens en una lista de palabras.

    Los archivos entregados almacenan los tokens como texto separado por espacios.
    Esta función normaliza valores nulos, listas existentes y cadenas para producir
    una estructura uniforme que pueda ser consumida por Word2Vec y el resto del flujo.

    Args:
    valor (Any): Valor original de la columna de tokens.

    Returns:
    list[str]:
    Lista limpia de tokens no vacíos.

    Example:
    >>> convertir_tokens("ecuador gobierno asamblea")
    ['ecuador', 'gobierno', 'asamblea']
    """
    if isinstance(valor, list):
        return [str(x).strip() for x in valor if str(x).strip()]
    if pd.isna(valor):
        return []
    return [token for token in str(valor).strip().split() if token]

In [38]:
def preparar_tokens_dataframe(df: pd.DataFrame, columna_tokens: str = "tokens_sin_stopwords") -> list[list[str]]:
    """
    Extrae del DataFrame el corpus tokenizado que utilizará el modelado semántico.

    Valida la existencia de la columna indicada y convierte cada registro a una lista
    de tokens. Los documentos vacíos se conservan como listas vacías para mantener la
    alineación con las filas originales del DataFrame.

    Args:
    df (pd.DataFrame): Dataset que contiene los documentos preprocesados.
    columna_tokens (str): Nombre de la columna que contiene los tokens.
    *Valor por defecto: "tokens_sin_stopwords".*

    Returns:
    list[list[str]]:
    Corpus tokenizado, con una lista de tokens por cada fila del DataFrame.

    Example:
    >>> preparar_tokens_dataframe(pd.DataFrame({"tokens_sin_stopwords": ["a b", "c"]}))
    [['a', 'b'], ['c']]
    """
    if columna_tokens not in df.columns:
        raise KeyError(f"La columna '{columna_tokens}' no existe. Columnas: {list(df.columns)}")
    return [convertir_tokens(valor) for valor in df[columna_tokens].tolist()]

In [39]:
def validar_consistencia_eventos(tweets: pd.DataFrame, paro: pd.DataFrame, muerte: pd.DataFrame) -> pd.DataFrame:
    """
    Compara el consolidado con los archivos separados de Paro Nacional y Muerte Cruzada.

    La función contrasta la cantidad de filas por evento del archivo consolidado con
    las cantidades de los archivos individuales. Este control ayuda a detectar pérdidas
    o duplicaciones accidentales antes de iniciar el modelado.

    Args:
    tweets (pd.DataFrame): Dataset consolidado de tweets limpios.
    paro (pd.DataFrame): Dataset limpio del Paro Nacional.
    muerte (pd.DataFrame): Dataset limpio de Muerte Cruzada.

    Returns:
    pd.DataFrame:
    Tabla de comparación con conteos y diferencias por evento.

    Example:
    >>> validar_consistencia_eventos(tweets, paro, muerte).columns.tolist()
    ['Evento', 'Consolidado', 'Archivo_separado', 'Diferencia']
    """
    conteos = tweets["evento"].astype(str).str.strip().value_counts()
    filas = []
    for evento, separado in [("Paro Nacional", paro), ("Muerte Cruzada", muerte)]:
        consolidado = int(conteos.get(evento, 0))
        separado_n = int(len(separado))
        filas.append({
            "Evento": evento,
            "Consolidado": consolidado,
            "Archivo_separado": separado_n,
            "Diferencia": consolidado - separado_n,
        })
    return pd.DataFrame(filas)

In [40]:
def crear_id_registro(df: pd.DataFrame) -> pd.Series:
    """
    Crea un identificador estable a partir del evento y del texto fuente antes de la limpieza.

    Si existe `texto_id_fuente`, esa columna tiene prioridad. Esto evita que el
    identificador cambie cuando la nueva limpieza elimina stopwords, verbos o tokens
    cortos. Como respaldo usa `texto_limpio_fuente` y finalmente `texto_limpio`.

    Args:
    df (pd.DataFrame): DataFrame con `evento` y alguna columna textual válida.

    Returns:
    pd.Series:
    Identificadores SHA-256 truncados a 20 caracteres.

    Example:
    >>> ids = crear_id_registro(tweets.head(2))
    >>> len(ids)
    2
    """
    if "evento" not in df.columns:
        raise KeyError("Falta la columna `evento` para construir identificadores estables.")

    candidatas = [
        "texto_id_fuente",
        "texto_limpio_fuente",
        "texto_limpio",
        "texto_original",
        "texto",
    ]
    columna = next((c for c in candidatas if c in df.columns), None)
    if columna is None:
        raise KeyError("No existe una columna textual utilizable para construir `id_registro`.")

    claves = (
        df["evento"].fillna("").astype(str).str.strip()
        + "||"
        + df[columna].fillna("").astype(str).str.strip()
    )
    return claves.map(
        lambda valor: hashlib.sha256(valor.encode("utf-8")).hexdigest()[:20]
    )

In [41]:
def asignar_ids_estables_pre_limpieza(df: pd.DataFrame) -> pd.DataFrame:
    """
    Preserva el texto fuente y crea identificadores reproducibles antes de la limpieza.

    La función debe ejecutarse inmediatamente después de cargar el consolidado y antes de aplicar la limpieza. Guarda
    una copia de `texto_limpio` en `texto_id_fuente` y genera `id_registro` antes de
    modificar las features textuales.

    Args:
    df (pd.DataFrame): Dataset consolidado recién cargado.

    Returns:
    pd.DataFrame:
    Copia del dataset con `texto_id_fuente` e `id_registro`.

    Example:
    >>> preparado = asignar_ids_estables_pre_limpieza(tweets)
    >>> "id_registro" in preparado.columns
    True
    """
    trabajo = df.copy()
    if "texto_limpio" not in trabajo.columns:
        raise KeyError("Se requiere `texto_limpio` para estabilizar los IDs antes de limpiar.")
    trabajo["texto_id_fuente"] = (
        trabajo["texto_limpio"].fillna("").astype(str).str.strip()
    )
    trabajo["id_registro"] = crear_id_registro(trabajo)
    return trabajo

In [42]:
def resolver_columna_texto_revision(
    df: pd.DataFrame,
    candidatas: Sequence[str],
) -> str:
    """
    Selecciona la primera columna textual disponible entre varias candidatas.

    Permite aplicar la misma limpieza a tweets y trends aunque sus archivos utilicen
    nombres de columnas diferentes.

    Args:
    df (pd.DataFrame): Dataset que contiene texto.
    candidatas (Sequence[str]): Columnas ordenadas por prioridad.

    Returns:
    str:
    Nombre de la primera columna existente.

    Example:
    >>> resolver_columna_texto_revision(pd.DataFrame({"texto": ["hola"]}), ["texto_original", "texto"])
    'texto'
    """
    for columna in candidatas:
        if columna in df.columns:
            return str(columna)
    raise KeyError(
        "No se encontró ninguna columna textual. "
        f"Candidatas evaluadas: {list(candidatas)}"
    )

In [43]:
def cargar_modelo_spacy_espanol(nombre_modelo: str) -> Language:
    """
    Carga el modelo lingüístico español usado para POS tagging y stopwords.

    Desactiva parser y NER porque no son necesarios. Mantiene los componentes que
    permiten obtener POS, morfología y lemas. Si el modelo no está instalado, muestra
    el comando exacto requerido.

    Args:
    nombre_modelo (str): Nombre del paquete spaCy de español.

    Returns:
    Language:
    Pipeline spaCy listo para procesar el corpus.

    Example:
    >>> nlp_es = cargar_modelo_spacy_espanol("es_core_news_sm")
    >>> nlp_es.lang
    'es'
    """
    try:
        return spacy.load(nombre_modelo, disable=["parser", "ner"])
    except OSError as exc:
        raise RuntimeError(
            f"No está instalado `{nombre_modelo}`. Active tweetsCYG y ejecute: "
            f"`python -m spacy download {nombre_modelo}`; luego reinicie el kernel."
        ) from exc

In [44]:
def proteger_entidades_sociales_revision(texto: str) -> str:
    """
    Protege hashtags y menciones antes de tokenizar con spaCy.

    Convierte temporalmente `#termino` y `@usuario` en tokens alfabéticos con prefijos
    internos. Después del análisis lingüístico se restauran sus símbolos originales.

    Args:
    texto (str): Texto original que puede contener hashtags o menciones.

    Returns:
    str:
    Texto con marcadores temporales compatibles con la tokenización de spaCy.

    Example:
    >>> proteger_entidades_sociales_revision("#Paro @usuario")
    ' hashtagzzParo   mentionzzusuario '
    """
    valor = str(texto)
    valor = re.sub(
        r"(?<!\w)#([\wáéíóúüñÁÉÍÓÚÜÑ]+)",
        r" hashtagzz\1 ",
        valor,
    )
    valor = re.sub(
        r"(?<!\w)@([\wáéíóúüñÁÉÍÓÚÜÑ]+)",
        r" mentionzz\1 ",
        valor,
    )
    return valor

In [45]:
def filtrar_documento_spacy_revision(
    doc: Doc,
    inclusiones: set[str],
    exclusiones: set[str],
    pos_excluidos: set[str],
    longitud_minima: int,
    conservar_hashtags: bool,
    conservar_menciones: bool,
) -> tuple[list[str], dict[str, int]]:
    """
    Filtra un documento spaCy aplicando las reglas finales del experimento.

    Las exclusiones explícitas tienen prioridad absoluta. Las inclusiones explícitas
    tienen prioridad sobre longitud, stopwords y POS. Para el resto se eliminan
    stopwords, VERB/AUX/DET, URLs, correos, ruido no alfabético y tokens demasiado
    cortos.

    Args:
    doc (Doc): Documento procesado por spaCy.
    inclusiones (set[str]): Lista blanca de términos que siempre se conservan.
    exclusiones (set[str]): Lista negra de términos que siempre se eliminan.
    pos_excluidos (set[str]): Etiquetas POS que deben eliminarse.
    longitud_minima (int): Longitud mínima aceptada para palabras normales.
    conservar_hashtags (bool): Indica si los hashtags deben permanecer como features.
    conservar_menciones (bool): Indica si las menciones deben permanecer como features.

    Returns:
    tuple[list[str], dict[str, int]]:
    Tokens conservados y conteos por razón de eliminación.

    Example:
    >>> tokens, conteos = filtrar_documento_spacy_revision(
    ...     nlp_es("ec gobierno corre"), {"ec"}, {"jajaja"}, {"VERB", "AUX", "DET"}, 3, True, True
    ... )
    """
    incl = {str(x).strip().lower() for x in inclusiones}
    excl = {str(x).strip().lower() for x in exclusiones}
    pos_bloqueados = {str(x).strip().upper() for x in pos_excluidos}

    conteos: Counter[str] = Counter()
    salida: list[str] = []

    for token in doc:
        bruto = token.text.strip().lower()
        if not bruto or token.is_space:
            continue

        es_hashtag = bruto.startswith("hashtagzz")
        es_mencion = bruto.startswith("mentionzz")

        if es_hashtag:
            nucleo = bruto[len("hashtagzz"):].strip("_")
            normal = f"#{nucleo}" if nucleo else ""
            clave_regla = nucleo
        elif es_mencion:
            nucleo = bruto[len("mentionzz"):].strip("_")
            normal = f"@{nucleo}" if nucleo else ""
            clave_regla = nucleo
        else:
            normal = bruto
            clave_regla = normal

        conteos["tokens_evaluados"] += 1

        if not normal:
            conteos["eliminados_ruido"] += 1
            continue

        if clave_regla in excl or normal in excl:
            conteos["eliminados_lista_negra"] += 1
            continue

        if clave_regla in incl or normal in incl:
            salida.append(normal)
            conteos["conservados_lista_blanca"] += 1
            conteos["tokens_conservados"] += 1
            continue

        if es_hashtag:
            if not conservar_hashtags:
                conteos["eliminados_hashtag"] += 1
                continue
            if len(clave_regla) < int(longitud_minima):
                conteos["eliminados_longitud"] += 1
                continue
            salida.append(normal)
            conteos["tokens_conservados"] += 1
            continue

        if es_mencion:
            if not conservar_menciones:
                conteos["eliminados_mencion"] += 1
                continue
            if len(clave_regla) < int(longitud_minima):
                conteos["eliminados_longitud"] += 1
                continue
            salida.append(normal)
            conteos["tokens_conservados"] += 1
            continue

        if token.like_url or token.like_email:
            conteos["eliminados_url_email"] += 1
            continue

        if not token.is_alpha:
            conteos["eliminados_ruido"] += 1
            continue

        if token.is_stop:
            conteos["eliminados_stopword"] += 1
            continue

        if token.pos_.upper() in pos_bloqueados:
            conteos["eliminados_pos"] += 1
            continue

        if len(normal) < int(longitud_minima):
            conteos["eliminados_longitud"] += 1
            continue

        salida.append(normal)
        conteos["tokens_conservados"] += 1

    return salida, dict(conteos)

In [46]:
def limpiar_dataset_spacy_revision(
    df: pd.DataFrame,
    nlp: Language,
    candidatas_texto: Sequence[str],
    inclusiones: set[str],
    exclusiones: set[str],
    pos_excluidos: set[str],
    longitud_minima: int,
    batch_size: int,
    tipo: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Ejecuta la limpieza lingüística principal sobre tweets o trends.

    Conserva el texto fuente para anotación/hashtags y genera las columnas limpias que
    consumirá el modelado. El procesamiento usa `nlp.pipe` para evitar llamadas spaCy
    individuales costosas.

    Args:
    df (pd.DataFrame): Dataset cargado que se desea limpiar.
    nlp (Language): Pipeline spaCy español.
    candidatas_texto (Sequence[str]): Columnas posibles de texto fuente.
    inclusiones (set[str]): Lista blanca de palabras.
    exclusiones (set[str]): Lista negra de palabras.
    pos_excluidos (set[str]): POS que deben eliminarse.
    longitud_minima (int): Longitud mínima de token.
    batch_size (int): Tamaño de lote para `nlp.pipe`.
    tipo (str): `tweets`, `evento` o `trends`.

    Returns:
    tuple[pd.DataFrame, pd.DataFrame]:
    Dataset limpiado y tabla de auditoría de eliminaciones.

    Example:
    >>> limpio, auditoria = limpiar_dataset_spacy_revision(
    ...     tweets.head(10), nlp_es, ["texto_original", "texto_limpio"],
    ...     {"ec"}, {"jajaja"}, {"VERB", "AUX", "DET"}, 3, 128, "tweets"
    ... )
    """
    trabajo = df.copy()
    columna_fuente = resolver_columna_texto_revision(trabajo, candidatas_texto)

    fuente = trabajo[columna_fuente].fillna("").astype(str)
    trabajo["texto_para_anotacion"] = fuente
    trabajo["texto_para_hashtags"] = fuente

    if "texto_limpio" in trabajo.columns:
        trabajo["texto_limpio_fuente"] = (
            trabajo["texto_limpio"].fillna("").astype(str)
        )
    elif tipo == "trends" and "trend_limpio" in trabajo.columns:
        trabajo["texto_limpio_fuente"] = (
            trabajo["trend_limpio"].fillna("").astype(str)
        )
    else:
        trabajo["texto_limpio_fuente"] = fuente

    textos_protegidos = [
        proteger_entidades_sociales_revision(x)
        for x in fuente.tolist()
    ]

    tokens_finales: list[list[str]] = []
    acumulado: Counter[str] = Counter()
    documentos_vacios = 0

    for doc in nlp.pipe(
        textos_protegidos,
        batch_size=int(batch_size),
        n_process=1,
    ):
        tokens, conteos = filtrar_documento_spacy_revision(
            doc=doc,
            inclusiones=inclusiones,
            exclusiones=exclusiones,
            pos_excluidos=pos_excluidos,
            longitud_minima=longitud_minima,
            conservar_hashtags=CONSERVAR_HASHTAGS_MODELO,
            conservar_menciones=CONSERVAR_MENCIONES_MODELO,
        )
        tokens_finales.append(tokens)
        acumulado.update(conteos)
        if not tokens:
            documentos_vacios += 1

    textos_finales = [" ".join(tokens) for tokens in tokens_finales]
    tokens_serializados = [" ".join(tokens) for tokens in tokens_finales]

    trabajo["texto_limpio"] = textos_finales
    trabajo["tokens_sin_stopwords"] = tokens_serializados
    trabajo["cantidad_tokens_sin_stopwords"] = [
        len(tokens) for tokens in tokens_finales
    ]

    if tipo == "trends":
        trabajo["trend_limpio"] = textos_finales

    auditoria = pd.DataFrame([
        {"Indicador": "Dataset", "Valor": str(tipo)},
        {"Indicador": "Columna fuente", "Valor": str(columna_fuente)},
        {"Indicador": "Documentos", "Valor": int(len(trabajo))},
        {"Indicador": "Documentos vacíos tras limpieza", "Valor": int(documentos_vacios)},
        {"Indicador": "Tokens evaluados", "Valor": int(acumulado.get("tokens_evaluados", 0))},
        {"Indicador": "Tokens conservados", "Valor": int(acumulado.get("tokens_conservados", 0))},
        {"Indicador": "Conservados por lista blanca", "Valor": int(acumulado.get("conservados_lista_blanca", 0))},
        {"Indicador": "Eliminados stopword", "Valor": int(acumulado.get("eliminados_stopword", 0))},
        {"Indicador": "Eliminados por POS", "Valor": int(acumulado.get("eliminados_pos", 0))},
        {"Indicador": "Eliminados por longitud", "Valor": int(acumulado.get("eliminados_longitud", 0))},
        {"Indicador": "Eliminados lista negra", "Valor": int(acumulado.get("eliminados_lista_negra", 0))},
        {"Indicador": "Eliminados URL/email", "Valor": int(acumulado.get("eliminados_url_email", 0))},
        {"Indicador": "Eliminados ruido", "Valor": int(acumulado.get("eliminados_ruido", 0))},
        {"Indicador": "Eliminados hashtag", "Valor": int(acumulado.get("eliminados_hashtag", 0))},
        {"Indicador": "Eliminados mención", "Valor": int(acumulado.get("eliminados_mencion", 0))},
    ])
    return trabajo.reset_index(drop=True), auditoria

In [47]:
def auditar_dataset_revision(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """
    Audita tamaño, unicidad, balance, valores faltantes y ruido básico del corpus.

    La auditoría cuantifica los aspectos solicitados por los revisores: número de registros,
    tweets únicos según el texto disponible, distribución por evento, campos faltantes,
    textos vacíos, tokens vacíos y disponibilidad de ubicación. Se reportan por separado
    la unicidad global y la unicidad por `evento + texto_limpio`, ya que el archivo limpio
    no conserva el Tweet ID original.

    Args:
    df (pd.DataFrame): Dataset consolidado limpio y preprocesado.

    Returns:
    dict[str, pd.DataFrame]:
    Tablas `resumen`, `eventos`, `faltantes` y `longitud_tokens`.

    Example:
    >>> auditoria = auditar_dataset_revision(tweets)
    >>> "resumen" in auditoria
    True
    """
    if "texto_limpio" not in df.columns:
        raise KeyError("El dataset debe contener la columna `texto_limpio`.")

    trabajo = df.copy()
    texto = trabajo["texto_limpio"].fillna("").astype(str).str.strip()
    evento = trabajo["evento"].fillna("Sin evento").astype(str).str.strip() if "evento" in trabajo.columns else pd.Series(["Sin evento"] * len(trabajo))
    location = trabajo["location"] if "location" in trabajo.columns else pd.Series([np.nan] * len(trabajo))
    tokens = trabajo["tokens_sin_stopwords"] if "tokens_sin_stopwords" in trabajo.columns else texto

    resumen = pd.DataFrame([
        {"Indicador": "Registros totales", "Valor": int(len(trabajo))},
        {"Indicador": "Textos únicos globales", "Valor": int(texto.nunique(dropna=False))},
        {"Indicador": "Duplicados globales por texto", "Valor": int(texto.duplicated(keep="first").sum())},
        {"Indicador": "Registros únicos por evento + texto", "Valor": int(trabajo.assign(_texto=texto, _evento=evento).drop_duplicates(["_evento", "_texto"]).shape[0])},
        {"Indicador": "Duplicados por evento + texto", "Valor": int(trabajo.assign(_texto=texto, _evento=evento).duplicated(["_evento", "_texto"], keep="first").sum())},
        {"Indicador": "Textos vacíos", "Valor": int((texto == "").sum())},
        {"Indicador": "Tokens vacíos", "Valor": int(tokens.fillna("").astype(str).str.strip().eq("").sum())},
        {"Indicador": "Ubicación faltante o vacía", "Valor": int(location.fillna("").astype(str).str.strip().eq("").sum())},
    ])

    eventos = (
        evento.value_counts(dropna=False)
        .rename_axis("Evento")
        .reset_index(name="Registros")
    )
    eventos["Porcentaje"] = eventos["Registros"] / max(len(trabajo), 1) * 100.0

    faltantes = pd.DataFrame({
        "Columna": trabajo.columns,
        "Faltantes": [int(trabajo[c].isna().sum()) for c in trabajo.columns],
        "Porcentaje_faltante": [float(trabajo[c].isna().mean() * 100.0) for c in trabajo.columns],
    }).sort_values("Porcentaje_faltante", ascending=False).reset_index(drop=True)

    longitudes = tokens.fillna("").astype(str).map(lambda x: len(convertir_tokens(x)))
    longitud_tokens = pd.DataFrame([{
        "Media": float(longitudes.mean()),
        "Mediana": float(longitudes.median()),
        "Desv_std": float(longitudes.std(ddof=1)) if len(longitudes) > 1 else 0.0,
        "Min": int(longitudes.min()) if len(longitudes) else 0,
        "P25": float(longitudes.quantile(0.25)) if len(longitudes) else 0.0,
        "P75": float(longitudes.quantile(0.75)) if len(longitudes) else 0.0,
        "Max": int(longitudes.max()) if len(longitudes) else 0,
    }])

    return {
        "resumen": resumen,
        "eventos": eventos,
        "faltantes": faltantes,
        "longitud_tokens": longitud_tokens,
    }

In [48]:
def graficar_balance_eventos_revision(eventos: pd.DataFrame) -> None:
    """
    Grafica la distribución de registros por evento con Bokeh.

    Permite observar el desbalance entre los dos acontecimientos políticos antes de
    construir los modelos y documentarlo de forma explícita.

    Args:
    eventos (pd.DataFrame): Tabla con columnas `Evento`, `Registros` y `Porcentaje`.

    Returns:
    None:
    Muestra la gráfica directamente dentro del notebook.

    Example:
    >>> graficar_balance_eventos_revision(auditoria["eventos"])
    """
    vista = eventos.copy()
    vista["Evento"] = vista["Evento"].astype(str)
    source = ColumnDataSource(vista)
    p = figure(
        x_range=vista["Evento"].tolist(),
        height=360,
        width=850,
        title="Distribución de tweets por evento",
        toolbar_location="above",
    )
    p.vbar(x="Evento", top="Registros", width=0.65, source=source)
    p.add_tools(HoverTool(tooltips=[("Evento", "@Evento"), ("Registros", "@Registros{0,0}"), ("Porcentaje", "@Porcentaje{0.00}%")]))
    p.xgrid.grid_line_color = None
    p.xaxis.major_label_orientation = 0.5
    show(p)

In [49]:
def preparar_dataset_revision(df: pd.DataFrame) -> pd.DataFrame:
    """
    Elimina textos vacíos y duplicados después de completar la limpieza.

    Los duplicados se identifican mediante `evento + texto_limpio` final. Si el
    `id_registro` ya fue creado antes de limpiar, se conserva. Esto mantiene la
    trazabilidad con archivos de anotación.

    Args:
    df (pd.DataFrame): Dataset consolidado después de la limpieza con spaCy.

    Returns:
    pd.DataFrame:
    Corpus único final para los experimentos.

    Example:
    >>> preparado = preparar_dataset_revision(tweets_limpios)
    >>> "id_registro" in preparado.columns
    True
    """
    trabajo = df.copy()
    if "texto_limpio" not in trabajo.columns:
        raise KeyError("Falta `texto_limpio` después de la limpieza.")

    trabajo["texto_limpio"] = (
        trabajo["texto_limpio"].fillna("").astype(str).str.strip()
    )
    trabajo = trabajo[trabajo["texto_limpio"] != ""].copy()

    subconjunto = ["texto_limpio"]
    if "evento" in trabajo.columns:
        subconjunto = ["evento", "texto_limpio"]

    trabajo = (
        trabajo
        .drop_duplicates(subset=subconjunto, keep="first")
        .reset_index(drop=True)
    )

    if "id_registro" not in trabajo.columns:
        trabajo["id_registro"] = crear_id_registro(trabajo)

    return trabajo

In [50]:
def graficar_trazabilidad_preprocesamiento(trazabilidad: pd.DataFrame, titulo: str) -> None:
    """
    Grafica la reducción de registros a través de las etapas de preprocesamiento.

    Utiliza la tabla de trazabilidad incluida en el archivo de resumen para mostrar
    cuántos registros permanecen después de cada etapa. Si existen varias columnas
    numéricas, se representa cada una como una serie independiente.

    Args:
    trazabilidad (pd.DataFrame): Tabla con etapas y cantidades del preprocesamiento.
    titulo (str): Título descriptivo de la gráfica.

    Returns:
    None:
    Muestra una gráfica Bokeh directamente en el notebook.

    Example:
    >>> graficar_trazabilidad_preprocesamiento(df_trazabilidad, "Trazabilidad")
    """
    df = trazabilidad.copy()
    etapa_col = df.columns[0]
    etapas = df[etapa_col].astype(str).tolist()
    p = figure(
        x_range=etapas,
        width=1000,
        height=420,
        title=titulo,
        toolbar_location="above",
    )
    num_cols = [c for c in df.columns[1:] if pd.api.types.is_numeric_dtype(df[c])]
    colores = Category10[10]
    for i, col_name in enumerate(num_cols):
        p.line(etapas, df[col_name].astype(float), line_width=2, legend_label=str(col_name), color=colores[i % len(colores)])
        p.scatter(etapas, df[col_name].astype(float), size=7, color=colores[i % len(colores)])
    p.xaxis.major_label_orientation = math.pi / 3
    p.xaxis.axis_label = "Etapa"
    p.yaxis.axis_label = "Cantidad"
    p.legend.click_policy = "hide"
    show(p)

In [51]:
def graficar_top_tokens(top_tokens: pd.DataFrame, top_n: int = 50) -> None:
    """
    Grafica los tokens más frecuentes del corpus preprocesado.

    Ordena la tabla de frecuencias y construye un gráfico de barras horizontales para
    reproducir el análisis de términos de alta frecuencia mostrado en la tesis.

    Args:
    top_tokens (pd.DataFrame): DataFrame con columnas de token y frecuencia.
    top_n (int): Número de tokens que se mostrarán.
    *Valor por defecto: 50.*

    Returns:
    None:
    Muestra el gráfico Bokeh en la salida del notebook.

    Example:
    >>> graficar_top_tokens(resumen["Top_50_Tokens"], 20)
    """
    df = top_tokens.iloc[:, :2].copy()
    df.columns = ["Token", "Frecuencia"]
    df = df.sort_values("Frecuencia", ascending=False).head(top_n)
    factores = df["Token"].astype(str).tolist()[::-1]
    valores = df["Frecuencia"].astype(float).tolist()[::-1]
    source = ColumnDataSource({"token": factores, "frecuencia": valores})
    p = figure(y_range=factores, width=1000, height=max(450, 15 * len(factores)), title=f"Top {len(factores)} tokens por frecuencia")
    p.hbar(y="token", right="frecuencia", height=0.75, source=source)
    p.xaxis.axis_label = "Frecuencia"
    p.yaxis.axis_label = "Token"
    p.add_tools(HoverTool(tooltips=[("Token", "@token"), ("Frecuencia", "@frecuencia{0,0}")]))
    show(p)

In [52]:
def graficar_disponibilidad_ubicacion(tweets: pd.DataFrame, columna: str = "location") -> pd.DataFrame:
    """
    Calcula y grafica la proporción de tweets con y sin ubicación registrada.

    Considera como ausencia de ubicación los valores nulos y las cadenas vacías.
    Devuelve el resumen porcentual y muestra un gráfico de sectores Bokeh, equivalente
    al análisis de disponibilidad de ubicación descrito en la tesis.

    Args:
    tweets (pd.DataFrame): Dataset consolidado de tweets.
    columna (str): Columna que contiene la ubicación del usuario.
    *Valor por defecto: "location".*

    Returns:
    pd.DataFrame:
    Tabla con categoría, cantidad y porcentaje de tweets.

    Example:
    >>> resumen_ubicacion = graficar_disponibilidad_ubicacion(tweets)
    """
    ubicacion = tweets[columna].fillna("").astype(str).str.strip()
    con = int((ubicacion != "").sum())
    sin = int((ubicacion == "").sum())
    total = max(con + sin, 1)
    df = pd.DataFrame({
        "Categoria": ["Con ubicación", "Sin ubicación"],
        "Cantidad": [con, sin],
        "Porcentaje": [100 * con / total, 100 * sin / total],
    })
    df["angulo"] = df["Cantidad"] / total * 2 * math.pi
    df["inicio"] = np.cumsum([0.0] + df["angulo"].tolist()[:-1])
    df["fin"] = np.cumsum(df["angulo"].tolist())
    df["color"] = [Category10[10][0], Category10[10][1]]
    source = ColumnDataSource(df)
    p = figure(width=650, height=420, title="Tweets con y sin ubicación", toolbar_location=None, tools="hover", tooltips="@Categoria: @Porcentaje{0.00}%")
    p.wedge(x=0, y=1, radius=0.35, start_angle="inicio", end_angle="fin", color="color", legend_field="Categoria", source=source)
    p.axis.visible = False
    p.grid.grid_line_color = None
    show(p)
    return df.drop(columns=["angulo", "inicio", "fin", "color"])

In [53]:
def calcular_frecuencia_trends(tweets: pd.DataFrame, trends: pd.DataFrame, max_ngram: int = 4) -> pd.DataFrame:
    """
    Calcula en cuántos tweets aparece cada trend limpio.

    Construye un vocabulario de n-gramas a partir de `trend_limpio` y utiliza una
    matriz dispersa de presencia documental para contar de forma eficiente la cantidad
    de tweets que contienen cada trend. Los trends con más palabras que `max_ngram`
    se excluyen de este cálculo para controlar el costo computacional.

    Args:
    tweets (pd.DataFrame): Dataset de tweets con la columna `texto_limpio`.
    trends (pd.DataFrame): Dataset de trends con la columna `trend_limpio`.
    max_ngram (int): Longitud máxima, en palabras, de un trend evaluado.
    *Valor por defecto: 4.*

    Returns:
    pd.DataFrame:
    Tabla ordenada con trend, frecuencia documental y porcentaje del corpus.

    Example:
    >>> frecuencias = calcular_frecuencia_trends(tweets, trends, 4)
    """
    if "texto_limpio" not in tweets.columns or "trend_limpio" not in trends.columns:
        raise KeyError("Se requieren las columnas 'texto_limpio' y 'trend_limpio'.")

    candidatos = []
    for trend in trends["trend_limpio"].dropna().astype(str).str.strip().str.lower():
        n = len(trend.split())
        if trend and 1 <= n <= max_ngram:
            candidatos.append(trend)
    candidatos = list(dict.fromkeys(candidatos))
    vocabulario = {term: i for i, term in enumerate(candidatos)}

    vectorizador = CountVectorizer(
        vocabulary=vocabulario,
        ngram_range=(1, max_ngram),
        binary=True,
        lowercase=True,
        token_pattern=r"(?u)[#@]?\b\w+\b",
    )
    matriz = vectorizador.transform(tweets["texto_limpio"].fillna("").astype(str))
    frecuencias = np.asarray(matriz.sum(axis=0)).ravel()
    total = max(len(tweets), 1)
    salida = pd.DataFrame({
        "Trend": candidatos,
        "Frecuencia": frecuencias.astype(int),
        "Porcentaje_tweets": 100 * frecuencias / total,
    })
    return salida.sort_values(["Frecuencia", "Trend"], ascending=[False, True]).reset_index(drop=True)

In [54]:
def graficar_top_trends(frecuencias: pd.DataFrame, top_n: int = 30) -> None:
    """
    Grafica los trends con mayor presencia en el corpus de tweets.

    Selecciona los trends con más apariciones documentales y los presenta como barras
    horizontales interactivas, permitiendo inspeccionar frecuencia y porcentaje.

    Args:
    frecuencias (pd.DataFrame): Tabla generada por `calcular_frecuencia_trends`.
    top_n (int): Cantidad de trends a representar.
    *Valor por defecto: 30.*

    Returns:
    None:
    Muestra la visualización Bokeh en el notebook.

    Example:
    >>> graficar_top_trends(frecuencias, 25)
    """
    df = frecuencias.head(top_n).iloc[::-1].copy()
    source = ColumnDataSource(df)
    p = figure(y_range=df["Trend"].astype(str).tolist(), width=1000, height=max(450, 18 * len(df)), title=f"Top {len(df)} trends por presencia en tweets")
    p.hbar(y="Trend", right="Frecuencia", height=0.7, source=source)
    p.xaxis.axis_label = "Tweets que contienen el trend"
    p.add_tools(HoverTool(tooltips=[("Trend", "@Trend"), ("Frecuencia", "@Frecuencia{0,0}"), ("Porcentaje", "@Porcentaje_tweets{0.000}%")]))
    show(p)

In [55]:
rutas = verificar_archivos(ARCHIVOS, DATA_DIR)
datos = cargar_datasets(rutas)

In [56]:
tweets = datos["tweets"].reset_index(drop=True)
trends = datos["trends"].reset_index(drop=True)
paro = datos["paro"].reset_index(drop=True)
muerte = datos["muerte"].reset_index(drop=True)
resumen_preprocesamiento = datos["resumen"]

In [57]:
tweets = asignar_ids_estables_pre_limpieza(tweets)

In [58]:
nlp_es = cargar_modelo_spacy_espanol(MODELO_SPACY_ES)
print(f"Modelo de limpieza lingüística cargado: {MODELO_SPACY_ES}")

Modelo de limpieza lingüística cargado: es_core_news_sm


In [59]:
tweets, AUDITORIA_LIMPIEZA_TWEETS = limpiar_dataset_spacy_revision(
    tweets,
    nlp_es,
    ["texto_original", "texto", "full_text", "text", "texto_limpio"],
    PALABRAS_INCLUIDAS,
    PALABRAS_EXCLUIDAS,
    POS_EXCLUIDOS,
    LONGITUD_MINIMA_TOKEN,
    SPACY_BATCH_SIZE,
    "tweets",
)

In [60]:
paro, AUDITORIA_LIMPIEZA_PARO = limpiar_dataset_spacy_revision(
    paro,
    nlp_es,
    ["texto_original", "texto", "full_text", "text", "texto_limpio"],
    PALABRAS_INCLUIDAS,
    PALABRAS_EXCLUIDAS,
    POS_EXCLUIDOS,
    LONGITUD_MINIMA_TOKEN,
    SPACY_BATCH_SIZE,
    "evento_paro",
)

In [61]:
muerte, AUDITORIA_LIMPIEZA_MUERTE = limpiar_dataset_spacy_revision(
    muerte,
    nlp_es,
    ["texto_original", "texto", "full_text", "text", "texto_limpio"],
    PALABRAS_INCLUIDAS,
    PALABRAS_EXCLUIDAS,
    POS_EXCLUIDOS,
    LONGITUD_MINIMA_TOKEN,
    SPACY_BATCH_SIZE,
    "evento_muerte_cruzada",
)

In [62]:
trends, AUDITORIA_LIMPIEZA_TRENDS = limpiar_dataset_spacy_revision(
    trends,
    nlp_es,
    ["trend", "trend_original", "trend_limpio"],
    PALABRAS_INCLUIDAS,
    PALABRAS_EXCLUIDAS,
    POS_EXCLUIDOS,
    LONGITUD_MINIMA_TOKEN,
    SPACY_BATCH_SIZE,
    "trends",
)

In [63]:
AUDITORIA_LIMPIEZA = pd.concat(
    [
        AUDITORIA_LIMPIEZA_TWEETS.assign(Origen="tweets"),
        AUDITORIA_LIMPIEZA_PARO.assign(Origen="paro"),
        AUDITORIA_LIMPIEZA_MUERTE.assign(Origen="muerte"),
        AUDITORIA_LIMPIEZA_TRENDS.assign(Origen="trends"),
    ],
    ignore_index=True,
)

tweets_revision = preparar_dataset_revision(tweets)

In [64]:
AUDITORIA_DATASET = auditar_dataset_revision(tweets_revision)

print(f"Tweets consolidados cargados y limpiados: {len(tweets):,}")
print(f"Corpus final después de limpieza y deduplicación: {len(tweets_revision):,}")
print(f"Trends después de limpieza: {len(trends):,}")
print(f"Paro Nacional después de limpieza: {len(paro):,}")
print(f"Muerte Cruzada después de limpieza: {len(muerte):,}")
print(f"Lista blanca: {sorted(PALABRAS_INCLUIDAS)}")
print(f"Lista negra: {sorted(PALABRAS_EXCLUIDAS)}")
print(f"POS eliminados: {sorted(POS_EXCLUIDOS)}")
print(f"Longitud mínima de token: {LONGITUD_MINIMA_TOKEN}")

Tweets consolidados cargados y limpiados: 144,704
Corpus final después de limpieza y deduplicación: 138,400
Trends después de limpieza: 1,161
Paro Nacional después de limpieza: 29,642
Muerte Cruzada después de limpieza: 115,062
Lista blanca: ['ec']
Lista negra: ['jajaja']
POS eliminados: ['AUX', 'DET', 'VERB']
Longitud mínima de token: 3


In [65]:
mostrar_tabla_bokeh(
    AUDITORIA_LIMPIEZA,
    "Auditoría del proceso de limpieza lingüística",
    ancho=1300,
    alto=520,
    max_filas=80,
)

In [66]:
mostrar_tabla_bokeh(
    validar_consistencia_eventos(tweets, paro, muerte),
    "Consistencia entre consolidado y archivos por evento después de la limpieza",
    alto=180,
)

In [67]:
mostrar_tabla_bokeh(
    AUDITORIA_DATASET["resumen"],
    "Auditoría del corpus después de limpieza y deduplicación",
    alto=300,
)

In [68]:
mostrar_tabla_bokeh(
    AUDITORIA_DATASET["eventos"],
    "Distribución por evento después de limpieza y deduplicación",
    alto=200,
)

In [69]:
mostrar_tabla_bokeh(
    AUDITORIA_DATASET["faltantes"],
    "Valores faltantes después de limpieza y deduplicación",
    alto=380,
)

In [70]:
mostrar_tabla_bokeh(
    AUDITORIA_DATASET["longitud_tokens"],
    "Longitud de tweets después de limpieza y deduplicación",
    alto=170,
)

In [71]:
graficar_balance_eventos_revision(AUDITORIA_DATASET["eventos"])

In [72]:
resumen_ubicacion = graficar_disponibilidad_ubicacion(tweets_revision)

In [73]:
mostrar_tabla_bokeh(
    resumen_ubicacion,
    "Disponibilidad de ubicación después de la limpieza",
    alto=170,
)

In [74]:
frecuencias_trends = calcular_frecuencia_trends(
    tweets_revision,
    trends,
    MAX_NGRAM_TREND,
)

In [75]:
graficar_top_trends(frecuencias_trends, 30)

In [76]:
mostrar_tabla_bokeh(
    frecuencias_trends,
    "Frecuencia de trends después de la limpieza",
    alto=340,
    max_filas=30,
)

# 4. Referencias externas y preparación de anotaciones humanas

In [77]:
def normalizar_hashtag(hashtag: str) -> str:
    """
    Normaliza un hashtag para comparaciones reproducibles.

    Convierte el valor a minúsculas, elimina espacios externos y garantiza que el
    resultado empiece con `#`. La función no intenta corregir ortografía ni fusionar
    variantes semánticas, porque esas decisiones deben permanecer explícitas.

    Args:
    hashtag (str): Hashtag original o término que se desea normalizar.

    Returns:
    str:
    Hashtag normalizado en minúsculas y con prefijo `#`.

    Example:
    >>> normalizar_hashtag(" ParoNacional ")
    '#paronacional'
    """
    valor = str(hashtag).strip().lower()
    if not valor:
        return ""
    return valor if valor.startswith("#") else f"#{valor}"

In [78]:
def extraer_hashtags_texto(texto: str) -> list[str]:
    """
    Extrae los hashtags presentes en un texto preservando su forma normalizada.

    Utiliza una expresión regular compatible con letras, números y caracteres
    alfanuméricos Unicode. Los hashtags se devuelven sin duplicados y en el orden de
    primera aparición para facilitar auditorías del silver standard.

    Args:
    texto (str): Texto original o preprocesado del tweet.

    Returns:
    list[str]:
    Lista ordenada de hashtags normalizados encontrados en el texto.

    Example:
    >>> extraer_hashtags_texto("Hoy #ParoNacional y #Ecuador")
    ['#paronacional', '#ecuador']
    """
    encontrados = re.findall(r"(?<!\w)#[\wáéíóúüñÁÉÍÓÚÜÑ]+", str(texto).lower())
    return list(dict.fromkeys(normalizar_hashtag(x) for x in encontrados if x))

In [79]:
def construir_inventario_hashtags_revision(df: pd.DataFrame, hashtags_politicos: set[str], hashtags_no_politicos: set[str]) -> pd.DataFrame:
    """
    Construye un inventario de hashtags con frecuencia, evento y categoría silver.

    El inventario permite auditar qué hashtags aparecen realmente en el corpus y qué
    términos participan en la regla de etiquetado débil. Los hashtags no incluidos en
    las listas estrictas quedan como `No clasificado` y no generan etiquetas silver.

    Args:
    df (pd.DataFrame): Corpus de tweets con `texto_limpio` y, opcionalmente, `evento`.
    hashtags_politicos (set[str]): Hashtags considerados señales políticas de alta precisión.
    hashtags_no_politicos (set[str]): Hashtags considerados señales no políticas de alta precisión.

    Returns:
    pd.DataFrame:
    Inventario ordenado por frecuencia con hashtag, categoría y distribución por evento.

    Example:
    >>> inventario = construir_inventario_hashtags_revision(tweets_revision, HASHTAGS_POLITICOS_ESTRICTOS, HASHTAGS_NO_POLITICOS_ESTRICTOS)
    """
    politicos = {normalizar_hashtag(x) for x in hashtags_politicos}
    no_politicos = {normalizar_hashtag(x) for x in hashtags_no_politicos}
    contador_total: Counter[str] = Counter()
    contador_evento: dict[str, Counter[str]] = {}

    for _, fila in df.iterrows():
        evento = str(fila.get("evento", "Sin evento"))
        columna_hash = "texto_para_hashtags" if "texto_para_hashtags" in df.columns else "texto_limpio"
        hashtags = extraer_hashtags_texto(fila.get(columna_hash, ""))
        contador_total.update(hashtags)
        contador_evento.setdefault(evento, Counter()).update(hashtags)

    filas: list[dict[str, Any]] = []
    eventos = sorted(contador_evento)
    for hashtag, frecuencia in contador_total.most_common():
        if hashtag in politicos:
            categoria = "Político estricto"
        elif hashtag in no_politicos:
            categoria = "No político estricto"
        else:
            categoria = "No clasificado"
        registro: dict[str, Any] = {
            "Hashtag": hashtag,
            "Frecuencia": int(frecuencia),
            "Categoria_silver": categoria,
        }
        for evento in eventos:
            registro[f"Frecuencia_{evento}"] = int(contador_evento[evento].get(hashtag, 0))
        filas.append(registro)

    return pd.DataFrame(filas)

In [80]:
def asignar_etiqueta_silver_hashtags(texto: str, hashtags_politicos: set[str], hashtags_no_politicos: set[str]) -> tuple[int | None, str, list[str], list[str], list[str]]:
    """
    Asigna una etiqueta débil política/no política usando hashtags explícitos.

    Un tweet recibe etiqueta 1 cuando contiene al menos un hashtag político estricto y
    ninguno no político. Recibe 0 cuando contiene al menos un hashtag no político
    estricto y ninguno político. Si contiene ambos tipos se marca como ambiguo; si no
    contiene ninguna señal estricta queda sin etiqueta y no se usa en la evaluación.

    Args:
    texto (str): Texto del tweet del que se extraen hashtags.
    hashtags_politicos (set[str]): Conjunto de hashtags políticos de alta precisión.
    hashtags_no_politicos (set[str]): Conjunto de hashtags no políticos de alta precisión.

    Returns:
    tuple[int | None, str, list[str], list[str], list[str]]:
    Etiqueta silver, motivo, todos los hashtags, hashtags políticos y hashtags no políticos.

    Example:
    >>> asignar_etiqueta_silver_hashtags("#ParoNacional cierre de vías", {"#paronacional"}, {"#championsleague"})[0]
    1
    """
    politicos = {normalizar_hashtag(x) for x in hashtags_politicos}
    no_politicos = {normalizar_hashtag(x) for x in hashtags_no_politicos}
    hashtags = extraer_hashtags_texto(texto)
    encontrados_politicos = sorted(set(hashtags).intersection(politicos))
    encontrados_no_politicos = sorted(set(hashtags).intersection(no_politicos))

    if encontrados_politicos and not encontrados_no_politicos:
        return 1, "Hashtag político estricto", hashtags, encontrados_politicos, encontrados_no_politicos
    if encontrados_no_politicos and not encontrados_politicos:
        return 0, "Hashtag no político estricto", hashtags, encontrados_politicos, encontrados_no_politicos
    if encontrados_politicos and encontrados_no_politicos:
        return None, "Ambiguo: contiene señales de ambas clases", hashtags, encontrados_politicos, encontrados_no_politicos
    return None, "Sin hashtag estricto clasificable", hashtags, encontrados_politicos, encontrados_no_politicos

In [81]:
def construir_silver_standard_hashtags(df: pd.DataFrame, hashtags_politicos: set[str], hashtags_no_politicos: set[str]) -> pd.DataFrame:
    """
    Genera el silver standard de tweets a partir de señales humanas en hashtags.

    Aplica la regla estricta de etiquetado hashtag a cada registro y conserva únicamente
    las filas con etiqueta 0 o 1. El resultado mantiene `id_registro`, el evento y el texto
    para poder excluir exactamente estas filas del corpus de desarrollo.

    Args:
    df (pd.DataFrame): Corpus único preparado para revisión y con `id_registro`.
    hashtags_politicos (set[str]): Hashtags políticos de alta precisión.
    hashtags_no_politicos (set[str]): Hashtags no políticos de alta precisión.

    Returns:
    pd.DataFrame:
    Tweets con etiqueta silver, evidencia de hashtags y motivo de asignación.

    Example:
    >>> silver = construir_silver_standard_hashtags(tweets_revision, HASHTAGS_POLITICOS_ESTRICTOS, HASHTAGS_NO_POLITICOS_ESTRICTOS)
    """
    if "id_registro" not in df.columns:
        raise KeyError("El dataset debe contener `id_registro`; ejecute preparar_dataset_revision primero.")

    filas: list[dict[str, Any]] = []
    for _, fila in df.iterrows():
        etiqueta, motivo, hashtags, politicos, no_politicos = asignar_etiqueta_silver_hashtags(
            fila.get("texto_para_hashtags", fila.get("texto_limpio", "")),
            hashtags_politicos,
            hashtags_no_politicos,
        )
        if etiqueta is None:
            continue
        registro = fila.to_dict()
        registro.update({
            "etiqueta_silver_hashtag": int(etiqueta),
            "motivo_silver": motivo,
            "hashtags_detectados": " ".join(hashtags),
            "hashtags_politicos": " ".join(politicos),
            "hashtags_no_politicos": " ".join(no_politicos),
        })
        filas.append(registro)

    resultado = pd.DataFrame(filas)
    if resultado.empty:
        raise ValueError("No se obtuvieron tweets con etiquetas silver a partir de los hashtags configurados.")
    return resultado.reset_index(drop=True)

In [82]:
def eliminar_hashtags_texto(texto: str) -> str:
    """
    Elimina completamente los hashtags de un texto para evitar fuga de etiqueta.

    La validación silver usa hashtags como referencia. Por tanto, permitir que el mismo
    hashtag permanezca como característica produciría una evaluación circular. Esta
    función retira todos los tokens que comienzan con `#` y normaliza espacios.

    Args:
    texto (str): Texto que puede contener uno o más hashtags.

    Returns:
    str:
    Texto sin hashtags y con espacios normalizados.

    Example:
    >>> eliminar_hashtags_texto("cierre de vías #ParoNacional hoy")
    'cierre de vías hoy'
    """
    limpio = re.sub(r"(?<!\w)#[\wáéíóúüñÁÉÍÓÚÜÑ]+", " ", str(texto))
    limpio = limpio.replace("#", " ")
    return re.sub(r"\s+", " ", limpio).strip()

In [83]:
def preparar_corpus_ciego_hashtags(df: pd.DataFrame) -> pd.DataFrame:
    """
    Crea una copia del corpus en la que los hashtags no pueden actuar como features.

    Se eliminan hashtags de `texto_limpio`, `texto`, `texto_original`, `tokens` y
    `tokens_sin_stopwords` cuando esas columnas existen. Las etiquetas silver y las
    columnas de evidencia se conservan para evaluación, pero nunca se incorporan a los
    vectores de entrada.

    Args:
    df (pd.DataFrame): Corpus de desarrollo o holdout silver.

    Returns:
    pd.DataFrame:
    Copia ciega a hashtags lista para modelado semántico.

    Example:
    >>> ciego = preparar_corpus_ciego_hashtags(silver)
    """
    trabajo = df.copy()
    for columna in ["texto_limpio", "texto", "texto_original", "tokens", "tokens_sin_stopwords"]:
        if columna in trabajo.columns:
            trabajo[columna] = trabajo[columna].fillna("").astype(str).map(eliminar_hashtags_texto)
    if "tokens_sin_stopwords" in trabajo.columns:
        trabajo["cantidad_tokens_sin_stopwords"] = trabajo["tokens_sin_stopwords"].map(lambda x: len(convertir_tokens(x)))
    return trabajo

In [84]:
def separar_holdout_anotacion_manual(df: pd.DataFrame, n_muestra: int, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Separa antes del modelado una muestra independiente destinada a anotación humana.

    La separación se estratifica por evento y se ejecuta antes de Word2Vec, clustering,
    expansión léxica o pseudoetiquetado. De esta forma, las filas reservadas no pueden
    influir en el espacio semántico utilizado posteriormente para validarlas.

    Args:
    df (pd.DataFrame): Corpus único preparado para los experimentos de revisión.
    n_muestra (int): Número de tweets reservados para anotación manual.
    seed (int): Semilla de reproducibilidad.

    Returns:
    tuple[pd.DataFrame, pd.DataFrame]:
    Corpus de desarrollo y holdout manual independiente.

    Example:
    >>> desarrollo, holdout = separar_holdout_anotacion_manual(tweets_revision, 100, 5)
    """
    if n_muestra <= 0 or n_muestra >= len(df):
        raise ValueError("n_muestra debe ser mayor que 0 y menor que el número de registros.")

    estrato = df["evento"].fillna("Sin evento").astype(str)
    desarrollo, holdout = train_test_split(
        df,
        test_size=int(n_muestra),
        random_state=seed,
        stratify=estrato,
    )
    return desarrollo.reset_index(drop=True), holdout.reset_index(drop=True)

In [85]:
def guardar_plantilla_anotacion_manual(
    holdout: pd.DataFrame,
    ruta: Path,
    version_protocolo: str,
) -> Path:
    """
    Genera y sincroniza la plantilla ciega de anotación humana principal.

    Los dos anotadores únicamente deben completar sus respectivas columnas en la
    primera etapa. La columna `etiqueta_consenso` se deja vacía: posteriormente el
    notebook la completa automáticamente cuando ambos anotadores coinciden y reserva
    únicamente los desacuerdos para adjudicación humana.

    Si existe una plantilla compatible se conserva. Si existe una versión antigua sin
    etiquetas, se respalda. Si contiene anotaciones incompatibles, se detiene para no
    destruir trabajo humano.

    Args:
    holdout (pd.DataFrame): Holdout humano reservado antes del modelado.
    ruta (Path): Archivo Excel de destino.
    version_protocolo (str): Identificador de la versión metodológica.

    Returns:
    Path:
    Ruta de la plantilla válida creada o reutilizada.

    Example:
    >>> ruta = guardar_plantilla_anotacion_manual(
    ...     holdout_manual,
    ...     Path("data/anotacion_manual_politica.xlsx"),
    ...     "v1",
    ... )
    """
    ruta.parent.mkdir(parents=True, exist_ok=True)

    if "id_registro" not in holdout.columns:
        raise KeyError(
            "El holdout no contiene la columna obligatoria `id_registro`."
        )

    ids_esperados = set(
        holdout["id_registro"].astype(str).str.strip()
    )

    if ruta.exists():
        try:
            with pd.ExcelFile(ruta, engine="openpyxl") as xls:
                if "Anotacion" not in xls.sheet_names:
                    existente = pd.DataFrame()
                else:
                    existente = pd.read_excel(
                        xls,
                        sheet_name="Anotacion",
                    )

                if "id_registro" in existente.columns:
                    ids_existentes = set(
                        existente["id_registro"]
                        .astype(str)
                        .str.strip()
                    )
                else:
                    ids_existentes = set()

                columnas_etiquetas = [
                    columna
                    for columna in [
                        "etiqueta_anotador_1",
                        "etiqueta_anotador_2",
                        "etiqueta_consenso",
                    ]
                    if columna in existente.columns
                ]

                tiene_anotaciones = bool(
                    columnas_etiquetas
                    and existente[columnas_etiquetas]
                    .notna()
                    .any()
                    .any()
                )

                version_ok = False
                if "Metadatos" in xls.sheet_names:
                    metadatos_existentes = pd.read_excel(
                        xls,
                        sheet_name="Metadatos",
                    )
                    if {
                        "Clave",
                        "Valor",
                    }.issubset(metadatos_existentes.columns):
                        mapa_metadatos = dict(
                            zip(
                                metadatos_existentes["Clave"].astype(str),
                                metadatos_existentes["Valor"].astype(str),
                            )
                        )
                        version_ok = (
                            mapa_metadatos.get("version_protocolo")
                            == str(version_protocolo)
                        )

        except PermissionError as exc:
            raise PermissionError(
                "\nNo fue posible leer correctamente el archivo:\n"
                f"{ruta}\n\n"
                "Cierre Microsoft Excel, LibreOffice y el panel de vista previa "
                "del Explorador de Windows; luego vuelva a ejecutar la celda."
            ) from exc

        if ids_existentes == ids_esperados and version_ok:
            print(
                "La plantilla existente es compatible con el protocolo actual. "
                "Se conservarán las etiquetas ya ingresadas."
            )
            return ruta

        if tiene_anotaciones:
            raise RuntimeError(
                "\nExiste una plantilla anterior con anotaciones humanas:\n"
                f"{ruta}\n\n"
                "No se sobrescribirá automáticamente. Respáldela manualmente "
                "antes de regenerar una plantilla incompatible."
            )

        respaldo = ruta.with_name(
            f"{ruta.stem}_backup_"
            f"{time.strftime('%Y%m%d_%H%M%S')}"
            f"{ruta.suffix}"
        )

        try:
            ruta.replace(respaldo)
        except PermissionError as exc:
            raise PermissionError(
                "\nNo se pudo respaldar la plantilla antigua:\n"
                f"{ruta}\n\n"
                "El archivo continúa bloqueado por Windows. Cierre Excel, "
                "desactive el panel de vista previa con Alt + P y compruebe "
                "que EXCEL.EXE no permanezca abierto."
            ) from exc

        print(f"Plantilla antigua sin etiquetas respaldada en: {respaldo}")

    columnas = [
        columna
        for columna in [
            "id_registro",
            "evento",
            "texto_para_anotacion",
            "texto_limpio",
            "location",
        ]
        if columna in holdout.columns
    ]

    plantilla = holdout[columnas].copy()
    plantilla = plantilla.rename(
        columns={
            "texto_para_anotacion":
            "texto_original_para_anotar"
        }
    )

    plantilla["etiqueta_anotador_1"] = pd.NA
    plantilla["etiqueta_anotador_2"] = pd.NA
    plantilla["etiqueta_consenso"] = pd.NA
    plantilla["observacion"] = ""

    instrucciones = pd.DataFrame(
        {
            "Instrucción": [
                "Leer `texto_original_para_anotar`; no decidir a partir del evento.",
                "Anotador 1 completa únicamente `etiqueta_anotador_1` con 1=político o 0=no político.",
                "Anotador 2 completa únicamente `etiqueta_anotador_2` con 1=político o 0=no político.",
                "Los dos anotadores deben trabajar independientemente.",
                "NO llenar inicialmente `etiqueta_consenso`.",
                "Después de terminar ambos anotadores, ejecutar la etapa de consenso automático del notebook.",
                "Cuando ambos coinciden, el notebook completa `etiqueta_consenso` automáticamente.",
                "Cuando discrepan, el caso aparece en la hoja `Desacuerdos_Adjudicar`.",
                "Un tercer adjudicador, o uno de los dos anotadores tras una revisión explícita, decide 0/1 únicamente en los desacuerdos.",
                "No modificar `id_registro`.",
            ]
        }
    )

    metadatos = pd.DataFrame(
        {
            "Clave": [
                "version_protocolo",
                "tipo_muestra",
                "n_registros",
                "semilla",
                "politico",
                "no_politico",
            ],
            "Valor": [
                str(version_protocolo),
                "holdout humano independiente previo al modelado",
                str(len(plantilla)),
                str(RANDOM_STATE),
                "1",
                "0",
            ],
        }
    )

    try:
        with pd.ExcelWriter(
            ruta,
            engine="openpyxl",
            mode="w",
        ) as writer:
            plantilla.to_excel(
                writer,
                sheet_name="Anotacion",
                index=False,
            )
            instrucciones.to_excel(
                writer,
                sheet_name="Instrucciones",
                index=False,
            )
            metadatos.to_excel(
                writer,
                sheet_name="Metadatos",
                index=False,
            )
    except PermissionError as exc:
        raise PermissionError(
            "\nNo se pudo crear la plantilla:\n"
            f"{ruta}\n\n"
            "Cierre Excel y cualquier aplicación que esté utilizando el archivo."
        ) from exc

    print(f"Plantilla de anotación humana creada correctamente: {ruta}")
    return ruta

In [86]:
def cargar_anotaciones_manuales(ruta: Path) -> pd.DataFrame:
    """
    Carga y valida las etiquetas humanas de la plantilla de anotación.

    Verifica que las etiquetas disponibles pertenezcan al conjunto {0, 1}. Las filas
    sin consenso se conservan en el archivo, pero se excluirán de las métricas de
    validación hasta que hayan sido adjudicadas.

    Args:
    ruta (Path): Ruta al archivo Excel de anotación manual.

    Returns:
    pd.DataFrame:
    Tabla de anotaciones con etiquetas numéricas cuando están disponibles.

    Example:
    >>> anotaciones = cargar_anotaciones_manuales(Path("data/anotacion_manual_politica.xlsx"))
    """
    if not ruta.exists():
        raise FileNotFoundError(f"No existe el archivo de anotación: {ruta}")

    anotaciones = pd.read_excel(ruta, sheet_name="Anotacion")
    requeridas = {"id_registro", "etiqueta_anotador_1", "etiqueta_anotador_2", "etiqueta_consenso"}
    faltantes = requeridas.difference(anotaciones.columns)
    if faltantes:
        raise KeyError(f"Faltan columnas en la plantilla: {sorted(faltantes)}")

    for columna in ["etiqueta_anotador_1", "etiqueta_anotador_2", "etiqueta_consenso"]:
        valores = pd.to_numeric(anotaciones[columna], errors="coerce")
        invalidos = valores.dropna()[~valores.dropna().isin([0, 1])]
        if not invalidos.empty:
            raise ValueError(f"La columna {columna} contiene etiquetas distintas de 0/1.")
        anotaciones[columna] = valores.astype("Int64")

    return anotaciones

In [87]:
def calcular_acuerdo_anotadores(anotaciones: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula el acuerdo observado y Cohen's kappa entre dos anotadores.

    El cálculo usa únicamente filas etiquetadas por ambos anotadores. Esto permite
    documentar la confiabilidad de la anotación manual independiente solicitada por
    los revisores.

    Args:
    anotaciones (pd.DataFrame): Tabla con `etiqueta_anotador_1` y `etiqueta_anotador_2`.

    Returns:
    pd.DataFrame:
    Número de pares válidos, acuerdo porcentual y Cohen's kappa.

    Example:
    >>> acuerdo = calcular_acuerdo_anotadores(anotaciones)
    """
    pares = anotaciones.dropna(subset=["etiqueta_anotador_1", "etiqueta_anotador_2"]).copy()
    if pares.empty:
        return pd.DataFrame([{"N_pares": 0, "Acuerdo_porcentual": np.nan, "Cohen_kappa": np.nan}])

    y1 = pares["etiqueta_anotador_1"].astype(int).to_numpy()
    y2 = pares["etiqueta_anotador_2"].astype(int).to_numpy()
    acuerdo = float(np.mean(y1 == y2) * 100.0)
    kappa = float(cohen_kappa_score(y1, y2)) if len(np.unique(np.concatenate([y1, y2]))) > 1 else np.nan
    return pd.DataFrame([{"N_pares": int(len(pares)), "Acuerdo_porcentual": acuerdo, "Cohen_kappa": kappa}])

In [88]:
def actualizar_consenso_automatico_excel(
    ruta: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Completa automáticamente el consenso y prepara los desacuerdos para adjudicación.

    Primero recupera, si existe, cualquier decisión previamente escrita en la hoja
    `Desacuerdos_Adjudicar`. Después completa `etiqueta_consenso` automáticamente
    cuando ambos anotadores coinciden. Los desacuerdos sin decisión quedan vacíos y
    se copian a una hoja específica para que un tercer adjudicador, o uno de los dos
    anotadores tras revisión, decida si el tweet es político (1) o no político (0).

    La función nunca inventa el consenso de un desacuerdo.

    Args:
    ruta (Path): Ruta de `anotacion_manual_politica.xlsx`.

    Returns:
    tuple[pd.DataFrame, pd.DataFrame]:
    Tabla completa de anotaciones actualizada y tabla de desacuerdos aún pendientes.

    Example:
    >>> anotaciones, pendientes = actualizar_consenso_automatico_excel(
    ...     Path("data/anotacion_manual_politica.xlsx")
    ... )
    """
    if not ruta.exists():
        raise FileNotFoundError(
            f"No existe el archivo de anotaciones: {ruta}"
        )

    try:
        with pd.ExcelFile(ruta, engine="openpyxl") as xls:
            hojas = {
                nombre: pd.read_excel(
                    xls,
                    sheet_name=nombre,
                )
                for nombre in xls.sheet_names
            }
    except PermissionError as exc:
        raise PermissionError(
            "\nNo se puede leer el archivo porque está abierto o bloqueado:\n"
            f"{ruta}\n\n"
            "Cierre Excel y vuelva a ejecutar esta celda."
        ) from exc

    if "Anotacion" not in hojas:
        raise KeyError(
            "El archivo no contiene la hoja obligatoria `Anotacion`."
        )

    anotaciones = hojas["Anotacion"].copy()

    requeridas = {
        "id_registro",
        "etiqueta_anotador_1",
        "etiqueta_anotador_2",
        "etiqueta_consenso",
    }
    faltantes = requeridas.difference(anotaciones.columns)
    if faltantes:
        raise KeyError(
            f"Faltan columnas requeridas: {sorted(faltantes)}"
        )

    for columna in [
        "etiqueta_anotador_1",
        "etiqueta_anotador_2",
        "etiqueta_consenso",
    ]:
        valores = pd.to_numeric(
            anotaciones[columna],
            errors="coerce",
        )
        invalidos = valores.dropna()[
            ~valores.dropna().isin([0, 1])
        ]
        if not invalidos.empty:
            raise ValueError(
                f"La columna `{columna}` contiene valores distintos de 0/1."
            )
        anotaciones[columna] = valores.astype("Int64")

    # Recuperar adjudicaciones guardadas anteriormente en la hoja específica.
    adjudicacion_previa = hojas.get(
        "Desacuerdos_Adjudicar",
        pd.DataFrame(),
    ).copy()

    if (
        not adjudicacion_previa.empty
        and {
            "id_registro",
            "etiqueta_consenso",
        }.issubset(adjudicacion_previa.columns)
    ):
        adjudicacion_previa["etiqueta_consenso"] = pd.to_numeric(
            adjudicacion_previa["etiqueta_consenso"],
            errors="coerce",
        ).astype("Int64")

        mapa_adjudicado = (
            adjudicacion_previa
            .dropna(subset=["etiqueta_consenso"])
            .set_index("id_registro")["etiqueta_consenso"]
            .to_dict()
        )

        for indice, identificador in anotaciones["id_registro"].items():
            if identificador in mapa_adjudicado:
                anotaciones.at[
                    indice,
                    "etiqueta_consenso",
                ] = int(mapa_adjudicado[identificador])

    ambos_completos = (
        anotaciones["etiqueta_anotador_1"].notna()
        & anotaciones["etiqueta_anotador_2"].notna()
    )

    coinciden = (
        ambos_completos
        & (
            anotaciones["etiqueta_anotador_1"]
            == anotaciones["etiqueta_anotador_2"]
        )
    )

    desacuerdan = (
        ambos_completos
        & (
            anotaciones["etiqueta_anotador_1"]
            != anotaciones["etiqueta_anotador_2"]
        )
    )

    # Los acuerdos son consenso directo y reproducible.
    anotaciones.loc[
        coinciden,
        "etiqueta_consenso",
    ] = anotaciones.loc[
        coinciden,
        "etiqueta_anotador_1",
    ]

    consenso_valido = anotaciones["etiqueta_consenso"].isin([0, 1])

    anotaciones["requiere_adjudicacion"] = desacuerdan
    anotaciones["estado_consenso"] = "ANOTACION_INCOMPLETA"

    anotaciones.loc[
        coinciden,
        "estado_consenso",
    ] = "ACUERDO_AUTOMATICO"

    anotaciones.loc[
        desacuerdan & consenso_valido,
        "estado_consenso",
    ] = "ADJUDICADO"

    anotaciones.loc[
        desacuerdan & ~consenso_valido,
        "estado_consenso",
    ] = "PENDIENTE_ADJUDICACION"

    pendientes = anotaciones.loc[
        desacuerdan & ~consenso_valido
    ].copy()

    columnas_adjudicacion = [
        columna
        for columna in [
            "id_registro",
            "evento",
            "texto_original_para_anotar",
            "texto_limpio",
            "location",
            "etiqueta_anotador_1",
            "etiqueta_anotador_2",
            "etiqueta_consenso",
        ]
        if columna in anotaciones.columns
    ]

    desacuerdos_todos = anotaciones.loc[
        desacuerdan,
        columnas_adjudicacion,
    ].copy()

    # Preservar quién adjudicó y las observaciones si ya existían.
    for columna in [
        "adjudicado_por",
        "observacion_adjudicacion",
    ]:
        if columna not in desacuerdos_todos.columns:
            desacuerdos_todos[columna] = ""

    if (
        not adjudicacion_previa.empty
        and "id_registro" in adjudicacion_previa.columns
    ):
        extras = [
            columna
            for columna in [
                "id_registro",
                "adjudicado_por",
                "observacion_adjudicacion",
            ]
            if columna in adjudicacion_previa.columns
        ]
        if len(extras) > 1:
            prev = adjudicacion_previa[extras].copy()
            desacuerdos_todos = (
                desacuerdos_todos
                .drop(
                    columns=[
                        c for c in [
                            "adjudicado_por",
                            "observacion_adjudicacion",
                        ]
                        if c in desacuerdos_todos.columns
                    ],
                    errors="ignore",
                )
                .merge(
                    prev,
                    on="id_registro",
                    how="left",
                )
            )
            for columna in [
                "adjudicado_por",
                "observacion_adjudicacion",
            ]:
                if columna not in desacuerdos_todos.columns:
                    desacuerdos_todos[columna] = ""
                desacuerdos_todos[columna] = (
                    desacuerdos_todos[columna]
                    .fillna("")
                )

    desacuerdos_todos["instruccion"] = (
        "Completar etiqueta_consenso con 1=político o 0=no político. "
        "Opcionalmente registrar adjudicado_por y observacion_adjudicacion."
    )

    hojas["Anotacion"] = anotaciones
    hojas["Desacuerdos_Adjudicar"] = desacuerdos_todos

    instrucciones_actualizadas = pd.DataFrame(
        {
            "Instrucción": [
                "Primera etapa: cada anotador llena solamente su propia columna con 0/1.",
                "No llenar manualmente los acuerdos en etiqueta_consenso.",
                "Ejecutar la etapa de consenso automático del notebook.",
                "Los acuerdos se copian automáticamente a etiqueta_consenso.",
                "Los desacuerdos aparecen en la hoja Desacuerdos_Adjudicar.",
                "Un tercer adjudicador es preferible; alternativamente, uno de los dos anotadores puede decidir tras una revisión explícita.",
                "En Desacuerdos_Adjudicar llenar únicamente etiqueta_consenso con 0/1 para los casos pendientes.",
                "Guardar y cerrar Excel; luego volver a ejecutar la misma celda del notebook.",
                "El pipeline no continuará mientras exista un desacuerdo sin adjudicar.",
                "No modificar id_registro.",
            ]
        }
    )
    hojas["Instrucciones"] = instrucciones_actualizadas

    try:
        with pd.ExcelWriter(
            ruta,
            engine="openpyxl",
            mode="w",
        ) as writer:
            for nombre_hoja, dataframe in hojas.items():
                dataframe.to_excel(
                    writer,
                    sheet_name=nombre_hoja,
                    index=False,
                )
    except PermissionError as exc:
        raise PermissionError(
            "\nNo se pudo actualizar el archivo de anotación:\n"
            f"{ruta}\n\n"
            "Cierre Excel y vuelva a ejecutar esta celda."
        ) from exc

    print(
        f"Acuerdos completados automáticamente: {int(coinciden.sum()):,}"
    )
    print(
        f"Desacuerdos totales entre anotadores: {int(desacuerdan.sum()):,}"
    )
    print(
        f"Desacuerdos pendientes de adjudicación: {len(pendientes):,}"
    )

    return anotaciones, pendientes

In [89]:
def calcular_metricas_silver_hashtags(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """
    Calcula métricas de concordancia entre etiquetas hashtag y predicciones automáticas.

    Además de las métricas usadas en la tesis, incorpora Balanced Accuracy, Cohen's
    kappa y Matthews Correlation Coefficient, útiles cuando la distribución de clases
    del silver standard no es equilibrada.

    Args:
    y_true (np.ndarray): Etiquetas silver derivadas de hashtags.
    y_pred (np.ndarray): Pseudoetiquetas o predicciones de un clasificador.

    Returns:
    dict[str, float]:
    Métricas de la tesis más Balanced Accuracy, Kappa y MCC.

    Example:
    >>> calcular_metricas_silver_hashtags(np.array([0, 1]), np.array([0, 1]))["Cohen_Kappa"]
    1.0
    """
    resultado = calcular_metricas_tesis(np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int))
    resultado.update({
        "Balanced_Accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "Cohen_Kappa": float(cohen_kappa_score(y_true, y_pred)),
        "MCC": float(matthews_corrcoef(y_true, y_pred)),
    })
    return resultado

In [90]:
def prueba_mcnemar_exacta(y_true: np.ndarray, pred_a: np.ndarray, pred_b: np.ndarray) -> pd.DataFrame:
    """
    Ejecuta una prueba exacta de McNemar para dos predictores sobre los mismos tweets.

    La prueba compara únicamente los casos discordantes: registros acertados por A y
    fallados por B frente a registros fallados por A y acertados por B. Resulta adecuada
    para contrastar Escenario 1 y Escenario 2 contra la misma referencia silver.

    Args:
    y_true (np.ndarray): Etiquetas silver de referencia.
    pred_a (np.ndarray): Predicciones del primer escenario.
    pred_b (np.ndarray): Predicciones del segundo escenario.

    Returns:
    pd.DataFrame:
    Conteos discordantes, estadístico basado en la diferencia y p-valor exacto bilateral.

    Example:
    >>> prueba_mcnemar_exacta(np.array([0, 1, 1]), np.array([0, 0, 1]), np.array([0, 1, 0]))
    """
    verdadero = np.asarray(y_true).astype(int)
    a = np.asarray(pred_a).astype(int)
    b = np.asarray(pred_b).astype(int)
    acierto_a = a == verdadero
    acierto_b = b == verdadero
    a_si_b_no = int(np.sum(acierto_a & ~acierto_b))
    a_no_b_si = int(np.sum(~acierto_a & acierto_b))
    discordantes = a_si_b_no + a_no_b_si
    p_valor = 1.0 if discordantes == 0 else float(binomtest(min(a_si_b_no, a_no_b_si), n=discordantes, p=0.5, alternative="two-sided").pvalue)
    return pd.DataFrame([{
        "A_acierta_B_falla": a_si_b_no,
        "A_falla_B_acierta": a_no_b_si,
        "Discordantes": discordantes,
        "Diferencia": int(a_no_b_si - a_si_b_no),
        "p_valor_exacto": p_valor,
    }])

In [91]:
def graficar_matriz_confusion_hashtags_revision(y_true: np.ndarray, y_pred: np.ndarray, titulo: str) -> None:
    """
    Muestra una matriz de confusión interactiva para la validación hashtag.

    La matriz usa la clase 1 como contenido político y la clase 0 como no político.
    Los valores se muestran directamente dentro de cada celda mediante Bokeh.

    Args:
    y_true (np.ndarray): Etiquetas silver de hashtags.
    y_pred (np.ndarray): Pseudoetiquetas o predicciones evaluadas.
    titulo (str): Título visible de la gráfica.

    Returns:
    None:
    La función muestra la figura dentro del notebook.

    Example:
    >>> graficar_matriz_confusion_hashtags_revision(np.array([0, 1]), np.array([0, 1]), "Matriz")
    """
    matriz = confusion_matrix(y_true, y_pred, labels=[0, 1])
    xs = ["Pred 0", "Pred 1", "Pred 0", "Pred 1"]
    ys = ["Real 0", "Real 0", "Real 1", "Real 1"]
    valores = [int(matriz[0, 0]), int(matriz[0, 1]), int(matriz[1, 0]), int(matriz[1, 1])]
    source = ColumnDataSource({"x": xs, "y": ys, "valor": valores, "texto": [str(v) for v in valores]})
    p = figure(x_range=["Pred 0", "Pred 1"], y_range=["Real 1", "Real 0"], width=520, height=360, title=titulo, toolbar_location=None)
    p.rect(x="x", y="y", width=1, height=1, source=source, fill_alpha=0.55, line_color="white")
    p.text(x="x", y="y", text="texto", source=source, text_align="center", text_baseline="middle", text_font_size="16px")
    p.xaxis.axis_label = "Predicción"
    p.yaxis.axis_label = "Silver label por hashtag"
    show(p)

In [92]:
def seleccionar_muestra_auditoria_hashtag(
    silver: pd.DataFrame,
    n_muestra: int,
    seed: int,
) -> pd.DataFrame:
    """
    Selecciona antes del modelado una muestra humana del silver standard.

    Intenta estratificar por evento y etiqueta silver; si algún estrato es demasiado
    pequeño, usa únicamente la etiqueta silver. Esta selección previa evita sesgo de
    post-selección basado en desacuerdos observados con el propio modelo.

    Args:
    silver (pd.DataFrame): Silver standard completo.
    n_muestra (int): Máximo de registros a anotar.
    seed (int): Semilla reproducible.

    Returns:
    pd.DataFrame:
    Muestra ciega reservada para auditoría humana del silver standard.

    Example:
    >>> muestra = seleccionar_muestra_auditoria_hashtag(silver_hashtags, 250, 5)
    >>> len(muestra) <= 250
    True
    """
    if "etiqueta_silver_hashtag" not in silver.columns:
        raise KeyError("El silver standard no contiene `etiqueta_silver_hashtag`.")

    n = min(int(n_muestra), len(silver))
    if n <= 0:
        raise ValueError("No existen registros disponibles para la auditoría hashtag.")

    if n == len(silver):
        return silver.copy().reset_index(drop=True)

    etiqueta = silver["etiqueta_silver_hashtag"].astype(str)
    if "evento" in silver.columns:
        estrato_compuesto = (
            silver["evento"].fillna("Sin evento").astype(str)
            + "||"
            + etiqueta
        )
        conteos = estrato_compuesto.value_counts()
        estrato = estrato_compuesto if int(conteos.min()) >= 2 else etiqueta
    else:
        estrato = etiqueta

    _, muestra = train_test_split(
        silver,
        test_size=n,
        random_state=int(seed),
        stratify=estrato,
    )
    return muestra.reset_index(drop=True)

In [93]:
def guardar_plantilla_auditoria_hashtag(
    muestra: pd.DataFrame,
    ruta: Path,
    version_protocolo: str,
) -> Path:
    """
    Genera la segunda plantilla humana antes de entrenar cualquier modelo.

    El nombre histórico del archivo se conserva por compatibilidad, pero la muestra
    se selecciona antes del entrenamiento. Si existe una plantilla compatible se
    reutiliza. Si existe una versión antigua sin etiquetas, se respalda y se crea
    una nueva. Si contiene trabajo humano, no se sobrescribe.

    El archivo existente se abre mediante un context manager para asegurar que
    Windows libere completamente el recurso antes de intentar renombrarlo.

    Args:
    muestra (pd.DataFrame): Muestra del silver standard seleccionada antes del modelado.
    ruta (Path): Archivo Excel de salida.
    version_protocolo (str): Identificador del protocolo vigente.

    Returns:
    Path:
    Ruta de la plantilla válida creada o reutilizada.

    Example:
    >>> ruta = guardar_plantilla_auditoria_hashtag(
    ...     muestra_auditoria_hashtag,
    ...     Path("data/anotacion_desacuerdos_hashtag.xlsx"),
    ...     "v1",
    ... )
    """
    ruta.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if "id_registro" not in muestra.columns:
        raise KeyError(
            "La muestra no contiene la columna "
            "obligatoria `id_registro`."
        )

    ids_esperados = set(
        muestra["id_registro"]
        .astype(str)
        .str.strip()
    )

    if ruta.exists():
        try:
            with pd.ExcelFile(
                ruta,
                engine="openpyxl",
            ) as xls:

                if "Anotacion" not in xls.sheet_names:
                    existente = pd.DataFrame()
                else:
                    existente = pd.read_excel(
                        xls,
                        sheet_name="Anotacion",
                    )

                if "id_registro" in existente.columns:
                    ids_existentes = set(
                        existente["id_registro"]
                        .astype(str)
                        .str.strip()
                    )
                else:
                    ids_existentes = set()

                tiene_anotaciones = bool(
                    "etiqueta_humana"
                    in existente.columns
                    and existente[
                        "etiqueta_humana"
                    ]
                    .notna()
                    .any()
                )

                version_ok = False

                if "Metadatos" in xls.sheet_names:

                    metadatos_existentes = pd.read_excel(
                        xls,
                        sheet_name="Metadatos",
                    )

                    if {
                        "Clave",
                        "Valor",
                    }.issubset(
                        metadatos_existentes.columns
                    ):

                        mapa_metadatos = dict(
                            zip(
                                metadatos_existentes[
                                    "Clave"
                                ].astype(str),
                                metadatos_existentes[
                                    "Valor"
                                ].astype(str),
                            )
                        )

                        version_ok = (
                            mapa_metadatos.get(
                                "version_protocolo"
                            )
                            == str(version_protocolo)
                        )

        except PermissionError as exc:
            raise PermissionError(
                "\nNo fue posible leer correctamente el archivo:\n"
                f"{ruta}\n\n"
                "El archivo puede estar abierto en Microsoft Excel, "
                "LibreOffice, el panel de vista previa del Explorador "
                "de Windows u otro proceso.\n\n"
                "Cierre completamente el archivo y vuelva a "
                "ejecutar esta celda."
            ) from exc

        if (
            ids_existentes == ids_esperados
            and version_ok
        ):
            print(
                "La plantilla hashtag existente es compatible "
                "con el protocolo actual."
            )
            return ruta

        if tiene_anotaciones:
            raise RuntimeError(
                "\nExiste una plantilla hashtag anterior que "
                "contiene anotaciones humanas:\n"
                f"{ruta}\n\n"
                "No se sobrescribirá automáticamente. "
                "Respáldela manualmente antes de regenerar "
                "la muestra del protocolo actual."
            )

        respaldo = ruta.with_name(
            f"{ruta.stem}_backup_"
            f"{time.strftime('%Y%m%d_%H%M%S')}"
            f"{ruta.suffix}"
        )

        try:
            ruta.replace(respaldo)

        except PermissionError as exc:
            raise PermissionError(
                "\nNo se pudo respaldar la plantilla hashtag "
                "anterior.\n\n"
                f"Archivo origen:\n{ruta}\n\n"
                f"Archivo de respaldo:\n{respaldo}\n\n"
                "Windows indica que el archivo está siendo "
                "utilizado por otro proceso.\n\n"
                "Cierre Excel, cierre cualquier vista previa "
                "del archivo y compruebe que EXCEL.EXE no "
                "continúe ejecutándose."
            ) from exc

        print(
            "Plantilla hashtag antigua sin etiquetas "
            "respaldada en:"
        )
        print(respaldo)

    columnas = [
        columna
        for columna in [
            "id_registro",
            "evento",
            "texto_para_anotacion",
            "texto_limpio",
            "location",
        ]
        if columna in muestra.columns
    ]

    anotacion = muestra[columnas].copy()

    anotacion = anotacion.rename(
        columns={
            "texto_para_anotacion":
            "texto_original_para_anotar"
        }
    )

    anotacion["etiqueta_humana"] = pd.NA
    anotacion["observacion"] = ""

    instrucciones = pd.DataFrame(
        {
            "Instrucción": [
                (
                    "Leer `texto_original_para_anotar` y decidir "
                    "sin ver etiquetas automáticas."
                ),
                (
                    "Etiquetar TODAS las filas: "
                    "1 = político, 0 = no político."
                ),
                "No modificar id_registro.",
                (
                    "El nombre del archivo se conserva por "
                    "compatibilidad; la muestra fue seleccionada "
                    "antes del modelado."
                ),
                (
                    "Después del modelado se añadirá una hoja "
                    "Auditoria_Modelo con silver y predicciones."
                ),
            ]
        }
    )

    metadatos = pd.DataFrame(
        {
            "Clave": [
                "version_protocolo",
                "tipo_muestra",
                "n_registros",
                "semilla",
            ],
            "Valor": [
                str(version_protocolo),
                (
                    "auditoría humana estratificada del "
                    "silver standard previa al modelado"
                ),
                str(len(anotacion)),
                str(RANDOM_STATE),
            ],
        }
    )

    try:
        with pd.ExcelWriter(
            ruta,
            engine="openpyxl",
            mode="w",
        ) as writer:

            anotacion.to_excel(
                writer,
                sheet_name="Anotacion",
                index=False,
            )

            instrucciones.to_excel(
                writer,
                sheet_name="Instrucciones",
                index=False,
            )

            metadatos.to_excel(
                writer,
                sheet_name="Metadatos",
                index=False,
            )

    except PermissionError as exc:
        raise PermissionError(
            "\nNo se pudo crear la nueva plantilla hashtag:\n"
            f"{ruta}\n\n"
            "El archivo continúa bloqueado por Windows. "
            "Cierre Excel y cualquier aplicación que lo "
            "esté utilizando antes de volver a ejecutar "
            "esta función."
        ) from exc

    print(
        "Plantilla de auditoría hashtag creada "
        "correctamente:"
    )
    print(ruta)

    return ruta

In [94]:
def cargar_anotacion_hashtag_humana(ruta: Path) -> pd.DataFrame:
    """
    Carga y valida la segunda plantilla de anotación humana.

    Convierte `etiqueta_humana` a entero nullable y rechaza cualquier valor distinto
    de 0/1. Las filas vacías se conservan para que la comprobación de completitud pueda
    detener el pipeline antes del modelado.

    Args:
    ruta (Path): Ruta de `anotacion_desacuerdos_hashtag.xlsx`.

    Returns:
    pd.DataFrame:
    Tabla de anotaciones hashtag con columna `etiqueta_humana` validada.

    Example:
    >>> tabla = cargar_anotacion_hashtag_humana(
    ...     Path("data/anotacion_desacuerdos_hashtag.xlsx")
    ... )
    """
    if not ruta.exists():
        raise FileNotFoundError(f"No existe el archivo de anotación hashtag: {ruta}")

    tabla = pd.read_excel(ruta, sheet_name="Anotacion")
    requeridas = {"id_registro", "etiqueta_humana"}
    faltantes = requeridas.difference(tabla.columns)
    if faltantes:
        raise KeyError(f"Faltan columnas requeridas: {sorted(faltantes)}")

    valores = pd.to_numeric(tabla["etiqueta_humana"], errors="coerce")
    invalidos = valores.dropna()[~valores.dropna().isin([0, 1])]
    if not invalidos.empty:
        raise ValueError("`etiqueta_humana` contiene valores diferentes de 0/1.")

    tabla["etiqueta_humana"] = valores.astype("Int64")
    return tabla

In [95]:
def auditar_completitud_anotaciones(
    ruta_manual: Path,
    ruta_hashtag: Path,
) -> pd.DataFrame:
    """
    Resume el estado de las dos anotaciones humanas y de la adjudicación.

    En el archivo principal distingue las dos anotaciones independientes, los acuerdos
    automáticos, los desacuerdos, los consensos finales y los desacuerdos aún sin
    adjudicar. En el segundo archivo controla `etiqueta_humana`.

    Args:
    ruta_manual (Path): Plantilla principal de dos anotadores.
    ruta_hashtag (Path): Plantilla de auditoría humana del silver standard.

    Returns:
    pd.DataFrame:
    Tabla de estado con totales, completados y pendientes.

    Example:
    >>> estado = auditar_completitud_anotaciones(
    ...     RUTA_ANOTACION_MANUAL,
    ...     RUTA_ANOTACION_DESACUERDOS_HASHTAG,
    ... )
    """
    filas: list[dict[str, Any]] = []

    if ruta_manual.exists():
        try:
            with pd.ExcelFile(
                ruta_manual,
                engine="openpyxl",
            ) as xls:
                manual = pd.read_excel(
                    xls,
                    sheet_name="Anotacion",
                )
        except PermissionError as exc:
            raise PermissionError(
                f"Cierre el archivo antes de auditarlo: {ruta_manual}"
            ) from exc

        total = len(manual)

        for columna in [
            "etiqueta_anotador_1",
            "etiqueta_anotador_2",
        ]:
            completas = (
                int(manual[columna].notna().sum())
                if columna in manual.columns
                else 0
            )
            filas.append({
                "Archivo": ruta_manual.name,
                "Etapa": columna,
                "Total": total,
                "Completas": completas,
                "Pendientes": int(total - completas),
            })

        if {
            "etiqueta_anotador_1",
            "etiqueta_anotador_2",
        }.issubset(manual.columns):
            a1 = pd.to_numeric(
                manual["etiqueta_anotador_1"],
                errors="coerce",
            )
            a2 = pd.to_numeric(
                manual["etiqueta_anotador_2"],
                errors="coerce",
            )
            ambos = a1.notna() & a2.notna()
            acuerdos = ambos & (a1 == a2)
            desacuerdos = ambos & (a1 != a2)
        else:
            acuerdos = pd.Series(False, index=manual.index)
            desacuerdos = pd.Series(False, index=manual.index)

        consenso = (
            pd.to_numeric(
                manual["etiqueta_consenso"],
                errors="coerce",
            )
            if "etiqueta_consenso" in manual.columns
            else pd.Series(np.nan, index=manual.index)
        )

        consensos_validos = consenso.isin([0, 1])
        pendientes_adjudicacion = desacuerdos & ~consensos_validos

        filas.extend([
            {
                "Archivo": ruta_manual.name,
                "Etapa": "acuerdos_automaticos",
                "Total": int(acuerdos.sum()),
                "Completas": int(
                    (acuerdos & consensos_validos).sum()
                ),
                "Pendientes": int(
                    (acuerdos & ~consensos_validos).sum()
                ),
            },
            {
                "Archivo": ruta_manual.name,
                "Etapa": "desacuerdos_entre_anotadores",
                "Total": int(desacuerdos.sum()),
                "Completas": int(
                    (desacuerdos & consensos_validos).sum()
                ),
                "Pendientes": int(
                    pendientes_adjudicacion.sum()
                ),
            },
            {
                "Archivo": ruta_manual.name,
                "Etapa": "etiqueta_consenso_final",
                "Total": total,
                "Completas": int(consensos_validos.sum()),
                "Pendientes": int(
                    total - consensos_validos.sum()
                ),
            },
        ])
    else:
        filas.append({
            "Archivo": ruta_manual.name,
            "Etapa": "ARCHIVO",
            "Total": 0,
            "Completas": 0,
            "Pendientes": 1,
        })

    if ruta_hashtag.exists():
        try:
            with pd.ExcelFile(
                ruta_hashtag,
                engine="openpyxl",
            ) as xls:
                hash_df = pd.read_excel(
                    xls,
                    sheet_name="Anotacion",
                )
        except PermissionError as exc:
            raise PermissionError(
                f"Cierre el archivo antes de auditarlo: {ruta_hashtag}"
            ) from exc

        total_hash = len(hash_df)
        completas_hash = (
            int(hash_df["etiqueta_humana"].notna().sum())
            if "etiqueta_humana" in hash_df.columns
            else 0
        )
        filas.append({
            "Archivo": ruta_hashtag.name,
            "Etapa": "etiqueta_humana",
            "Total": total_hash,
            "Completas": completas_hash,
            "Pendientes": int(
                total_hash - completas_hash
            ),
        })
    else:
        filas.append({
            "Archivo": ruta_hashtag.name,
            "Etapa": "ARCHIVO",
            "Total": 0,
            "Completas": 0,
            "Pendientes": 1,
        })

    return pd.DataFrame(filas)

In [96]:
def exigir_anotaciones_completas(
    ruta_manual: Path,
    ruta_hashtag: Path,
    holdout_manual: pd.DataFrame,
    muestra_hashtag: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Bloquea el modelado hasta completar anotaciones, consenso y adjudicación.

    Verifica que ambos anotadores hayan clasificado todos los tweets, que los acuerdos
    tengan consenso automático coherente, que todos los desacuerdos hayan sido
    adjudicados con 0/1 y que la segunda plantilla humana también esté completa.

    Args:
    ruta_manual (Path): Archivo principal de anotación humana.
    ruta_hashtag (Path): Archivo de auditoría humana del silver standard.
    holdout_manual (pd.DataFrame): Holdout esperado del primer archivo.
    muestra_hashtag (pd.DataFrame): Muestra esperada del segundo archivo.

    Returns:
    tuple[pd.DataFrame, pd.DataFrame]:
    Anotaciones principal y hashtag validadas para continuar con el modelado.

    Example:
    >>> manual_ok, hashtag_ok = exigir_anotaciones_completas(
    ...     RUTA_ANOTACION_MANUAL,
    ...     RUTA_ANOTACION_DESACUERDOS_HASHTAG,
    ...     holdout_manual,
    ...     muestra_auditoria_hashtag,
    ... )
    """
    manual = cargar_anotaciones_manuales(ruta_manual)
    hashtag = cargar_anotacion_hashtag_humana(ruta_hashtag)

    ids_manual_esperados = set(
        holdout_manual["id_registro"].astype(str)
    )
    ids_manual = set(
        manual["id_registro"].astype(str)
    )
    if ids_manual != ids_manual_esperados:
        raise RuntimeError(
            "Los IDs de anotacion_manual_politica.xlsx no coinciden "
            "con el holdout reservado por este protocolo."
        )

    ids_hash_esperados = set(
        muestra_hashtag["id_registro"].astype(str)
    )
    ids_hash = set(
        hashtag["id_registro"].astype(str)
    )
    if ids_hash != ids_hash_esperados:
        raise RuntimeError(
            "Los IDs de anotacion_desacuerdos_hashtag.xlsx no coinciden "
            "con la muestra reservada antes del modelado."
        )

    pendientes_a1 = int(
        manual["etiqueta_anotador_1"].isna().sum()
    )
    pendientes_a2 = int(
        manual["etiqueta_anotador_2"].isna().sum()
    )
    pendientes_hash = int(
        hashtag["etiqueta_humana"].isna().sum()
    )

    if pendientes_a1 > 0 or pendientes_a2 > 0:
        raise RuntimeError(
            "ANOTACIÓN INDEPENDIENTE INCOMPLETA. "
            f"Pendientes anotador 1={pendientes_a1}; "
            f"anotador 2={pendientes_a2}. "
            "Ambos anotadores deben terminar antes de calcular consenso."
        )

    if pendientes_hash > 0:
        raise RuntimeError(
            "ANOTACIÓN HASHTAG INCOMPLETA. "
            f"Faltan {pendientes_hash} etiquetas humanas en "
            "anotacion_desacuerdos_hashtag.xlsx."
        )

    a1 = manual["etiqueta_anotador_1"].astype(int)
    a2 = manual["etiqueta_anotador_2"].astype(int)
    consenso = manual["etiqueta_consenso"]

    coinciden = a1 == a2
    desacuerdan = a1 != a2

    incoherentes = manual.loc[
        coinciden
        & (
            consenso.isna()
            | (
                consenso.astype("Int64")
                != a1.astype("Int64")
            )
        )
    ]
    if not incoherentes.empty:
        raise RuntimeError(
            "Hay acuerdos entre anotadores cuyo consenso automático "
            "no está completo o no coincide. Vuelva a ejecutar "
            "`actualizar_consenso_automatico_excel()`."
        )

    pendientes_adjudicacion = manual.loc[
        desacuerdan
        & ~consenso.isin([0, 1])
    ]

    if not pendientes_adjudicacion.empty:
        raise RuntimeError(
            "ADJUDICACIÓN PENDIENTE. Existen "
            f"{len(pendientes_adjudicacion)} desacuerdos sin decisión final. "
            "Abra `anotacion_manual_politica.xlsx`, hoja "
            "`Desacuerdos_Adjudicar`, y complete `etiqueta_consenso` "
            "con 1=político o 0=no político. La decisión puede realizarla "
            "preferiblemente una tercera persona, o uno de los dos anotadores "
            "tras una revisión explícita. Guarde, cierre Excel y vuelva a "
            "ejecutar la etapa de consenso."
        )

    if manual["etiqueta_consenso"].isna().any():
        raise RuntimeError(
            "Todavía existen consensos vacíos. No puede comenzar la Sección 5."
        )

    if manual["etiqueta_consenso"].astype(int).nunique() < 2:
        raise RuntimeError(
            "El ground truth final debe contener las clases 0 y 1."
        )

    if hashtag["etiqueta_humana"].astype(int).nunique() < 2:
        raise RuntimeError(
            "La auditoría hashtag humana debe contener las clases 0 y 1."
        )

    return manual, hashtag

In [97]:
inventario_hashtags = construir_inventario_hashtags_revision(
    tweets_revision,
    HASHTAGS_POLITICOS_ESTRICTOS,
    HASHTAGS_NO_POLITICOS_ESTRICTOS,
)

In [98]:
silver_hashtags = construir_silver_standard_hashtags(
    tweets_revision,
    HASHTAGS_POLITICOS_ESTRICTOS,
    HASHTAGS_NO_POLITICOS_ESTRICTOS,
)

In [99]:
ids_silver = set(silver_hashtags["id_registro"].astype(str))
restantes_sin_silver = tweets_revision[
    ~tweets_revision["id_registro"].astype(str).isin(ids_silver)
].reset_index(drop=True)

In [100]:
n_manual = min(
    N_ANOTACION_MANUAL,
    max(20, len(restantes_sin_silver) // 10),
)
tweets_desarrollo, holdout_manual = separar_holdout_anotacion_manual(
    restantes_sin_silver,
    n_manual,
    RANDOM_STATE,
)

In [101]:
muestra_auditoria_hashtag = seleccionar_muestra_auditoria_hashtag(
    silver_hashtags,
    N_MANUAL_DESACUERDOS_HASHTAG,
    RANDOM_STATE,
)

In [102]:
ruta_plantilla_manual = guardar_plantilla_anotacion_manual(
    holdout_manual,
    RUTA_ANOTACION_MANUAL,
    VERSION_PROTOCOLO_ANOTACION,
)

Plantilla antigua sin etiquetas respaldada en: C:\Users\LABIA\Documents\Tesis cleo EPN\Codigo\data\anotacion_manual_politica_backup_20260915_092513.xlsx
Plantilla de anotación humana creada correctamente: C:\Users\LABIA\Documents\Tesis cleo EPN\Codigo\data\anotacion_manual_politica.xlsx


In [103]:
ruta_plantilla_hashtag = guardar_plantilla_auditoria_hashtag(
    muestra_auditoria_hashtag,
    RUTA_ANOTACION_DESACUERDOS_HASHTAG,
    VERSION_PROTOCOLO_ANOTACION,
)

Plantilla hashtag antigua sin etiquetas respaldada en:
C:\Users\LABIA\Documents\Tesis cleo EPN\Codigo\data\anotacion_desacuerdos_hashtag_backup_20260915_092514.xlsx
Plantilla de auditoría hashtag creada correctamente:
C:\Users\LABIA\Documents\Tesis cleo EPN\Codigo\data\anotacion_desacuerdos_hashtag.xlsx


In [104]:
tweets_desarrollo_ciego_hashtags = preparar_corpus_ciego_hashtags(
    tweets_desarrollo
)
silver_hashtags_ciego = preparar_corpus_ciego_hashtags(
    silver_hashtags
)
trends_ciegos_hashtags = preparar_corpus_ciego_hashtags(
    trends
)

In [105]:
resumen_silver = (
    silver_hashtags["etiqueta_silver_hashtag"]
    .value_counts()
    .sort_index()
    .rename_axis("Etiqueta_silver")
    .reset_index(name="N")
)
resumen_silver["Clase"] = resumen_silver["Etiqueta_silver"].map(
    {0: "No político", 1: "Político"}
)
resumen_silver["Porcentaje"] = (
    resumen_silver["N"] / resumen_silver["N"].sum() * 100.0
)

In [106]:
print(f"Corpus final deduplicado: {len(tweets_revision):,}")
print(f"Silver standard completo: {len(silver_hashtags):,}")
print(f"Holdout manual independiente: {len(holdout_manual):,}")
print(f"Muestra humana de auditoría hashtag: {len(muestra_auditoria_hashtag):,}")
print(f"Corpus de desarrollo: {len(tweets_desarrollo):,}")
print(f"Archivo 1: {ruta_plantilla_manual}")
print(f"Archivo 2: {ruta_plantilla_hashtag}")

Corpus final deduplicado: 138,400
Silver standard completo: 4,240
Holdout manual independiente: 600
Muestra humana de auditoría hashtag: 250
Corpus de desarrollo: 133,560
Archivo 1: C:\Users\LABIA\Documents\Tesis cleo EPN\Codigo\data\anotacion_manual_politica.xlsx
Archivo 2: C:\Users\LABIA\Documents\Tesis cleo EPN\Codigo\data\anotacion_desacuerdos_hashtag.xlsx


In [107]:
mostrar_tabla_bokeh(
    resumen_silver,
    "Distribución del silver standard basado en hashtags",
    alto=180,
)

In [108]:
mostrar_tabla_bokeh(
    inventario_hashtags.head(80),
    "Inventario de hashtags — 80 más frecuentes",
    ancho=1250,
    alto=420,
    max_filas=80,
)

## Realizar anotaciones humanas

In [109]:
ESTADO_ANTES_CONSENSO = auditar_completitud_anotaciones(
    RUTA_ANOTACION_MANUAL,
    RUTA_ANOTACION_DESACUERDOS_HASHTAG,
)

In [110]:
mostrar_tabla_bokeh(
    ESTADO_ANTES_CONSENSO,
    "Estado de las anotaciones antes del consenso/adjudicación",
    ancho=1250,
    alto=330,
    max_filas=20,
)

In [111]:
pendientes_iniciales = ESTADO_ANTES_CONSENSO.loc[
    ESTADO_ANTES_CONSENSO["Etapa"].isin(
        [
            "etiqueta_anotador_1",
            "etiqueta_anotador_2",
            "etiqueta_humana",
        ]
    ),
    "Pendientes",
].sum()

In [112]:
if int(pendientes_iniciales) > 0:
    raise RuntimeError(
        "Aún faltan etiquetas de los dos anotadores "
        "o de la auditoría hashtag. Complete ambas plantillas, guarde y cierre "
        "Excel, y vuelva a ejecutar esta celda."
    )

In [113]:
ANOTACIONES_CON_CONSENSO, DESACUERDOS_PENDIENTES = (
    actualizar_consenso_automatico_excel(
        RUTA_ANOTACION_MANUAL
    )
)

Acuerdos completados automáticamente: 513
Desacuerdos totales entre anotadores: 87
Desacuerdos pendientes de adjudicación: 87


In [114]:
ACUERDO_PREVIO_ANOTADORES = calcular_acuerdo_anotadores(
    ANOTACIONES_CON_CONSENSO
)

In [115]:
mostrar_tabla_bokeh(
    ACUERDO_PREVIO_ANOTADORES,
    "Acuerdo entre los dos anotadores antes de adjudicación",
    alto=180,
)

In [116]:
ESTADO_DESPUES_CONSENSO = auditar_completitud_anotaciones(
    RUTA_ANOTACION_MANUAL,
    RUTA_ANOTACION_DESACUERDOS_HASHTAG,
)

In [117]:
mostrar_tabla_bokeh(
    ESTADO_DESPUES_CONSENSO,
    "Estado después del consenso automático",
    ancho=1250,
    alto=330,
    max_filas=20,
)

In [118]:
if not DESACUERDOS_PENDIENTES.empty:
    columnas_mostrar = [
        columna
        for columna in [
            "id_registro",
            "texto_original_para_anotar",
            "etiqueta_anotador_1",
            "etiqueta_anotador_2",
            "etiqueta_consenso",
        ]
        if columna in DESACUERDOS_PENDIENTES.columns
    ]

    mostrar_tabla_bokeh(
        DESACUERDOS_PENDIENTES[columnas_mostrar],
        "Desacuerdos pendientes de adjudicación humana",
        ancho=1500,
        alto=480,
        max_filas=100,
    )

    raise RuntimeError(
        "PAUSA DE ADJUDICACIÓN. El consenso automático terminó, pero quedan "
        f"{len(DESACUERDOS_PENDIENTES)} desacuerdos. Abra "
        "`anotacion_manual_politica.xlsx`, hoja `Desacuerdos_Adjudicar`, "
        "complete `etiqueta_consenso` con 0/1, guarde, cierre Excel y vuelva "
        "a ejecutar ESTA MISMA CELDA. El notebook no debe continuar todavía."
    )

RuntimeError: PAUSA DE ADJUDICACIÓN. El consenso automático terminó, pero quedan 87 desacuerdos. Abra `anotacion_manual_politica.xlsx`, hoja `Desacuerdos_Adjudicar`, complete `etiqueta_consenso` con 0/1, guarde, cierre Excel y vuelva a ejecutar ESTA MISMA CELDA. El notebook no debe continuar todavía.

In [119]:
ANOTACIONES_MANUALES_PREVALIDADAS, ANOTACIONES_HASHTAG_PREVALIDADAS = (
    exigir_anotaciones_completas(
        RUTA_ANOTACION_MANUAL,
        RUTA_ANOTACION_DESACUERDOS_HASHTAG,
        holdout_manual,
        muestra_auditoria_hashtag,
    )
)

In [120]:
ESTADO_FINAL_ANOTACIONES = auditar_completitud_anotaciones(
    RUTA_ANOTACION_MANUAL,
    RUTA_ANOTACION_DESACUERDOS_HASHTAG,
)

In [121]:
mostrar_tabla_bokeh(
    ESTADO_FINAL_ANOTACIONES,
    "Validación final de anotaciones antes del modelado",
    ancho=1250,
    alto=330,
    max_filas=20,
)

# 5. Representación semántica, clustering y expansión léxica

In [122]:
def entrenar_word2vec(corpus: Sequence[Sequence[str]], dimension: int, window: int, min_count: int, epochs: int, seed: int) -> Word2Vec:
    """
    Entrena un modelo Word2Vec con la dimensión solicitada.

    Reproduce la representación vectorial utilizada en la tesis para convertir las
    palabras del corpus en embeddings. Se usa Skip-Gram para favorecer relaciones
    semánticas en un corpus con términos relativamente dispersos.

    Args:
    corpus (Sequence[Sequence[str]]): Documentos tokenizados utilizados en el entrenamiento.
    dimension (int): Número de dimensiones del vector de cada palabra.
    window (int): Tamaño de la ventana contextual de Word2Vec.
    min_count (int): Frecuencia mínima para conservar una palabra.
    epochs (int): Número de épocas de entrenamiento.
    seed (int): Semilla para favorecer la reproducibilidad.

    Returns:
    Word2Vec:
    Modelo Word2Vec entrenado y listo para consultar vectores y similitudes.

    Example:
    >>> modelo = entrenar_word2vec(corpus, 50, 5, 1, 10, 5)
    """
    corpus_util = [list(doc) for doc in corpus if len(doc) > 0]
    if not corpus_util:
        raise ValueError("El corpus está vacío; no se puede entrenar Word2Vec.")
    return Word2Vec(
        sentences=corpus_util,
        vector_size=dimension,
        window=window,
        min_count=min_count,
        sg=1,
        workers=W2V_WORKERS,
        seed=seed,
        epochs=epochs,
    )

In [159]:
def obtener_palabras_trend_unicas(
    tokens_trends: Sequence[Sequence[str]],
    modelo: Word2Vec,
) -> list[str]:
    """
    Obtiene las palabras únicas de los trends presentes en el vocabulario Word2Vec.

    Solo conserva términos que fueron aprendidos por Word2Vec en el conjunto de
    entrenamiento. Además, verifica que existan suficientes palabras para evaluar
    correctamente todos los valores candidatos de k definidos en K_CANDIDATOS.

    Debido a que el coeficiente Silhouette requiere que el número de clústeres sea
    menor que el número de muestras, se exige al menos max(K_CANDIDATOS) + 1
    palabras disponibles.

    Args:
    tokens_trends (Sequence[Sequence[str]]): Trends tokenizados después de la limpieza.
    modelo (Word2Vec): Modelo Word2Vec entrenado exclusivamente con el conjunto train.

    Returns:
    list[str]:
    Lista ordenada de palabras únicas de trends presentes en el vocabulario Word2Vec.

    Example:
    >>> palabras = obtener_palabras_trend_unicas(tokens_trends, modelo)
    >>> isinstance(palabras, list)
    True
    """
    palabras = sorted(
        {
            token
            for documento in tokens_trends
            for token in documento
            if token in modelo.wv
        }
    )

    if not K_CANDIDATOS:
        raise ValueError(
            "K_CANDIDATOS está vacío. Debe contener al menos "
            "un valor de k para evaluar el clustering."
        )

    k_maximo = max(K_CANDIDATOS)

    minimo_requerido = k_maximo + 1

    if len(palabras) < minimo_requerido:
        raise ValueError(
            "No existen suficientes palabras de trends presentes "
            "en el vocabulario Word2Vec para evaluar todos los "
            "valores candidatos de k.\n"
            f"Palabras disponibles: {len(palabras)}\n"
            f"Mayor k candidato: {k_maximo}\n"
            f"Mínimo requerido: {minimo_requerido}"
        )

    return palabras

In [124]:
def obtener_matriz_palabras(palabras: Sequence[str], modelo: Word2Vec) -> np.ndarray:
    """
    Construye la matriz de embeddings de una secuencia de palabras.

    Recupera el vector Word2Vec de cada término conservando el orden de entrada. La
    matriz resultante se utiliza para K-Means, PCA y cálculo de centroides.

    Args:
    palabras (Sequence[str]): Términos presentes en el vocabulario Word2Vec.
    modelo (Word2Vec): Modelo que contiene los vectores entrenados.

    Returns:
    np.ndarray:
    Matriz de forma `(n_palabras, dimension_embedding)`.

    Example:
    >>> obtener_matriz_palabras(["ecuador"], modelo).shape[0]
    1
    """
    return np.vstack([modelo.wv[p] for p in palabras]).astype(np.float32)

In [125]:
def entrenar_kmeans_palabras(vectores: np.ndarray, k: int, seed: int) -> tuple[KMeans, np.ndarray]:
    """
    Agrupa los vectores de palabras mediante K-Means.

    Utiliza inicialización k-means++, un máximo de 100 iteraciones y una semilla fija,
    reproduciendo la configuración general descrita en la tesis para formar los
    clústeres semánticos.

    Args:
    vectores (np.ndarray): Matriz de embeddings de palabras.
    k (int): Número de clústeres que se desea formar.
    seed (int): Semilla de aleatoriedad.

    Returns:    tuple[KMeans, np.ndarray]:
    Modelo K-Means ajustado y vector de etiqueta de clúster por palabra.

    Example:
    >>> kmeans, labels = entrenar_kmeans_palabras(vectores, 4, 5)
    """
    modelo = KMeans(n_clusters=k, init="k-means++", n_init=20, max_iter=300, random_state=seed)
    etiquetas = modelo.fit_predict(vectores)
    return modelo, etiquetas.astype(int)

In [126]:
def resumir_clusters(palabras: Sequence[str], etiquetas: np.ndarray) -> pd.DataFrame:
    """
    Resume el tamaño y una muestra de términos de cada clúster.

    Agrupa las palabras según la etiqueta generada por K-Means y presenta el número de
    elementos y una muestra lexicográfica para facilitar la inspección semántica.

    Args:
    palabras (Sequence[str]): Lista de palabras alineada con las etiquetas.
    etiquetas (np.ndarray): Identificador de clúster de cada palabra.

    Returns:
    pd.DataFrame:
    Tabla con clúster, número de palabras y muestra de términos.

    Example:
    >>> resumir_clusters(palabras, etiquetas).head()
    """
    filas = []
    for cluster_id in sorted(np.unique(etiquetas)):
        terminos = sorted([p for p, c in zip(palabras, etiquetas) if int(c) == int(cluster_id)])
        filas.append({
            "Cluster": int(cluster_id),
            "Numero_palabras": len(terminos),
            "Muestra": ", ".join(terminos[:30]),
        })
    return pd.DataFrame(filas)

In [127]:
def seleccionar_cluster_politico(palabras: Sequence[str], etiquetas: np.ndarray, vectores: np.ndarray, modelo: Word2Vec, semillas: Sequence[str]) -> tuple[int, pd.DataFrame]:
    """
    Identifica de forma reproducible el clúster con mayor afinidad política.

    La tesis identifica el clúster político mediante inspección del contenido. Debido
    a que los números de clúster de K-Means son arbitrarios entre ejecuciones, esta
    función calcula para cada clúster el número de semillas políticas contenidas y la
    similitud coseno promedio entre su centroide y las semillas disponibles en Word2Vec.
    El clúster con mayor puntuación se selecciona como político.

    Args:
    palabras (Sequence[str]): Palabras agrupadas por K-Means.
    etiquetas (np.ndarray): Etiqueta de clúster por palabra.
    vectores (np.ndarray): Vectores de las palabras.
    modelo (Word2Vec): Modelo Word2Vec usado para recuperar semillas semánticas.
    semillas (Sequence[str]): Términos políticos utilizados como referencia.

    Returns:
    tuple[int, pd.DataFrame]:
    Identificador del clúster político y tabla de puntuaciones por clúster.

    Example:
    >>> cluster_politico, scores = seleccionar_cluster_politico(palabras, labels, vectores, modelo, semillas)
    """
    semillas_limpias = list(dict.fromkeys(str(s).strip().lower() for s in semillas if str(s).strip()))
    mapa_cluster = {p: int(c) for p, c in zip(palabras, etiquetas)}
    semillas_vector = [s for s in semillas_limpias if s in modelo.wv]
    filas = []

    for cluster_id in sorted(np.unique(etiquetas)):
        mask = etiquetas == cluster_id
        centroide = vectores[mask].mean(axis=0)
        norm_c = np.linalg.norm(centroide) or 1.0
        similitudes = []
        for semilla in semillas_vector:
            v = modelo.wv[semilla]
            denom = norm_c * (np.linalg.norm(v) or 1.0)
            similitudes.append(float(np.dot(centroide, v) / denom))
        coincidencias = sum(1 for s in semillas_limpias if mapa_cluster.get(s) == int(cluster_id))
        sim_media = float(np.mean(similitudes)) if similitudes else 0.0
        score = coincidencias * 2.0 + sim_media
        filas.append({
            "Cluster": int(cluster_id),
            "Semillas_en_cluster": int(coincidencias),
            "Similitud_media_semillas": sim_media,
            "Score_politico": score,
        })

    scores = pd.DataFrame(filas).sort_values(["Score_politico", "Semillas_en_cluster"], ascending=False).reset_index(drop=True)
    return int(scores.iloc[0]["Cluster"]), scores

In [128]:
def calcular_centroides(vectores: np.ndarray, etiquetas: np.ndarray, k: int) -> np.ndarray:
    """
    Recalcula los centroides de los clústeres a partir de sus palabras actuales.

    Esta función se utiliza tanto en el Escenario 1 como después de mover palabras al
    clúster político en el Escenario 2. Cada centroide corresponde al promedio de los
    vectores pertenecientes al clúster.

    Args:
    vectores (np.ndarray): Matriz de vectores de palabras.
    etiquetas (np.ndarray): Etiqueta de clúster asignada a cada palabra.
    k (int): Número total de clústeres.

    Returns:
    np.ndarray:
    Matriz de centroides de forma `(k, dimension_embedding)`.

    Example:
    >>> centroides = calcular_centroides(vectores, labels, 4)
    """
    centroides = []
    global_mean = vectores.mean(axis=0)
    for cluster_id in range(k):
        mask = etiquetas == cluster_id
        centroides.append(vectores[mask].mean(axis=0) if np.any(mask) else global_mean)
    return np.vstack(centroides).astype(np.float32)

In [129]:
def vectorizar_documentos(tokens_documentos: Sequence[Sequence[str]], modelo: Word2Vec, agregacion: str = "suma") -> np.ndarray:
    """
    Convierte cada documento en un único vector a partir de sus palabras.

    Para seguir la tesis, la agregación predeterminada es la suma de los vectores de
    todas las palabras del tweet. También se admite `media` para realizar controles
    experimentales. Las palabras fuera del vocabulario se ignoran y los documentos
    sin palabras válidas reciben un vector de ceros.

    Args:
    tokens_documentos (Sequence[Sequence[str]]): Documentos tokenizados.
    modelo (Word2Vec): Modelo Word2Vec que contiene los vectores de palabras.
    agregacion (str): Estrategia de agregación: `suma` o `media`.
    *Valor por defecto: "suma".*

    Returns:
    np.ndarray:
    Matriz con un vector por documento.

    Example:
    >>> X = vectorizar_documentos(tokens_tweets, modelo, "suma")
    """
    dimension = modelo.vector_size
    salida = np.zeros((len(tokens_documentos), dimension), dtype=np.float32)
    for i, doc in enumerate(tokens_documentos):
        validos = [modelo.wv[t] for t in doc if t in modelo.wv]
        if not validos:
            continue
        matriz = np.vstack(validos)
        salida[i] = matriz.mean(axis=0) if agregacion == "media" else matriz.sum(axis=0)
    return salida

In [130]:
def etiquetar_por_centroides(vectores_tweets: np.ndarray, centroides: np.ndarray, cluster_politico: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Etiqueta tweets según el centroide de clúster más cercano por distancia coseno.

    Normaliza tweets y centroides, calcula todas las similitudes coseno de manera
    vectorizada y asigna etiqueta 1 cuando el clúster más cercano es el político.
    También devuelve el clúster asignado y la distancia específica al centroide político.

    Args:
    vectores_tweets (np.ndarray): Matriz de vectores agregados de tweets.
    centroides (np.ndarray): Matriz de centroides de los clústeres.
    cluster_politico (int): Identificador del clúster considerado político.

    Returns:
    tuple[np.ndarray, np.ndarray, np.ndarray]:
    Etiqueta binaria, clúster más cercano y distancia coseno al clúster político.

    Example:
    >>> labels, clusters, dist = etiquetar_por_centroides(X, centroides, 2)
    """
    eps = 1e-12
    norma_x = np.linalg.norm(vectores_tweets, axis=1, keepdims=True)
    norma_c = np.linalg.norm(centroides, axis=1, keepdims=True).T
    similitudes = (vectores_tweets @ centroides.T) / np.maximum(norma_x * norma_c, eps)
    distancias = 1.0 - similitudes

    vacios = norma_x.ravel() <= eps
    cluster_cercano = np.argmin(distancias, axis=1).astype(int)
    cluster_cercano[vacios] = -1
    etiquetas = (cluster_cercano == int(cluster_politico)).astype(int)
    distancia_politica = distancias[:, int(cluster_politico)].astype(np.float32)
    distancia_politica[vacios] = np.nan
    return etiquetas, cluster_cercano, distancia_politica

In [131]:
def evaluar_kmeans_objetivo(vectores: np.ndarray, valores_k: Sequence[int], seed: int) -> pd.DataFrame:
    """
    Evalúa K-Means con métricas internas objetivas para varios valores de k.

    Además de la inercia empleada por el método del codo, calcula Silhouette,
    Davies–Bouldin y Calinski–Harabasz. Esto permite documentar la elección de k con
    criterios cuantitativos y no únicamente mediante inspección semántica.

    Args:
    vectores (np.ndarray): Matriz de embeddings de palabras.
    valores_k (Sequence[int]): Valores de k que se desean comparar.
    seed (int): Semilla de K-Means.

    Returns:
    pd.DataFrame:
    Métricas internas y tiempo de ajuste para cada valor de k.

    Example:
    >>> tabla_k = evaluar_kmeans_objetivo(vectores, range(2, 8), 5)
    """
    filas: list[dict[str, float | int]] = []
    for k in valores_k:
        if k < 2 or k >= len(vectores):
            continue
        inicio = time.perf_counter()
        modelo = KMeans(
            n_clusters=int(k),
            init="k-means++",
            n_init=20,
            max_iter=300,
            random_state=seed,
        )
        labels = modelo.fit_predict(vectores)
        tiempo = time.perf_counter() - inicio
        filas.append({
            "k": int(k),
            "Inercia": float(modelo.inertia_),
            "Silhouette": float(silhouette_score(vectores, labels)),
            "Davies_Bouldin": float(davies_bouldin_score(vectores, labels)),
            "Calinski_Harabasz": float(calinski_harabasz_score(vectores, labels)),
            "Tiempo_s": float(tiempo),
        })
    return pd.DataFrame(filas)

In [132]:
def seleccionar_k_objetivo(tabla_k: pd.DataFrame) -> int:
    """
    Selecciona k maximizando Silhouette y usando Davies–Bouldin como desempate.

    La regla queda fijada antes de evaluar los clasificadores: primero se busca el
    mayor coeficiente Silhouette y, si existen empates numéricos, se prefiere el
    menor índice Davies–Bouldin.

    Args:
    tabla_k (pd.DataFrame): Resultados generados por `evaluar_kmeans_objetivo`.

    Returns:
    int:
    Valor de k seleccionado objetivamente.

    Example:
    >>> k_optimo = seleccionar_k_objetivo(tabla_k)
    """
    if tabla_k.empty:
        raise ValueError("La tabla de validación de k está vacía.")
    ordenada = tabla_k.sort_values(["Silhouette", "Davies_Bouldin"], ascending=[False, True])
    return int(ordenada.iloc[0]["k"])

In [257]:
def graficar_validacion_k_revision(tabla_k: pd.DataFrame, dimension: int) -> None:
    """
    Grafica Silhouette, Davies-Bouldin, Calinski-Harabasz e Inercia
    para comparar valores de k con Bokeh.

    Criterio de interpretación:
    - Silhouette: mayor es mejor.
    - Davies-Bouldin: menor es mejor.
    - Calinski-Harabasz: mayor es mejor.
    - Inercia: menor es mejor, aunque se interpreta principalmente
      observando la reducción marginal al aumentar k.

    Args:
        tabla_k (pd.DataFrame):
            Tabla de métricas internas por valor de k.
            Debe contener las columnas:
            'k',
            'Silhouette',
            'Davies_Bouldin',
            'Calinski_Harabasz',
            'Inercia'.

        dimension (int):
            Dimensión del embedding evaluado.

    Returns:
        None
            Muestra un panel interactivo Bokeh dentro del notebook.

    Example:
        >>> graficar_validacion_k_revision(tabla_k, 100)
    """

    columnas_requeridas = {
        "k",
        "Silhouette",
        "Davies_Bouldin",
        "Calinski_Harabasz",
        "Inercia",
    }

    columnas_faltantes = columnas_requeridas.difference(tabla_k.columns)

    if columnas_faltantes:
        raise ValueError(
            "Faltan columnas requeridas en tabla_k: "
            f"{sorted(columnas_faltantes)}. "
            f"Columnas disponibles: {tabla_k.columns.tolist()}"
        )

    datos = tabla_k.sort_values("k").copy()

    source = ColumnDataSource(datos)

    # ---------------------------------------------------------
    # 1. Silhouette
    # ---------------------------------------------------------
    p1 = figure(
        height=340,
        width=780,
        title=f"Silhouette por k — {dimension}D",
        x_axis_label="k",
        y_axis_label="Silhouette",
    )

    p1.line(
        "k",
        "Silhouette",
        source=source,
        line_width=2,
    )

    p1.scatter(
        "k",
        "Silhouette",
        source=source,
        size=8,
    )

    p1.add_tools(
        HoverTool(
            tooltips=[
                ("k", "@k"),
                ("Silhouette", "@Silhouette{0.0000}"),
            ]
        )
    )

    # ---------------------------------------------------------
    # 2. Davies-Bouldin
    # ---------------------------------------------------------
    p2 = figure(
        height=340,
        width=780,
        title=f"Davies-Bouldin por k — {dimension}D",
        x_axis_label="k",
        y_axis_label="Davies-Bouldin",
    )

    p2.line(
        "k",
        "Davies_Bouldin",
        source=source,
        line_width=2,
    )

    p2.scatter(
        "k",
        "Davies_Bouldin",
        source=source,
        size=8,
    )

    p2.add_tools(
        HoverTool(
            tooltips=[
                ("k", "@k"),
                ("Davies-Bouldin", "@Davies_Bouldin{0.0000}"),
            ]
        )
    )

    # ---------------------------------------------------------
    # 3. Calinski-Harabasz
    # ---------------------------------------------------------
    p3 = figure(
        height=340,
        width=780,
        title=f"Calinski-Harabasz por k — {dimension}D",
        x_axis_label="k",
        y_axis_label="Calinski-Harabasz",
    )

    p3.line(
        "k",
        "Calinski_Harabasz",
        source=source,
        line_width=2,
    )

    p3.scatter(
        "k",
        "Calinski_Harabasz",
        source=source,
        size=8,
    )

    p3.add_tools(
        HoverTool(
            tooltips=[
                ("k", "@k"),
                ("Calinski-Harabasz", "@Calinski_Harabasz{0.00}"),
            ]
        )
    )

    # ---------------------------------------------------------
    # 4. Inercia
    # ---------------------------------------------------------
    p4 = figure(
        height=340,
        width=780,
        title=f"Inercia por k — {dimension}D",
        x_axis_label="k",
        y_axis_label="Inercia",
    )

    p4.line(
        "k",
        "Inercia",
        source=source,
        line_width=2,
    )

    p4.scatter(
        "k",
        "Inercia",
        source=source,
        size=8,
    )

    p4.add_tools(
        HoverTool(
            tooltips=[
                ("k", "@k"),
                ("Inercia", "@Inercia{0.00}"),
            ]
        )
    )

    # ---------------------------------------------------------
    # Panel de pestañas
    # ---------------------------------------------------------
    tabs = Tabs(
        tabs=[
            TabPanel(
                child=p1,
                title="Silhouette",
            ),
            TabPanel(
                child=p2,
                title="Davies-Bouldin",
            ),
            TabPanel(
                child=p3,
                title="Calinski-Harabasz",
            ),
            TabPanel(
                child=p4,
                title="Inercia",
            ),
        ]
    )

    show(tabs)

In [134]:
def evaluar_hdbscan_revision(vectores: np.ndarray, min_cluster_size: int, min_samples: int) -> tuple[np.ndarray, pd.DataFrame]:
    """
    Ejecuta HDBSCAN y calcula métricas de calidad sobre los puntos no marcados como ruido.

    HDBSCAN se incorpora como benchmark de clustering basado en densidad. Las métricas
    internas se calculan únicamente cuando existen al menos dos clústeres válidos
    después de excluir la etiqueta -1 correspondiente a ruido.

    Args:
    vectores (np.ndarray): Matriz de embeddings de palabras.
    min_cluster_size (int): Tamaño mínimo permitido para un clúster.
    min_samples (int): Número de vecinos usado para estimar densidad.

    Returns:
    tuple[np.ndarray, pd.DataFrame]:
    Etiquetas HDBSCAN y una tabla con número de clústeres, ruido, métricas y tiempo.

    Example:
    >>> labels_hdb, tabla_hdb = evaluar_hdbscan_revision(vectores, 20, 5)
    """
    if HDBSCAN is None:
        return np.full(len(vectores), -1, dtype=int), pd.DataFrame([{
            "Algoritmo": "HDBSCAN",
            "Clusters": 0,
            "Ruido_pct": 100.0,
            "Silhouette": np.nan,
            "Davies_Bouldin": np.nan,
            "Calinski_Harabasz": np.nan,
            "Tiempo_s": np.nan,
            "Estado": "HDBSCAN no disponible en esta versión de scikit-learn",
        }])

    inicio = time.perf_counter()
    modelo = HDBSCAN(
        min_cluster_size=int(min_cluster_size),
        min_samples=int(min_samples),
        metric="euclidean",
        cluster_selection_method="eom",
    )
    labels = modelo.fit_predict(vectores).astype(int)
    tiempo = time.perf_counter() - inicio

    mask = labels != -1
    labels_validos = labels[mask]
    vectores_validos = vectores[mask]
    clusters = sorted(set(labels_validos.tolist()))
    ruido_pct = float(np.mean(labels == -1) * 100.0)

    sil = db = ch = np.nan
    if len(clusters) >= 2 and len(vectores_validos) > len(clusters):
        sil = float(silhouette_score(vectores_validos, labels_validos))
        db = float(davies_bouldin_score(vectores_validos, labels_validos))
        ch = float(calinski_harabasz_score(vectores_validos, labels_validos))

    tabla = pd.DataFrame([{
        "Algoritmo": "HDBSCAN",
        "Clusters": int(len(clusters)),
        "Ruido_pct": ruido_pct,
        "Silhouette": sil,
        "Davies_Bouldin": db,
        "Calinski_Harabasz": ch,
        "Tiempo_s": float(tiempo),
        "Estado": "OK",
    }])
    return labels, tabla

In [135]:
def graficar_tsne_revision(palabras: Sequence[str], vectores: np.ndarray, etiquetas: np.ndarray, titulo: str, seed: int, max_puntos: int = 1000) -> None:
    """
    Proyecta embeddings a dos dimensiones con t-SNE y los muestra con Bokeh.

    t-SNE se utiliza únicamente como técnica de visualización no lineal; no se trata
    como algoritmo de clustering. La gráfica permite inspeccionar si las etiquetas
    obtenidas por el algoritmo de agrupamiento presentan separación visual.

    Args:
    palabras (Sequence[str]): Palabras asociadas a los vectores.
    vectores (np.ndarray): Embeddings a proyectar.
    etiquetas (np.ndarray): Etiqueta de clúster por palabra.
    titulo (str): Título de la visualización.
    seed (int): Semilla de t-SNE.
    max_puntos (int): Máximo de puntos que se visualizarán.
    *Valor por defecto: 1000.*

    Returns:
    None:
    Muestra la visualización interactiva dentro del notebook.

    Example:
    >>> graficar_tsne_revision(palabras, vectores, labels, "t-SNE K-Means", 5)
    """
    n = min(len(vectores), int(max_puntos))
    if n < 5:
        print("No hay suficientes puntos para t-SNE.")
        return

    rng = np.random.default_rng(seed)
    idx = np.arange(len(vectores))
    if len(idx) > n:
        idx = rng.choice(idx, size=n, replace=False)

    x = vectores[idx]
    labels = np.asarray(etiquetas)[idx]
    words = np.asarray(list(palabras), dtype=object)[idx]
    perplexity = max(5.0, min(30.0, (n - 1) / 3.0))

    coords = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=seed,
    ).fit_transform(x)

    vista = pd.DataFrame({
        "x": coords[:, 0],
        "y": coords[:, 1],
        "Palabra": words.astype(str),
        "Cluster": labels.astype(str),
    })
    source = ColumnDataSource(vista)
    p = figure(height=560, width=900, title=titulo, tools="pan,wheel_zoom,box_zoom,reset,save")
    p.scatter("x", "y", source=source, size=7, alpha=0.65, legend_field="Cluster")
    p.add_tools(HoverTool(tooltips=[("Palabra", "@Palabra"), ("Cluster", "@Cluster")]))
    p.legend.location = "top_left"
    show(p)

In [136]:
def expandir_cluster_politico_top_n(palabras: Sequence[str], vectores: np.ndarray, etiquetas: np.ndarray, cluster_politico: int, top_n: int) -> tuple[np.ndarray, pd.DataFrame, float]:
    """
    Expande el clúster político con una regla Top-N totalmente reproducible.

    Se calcula la similitud coseno entre cada palabra externa y el centroide político.
    Los candidatos se ordenan de mayor a menor similitud y se añaden exactamente los
    `top_n` términos más próximos, o todos los disponibles si existen menos. El umbral
    reportado es la similitud del último término incorporado.

    Args:
    palabras (Sequence[str]): Vocabulario agrupado.
    vectores (np.ndarray): Vectores Word2Vec correspondientes a las palabras.
    etiquetas (np.ndarray): Etiquetas del clustering inicial.
    cluster_politico (int): Identificador del clúster político.
    top_n (int): Número máximo de términos externos que se incorporarán.

    Returns:
    tuple[np.ndarray, pd.DataFrame, float]:
    Nuevas etiquetas, detalle ordenado de términos añadidos y umbral de similitud.

    Example:
    >>> nuevas, detalle, umbral = expandir_cluster_politico_top_n(palabras, vectores, labels, 1, 100)
    """
    if top_n <= 0:
        raise ValueError("top_n debe ser mayor que cero.")

    etiquetas = np.asarray(etiquetas).astype(int)
    mask_pol = etiquetas == int(cluster_politico)
    if not np.any(mask_pol):
        raise ValueError("El clúster político no contiene palabras.")

    centroide = vectores[mask_pol].mean(axis=0)
    norma_c = max(float(np.linalg.norm(centroide)), 1e-12)
    indices_externos = np.where(~mask_pol)[0]
    if len(indices_externos) == 0:
        return etiquetas.copy(), pd.DataFrame(), np.nan

    ext = vectores[indices_externos]
    denom = np.maximum(np.linalg.norm(ext, axis=1) * norma_c, 1e-12)
    similitudes = (ext @ centroide) / denom
    orden = np.argsort(-similitudes)
    n_real = min(int(top_n), len(orden))
    seleccion_local = orden[:n_real]
    seleccion_global = indices_externos[seleccion_local]

    nuevas = etiquetas.copy()
    nuevas[seleccion_global] = int(cluster_politico)
    detalle = pd.DataFrame({
        "Rango": np.arange(1, n_real + 1, dtype=int),
        "Palabra": [palabras[i] for i in seleccion_global],
        "Cluster_original": etiquetas[seleccion_global],
        "Cluster_nuevo": int(cluster_politico),
        "Similitud_coseno_politica": similitudes[seleccion_local],
        "Distancia_coseno_politica": 1.0 - similitudes[seleccion_local],
    }).sort_values("Rango").reset_index(drop=True)

    umbral = float(detalle["Similitud_coseno_politica"].iloc[-1]) if not detalle.empty else np.nan
    return nuevas, detalle, umbral

In [137]:
def obtener_palabras_similares(modelo: Word2Vec, palabra: str, top_n: int = 15) -> pd.DataFrame:
    """
    Obtiene las palabras más similares a un término según Word2Vec.

    Consulta la similitud coseno aprendida por el modelo para reproducir las pruebas
    semánticas mostradas en la tesis. Si la palabra no está en el vocabulario, devuelve
    una tabla vacía en lugar de interrumpir la ejecución.

    Args:
    modelo (Word2Vec): Modelo Word2Vec entrenado.
    palabra (str): Término de consulta.
    top_n (int): Número de palabras similares solicitadas.
    *Valor por defecto: 15.*

    Returns:
    pd.DataFrame:
    Tabla con palabra similar y valor de similitud coseno.

    Example:
    >>> obtener_palabras_similares(modelo, "politica", 10)
    """
    palabra = palabra.strip().lower()
    if palabra not in modelo.wv:
        return pd.DataFrame(columns=["Palabra", "Similitud"])
    similares = modelo.wv.most_similar(palabra, topn=top_n)
    return pd.DataFrame(similares, columns=["Palabra", "Similitud"])

In [138]:
def calcular_similitud_con_conjunto(modelo: Word2Vec, palabra: str, conjunto: Sequence[str]) -> pd.DataFrame:
    """
    Calcula la similitud de una palabra respecto a un conjunto de términos.

    Evalúa cada término válido del conjunto mediante similitud coseno de Word2Vec y
    devuelve los resultados ordenados de mayor a menor, reproduciendo otra de las
    comprobaciones semánticas incluidas en la tesis.

    Args:
    modelo (Word2Vec): Modelo Word2Vec entrenado.
    palabra (str): Palabra de referencia.
    conjunto (Sequence[str]): Términos contra los cuales se calcula la similitud.

    Returns:
    pd.DataFrame:
    Tabla ordenada con término y similitud.

    Example:
    >>> calcular_similitud_con_conjunto(modelo, "politica", ["gobierno", "futbol"])
    """
    palabra = palabra.strip().lower()
    if palabra not in modelo.wv:
        return pd.DataFrame(columns=["Termino", "Similitud"])
    filas = []
    for termino in conjunto:
        termino = str(termino).strip().lower()
        if termino in modelo.wv:
            filas.append({"Termino": termino, "Similitud": float(modelo.wv.similarity(palabra, termino))})
    return pd.DataFrame(filas).sort_values("Similitud", ascending=False).reset_index(drop=True) if filas else pd.DataFrame(columns=["Termino", "Similitud"])

In [139]:
def resolver_analogia(modelo: Word2Vec, positivo: Sequence[str], negativo: Sequence[str], top_n: int = 10) -> pd.DataFrame:
    """
    Resuelve una analogía vectorial con el modelo Word2Vec.

    Calcula términos cercanos a la combinación vectorial de palabras positivas menos
    palabras negativas. Solo utiliza términos presentes en el vocabulario para evitar
    fallos por palabras desconocidas.

    Args:
    modelo (Word2Vec): Modelo Word2Vec entrenado.
    positivo (Sequence[str]): Palabras cuyos vectores se suman.
    negativo (Sequence[str]): Palabras cuyos vectores se restan.
    top_n (int): Número máximo de resultados devueltos.
    *Valor por defecto: 10.*

    Returns:
    pd.DataFrame:
    Tabla con términos candidatos y sus similitudes.

    Example:
    >>> resolver_analogia(modelo, ["presidente", "ecuador"], ["quito"], 5)
    """
    pos = [str(x).lower().strip() for x in positivo if str(x).lower().strip() in modelo.wv]
    neg = [str(x).lower().strip() for x in negativo if str(x).lower().strip() in modelo.wv]
    if not pos:
        return pd.DataFrame(columns=["Palabra", "Similitud"])
    try:
        resultado = modelo.wv.most_similar(positive=pos, negative=neg, topn=top_n)
    except Exception:
        return pd.DataFrame(columns=["Palabra", "Similitud"])
    return pd.DataFrame(resultado, columns=["Palabra", "Similitud"])

In [140]:
def graficar_similitudes(similares: pd.DataFrame, palabra: str, dimension: int) -> None:
    """
    Grafica los valores de similitud Word2Vec de una consulta.

    Presenta las palabras más cercanas en barras horizontales con información
    interactiva. Si la tabla está vacía, muestra un aviso dentro del notebook.

    Args:
    similares (pd.DataFrame): Tabla con columnas `Palabra` y `Similitud`.
    palabra (str): Palabra utilizada como referencia.
    dimension (int): Dimensión del Word2Vec evaluado.

    Returns:
    None:
    Muestra la gráfica Bokeh o un mensaje de ausencia de resultados.

    Example:
    >>> graficar_similitudes(similares, "politica", 100)
    """
    if similares.empty:
        show(Div(text=f"<b>No hubo resultados de similitud para '{palabra}'.</b>"))
        return
    df = similares.iloc[::-1].copy()
    source = ColumnDataSource(df)
    p = figure(y_range=df["Palabra"].astype(str).tolist(), width=850, height=max(350, 25 * len(df)), title=f"Palabras similares a '{palabra}' — {dimension}D")
    p.hbar(y="Palabra", right="Similitud", height=0.7, source=source)
    p.xaxis.axis_label = "Similitud coseno"
    p.add_tools(HoverTool(tooltips=[("Palabra", "@Palabra"), ("Similitud", "@Similitud{0.0000}")]))
    show(p)

In [141]:
def graficar_clusters_pca(palabras: Sequence[str], vectores: np.ndarray, etiquetas: np.ndarray, cluster_politico: int, dimension: int, max_puntos: int = 900) -> None:
    """
    Proyecta los clústeres de palabras a dos dimensiones y los muestra con Bokeh.

    Aplica PCA únicamente con fines de visualización. Si existen demasiadas palabras,
    toma una muestra reproducible para mantener la interacción fluida. El clúster
    identificado como político queda indicado en la información emergente.

    Args:
    palabras (Sequence[str]): Palabras agrupadas.
    vectores (np.ndarray): Embeddings de las palabras.
    etiquetas (np.ndarray): Etiquetas de clúster.
    cluster_politico (int): Identificador del clúster político.
    dimension (int): Dimensión original del embedding.
    max_puntos (int): Máximo de puntos a visualizar.
    *Valor por defecto: 900.*

    Returns:
    None:
    Muestra el diagrama interactivo de clústeres.

    Example:
    >>> graficar_clusters_pca(palabras, vectores, labels, 2, 100)
    """
    n = len(palabras)
    rng = np.random.default_rng(RANDOM_STATE)
    idx = np.arange(n)
    if n > max_puntos:
        idx = np.sort(rng.choice(idx, size=max_puntos, replace=False))

    coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(vectores[idx])
    labels_sub = etiquetas[idx]
    palabras_sub = [palabras[i] for i in idx]
    colores_base = Category10[10]
    colores = [colores_base[int(c) % len(colores_base)] for c in labels_sub]
    source = ColumnDataSource({
        "x": coords[:, 0],
        "y": coords[:, 1],
        "palabra": palabras_sub,
        "cluster": labels_sub.astype(str),
        "tipo": ["Político" if int(c) == cluster_politico else "Otro" for c in labels_sub],
        "color": colores,
    })
    p = figure(width=950, height=550, title=f"Proyección PCA de clústeres — Word2Vec {dimension}D", tools="pan,wheel_zoom,box_zoom,reset,save")
    p.scatter("x", "y", source=source, size=7, alpha=0.7, color="color")
    p.add_tools(HoverTool(tooltips=[("Palabra", "@palabra"), ("Clúster", "@cluster"), ("Tipo", "@tipo")]))
    p.xaxis.axis_label = "PCA 1"
    p.yaxis.axis_label = "PCA 2"
    show(p)

# 6. Clasificación y holdout sin fuga de información

In [142]:
def crear_modelos_clasificacion(knn_vecinos: int, seed: int) -> dict[str, Any]:
    """
    Construye los cuatro clasificadores supervisados evaluados en la tesis.

    Se emplean Árbol de Decisión, KNN, Gaussian Naïve Bayes y una SVM lineal escalable.
    LinearSVC sigue siendo una máquina de vectores de soporte y evita el costo cuadrático
    de un SVC con kernel sobre más de cien mil tweets.

    Args:
    knn_vecinos (int): Número de vecinos utilizados por KNN.
    seed (int): Semilla del árbol de decisión y de la SVM.

    Returns:
    dict[str, Any]:
    Diccionario con el nombre del algoritmo y su estimador de scikit-learn.

    Example:
    >>> modelos = crear_modelos_clasificacion(5, 5)
    """
    return {
        "Árbol de Decisión": DecisionTreeClassifier(random_state=seed),
        "KNN": KNeighborsClassifier(n_neighbors=knn_vecinos, n_jobs=-1),
        "Naïve Bayes": GaussianNB(),
        "SVM": LinearSVC(random_state=seed, max_iter=5000, dual="auto"),
    }

In [143]:
def calcular_metricas_tesis(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """
    Calcula todas las métricas de evaluación reportadas en la tesis.

    Trata la etiqueta 1 como clase política y obtiene Accuracy, Precision, Recall,
    F1-score, Sensibilidad, Especificidad, Precisión positiva, tasa de falsos positivos,
    F2 y F0.5. Sensibilidad equivale a Recall y Precisión positiva equivale a Precision,
    pero se conservan ambos nombres para reflejar las tablas originales.

    Args:
    y_true (np.ndarray): Etiquetas verdaderas del conjunto de prueba.
    y_pred (np.ndarray): Predicciones generadas por el clasificador.

    Returns:
    dict[str, float]:
    Diccionario con todas las métricas calculadas para la clase positiva.

    Example:
    >>> calcular_metricas_tesis(np.array([0, 1]), np.array([0, 1]))["F1-score"]
    1.0
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    especificidad = tn / (tn + fp) if (tn + fp) else 0.0
    fpr = fp / (tn + fp) if (tn + fp) else 0.0
    return {
        "Exactitud": float(accuracy_score(y_true, y_pred)),
        "Precisión": float(precision),
        "Recall": float(recall),
        "F1-score": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "Sensibilidad": float(recall),
        "Especificidad": float(especificidad),
        "Precisión positiva": float(precision),
        "Tasa de falsos positivos": float(fpr),
        "Valor F2": float(fbeta_score(y_true, y_pred, beta=2.0, pos_label=1, zero_division=0)),
        "Valor F0.5": float(fbeta_score(y_true, y_pred, beta=0.5, pos_label=1, zero_division=0)),
        "TP": float(tp),
        "TN": float(tn),
        "FP": float(fp),
        "FN": float(fn),
    }

In [144]:
def entrenar_evaluar_modelos_split(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray, dimension: int, escenario: str, seed: int, knn_vecinos: int) -> pd.DataFrame:
    """
    Entrena clasificadores en un split ya definido y evalúa exclusivamente el test externo.

    A diferencia de la función original, esta función no realiza una nueva partición.
    Recibe matrices de entrenamiento y prueba construidas después de separar los datos,
    evitando que el test participe en Word2Vec, clustering o expansión del léxico.

    Args:
    X_train (np.ndarray): Representaciones de tweets de entrenamiento.
    y_train (np.ndarray): Pseudoetiquetas del entrenamiento.
    X_test (np.ndarray): Representaciones de tweets de prueba.
    y_test (np.ndarray): Etiquetas de referencia usadas para evaluar el test.
    dimension (int): Dimensión del embedding.
    escenario (str): Nombre del escenario.
    seed (int): Semilla de los clasificadores.
    knn_vecinos (int): Número de vecinos de KNN.

    Returns:
    pd.DataFrame:
    Métricas de cada clasificador sobre el mismo conjunto de prueba.

    Example:
    >>> tabla = entrenar_evaluar_modelos_split(X_train, y_train, X_test, y_test, 100, "Escenario 1", 5, 5)
    """
    clases, conteos = np.unique(y_train, return_counts=True)
    if len(clases) < 2 or np.min(conteos) < 2:
        raise ValueError(f"El entrenamiento requiere dos clases. Distribución: {dict(zip(clases, conteos))}")

    filas: list[dict[str, Any]] = []
    for nombre, modelo in crear_modelos_clasificacion(knn_vecinos, seed).items():
        inicio = time.perf_counter()
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_test)
        tiempo = time.perf_counter() - inicio
        fila: dict[str, Any] = {
            "Dimension": int(dimension),
            "Escenario": escenario,
            "Algoritmo": nombre,
            "Tiempo_clasificador_s": float(tiempo),
        }
        fila.update(calcular_metricas_tesis(y_test, pred))
        filas.append(fila)
    return pd.DataFrame(filas)

In [160]:
def construir_pipeline_semantico_train(tokens_train: Sequence[Sequence[str]], tokens_trends: Sequence[Sequence[str]], dimension: int, semillas: Sequence[str], valores_k: Sequence[int], top_n: int, epochs: int, seed: int) -> dict[str, Any]:
    """
    Construye Word2Vec, clustering y expansión usando exclusivamente el conjunto de entrenamiento.

    Esta función corrige la fuga de información señalada por los revisores. Word2Vec
    se entrena solo con tweets del `train`; las palabras de trends actúan únicamente
    como vocabulario candidato y solo se conservan si fueron aprendidas desde `train`.
    El valor de k se selecciona con Silhouette y Davies–Bouldin antes de generar
    pseudoetiquetas del conjunto de prueba.

    Args:
    tokens_train (Sequence[Sequence[str]]): Tweets tokenizados del conjunto de entrenamiento.
    tokens_trends (Sequence[Sequence[str]]): Trends externos usados como vocabulario candidato.
    dimension (int): Dimensión de Word2Vec.
    semillas (Sequence[str]): Términos políticos de referencia.
    valores_k (Sequence[int]): Valores de k evaluados objetivamente.
    top_n (int): Número de palabras añadidas al clúster político.
    epochs (int): Épocas de Word2Vec.
    seed (int): Semilla reproducible.

    Returns:
    dict[str, Any]:
    Modelo, vocabulario, selección de k, centroides y expansión de ambos escenarios.

    Example:
    >>> pipe = construir_pipeline_semantico_train(tokens_train, tokens_trends, 100, semillas_politicas, range(2, 8), 100, 10, 5)
    """
    inicio_w2v = time.perf_counter()
    modelo = entrenar_word2vec(tokens_train, dimension, W2V_WINDOW, W2V_MIN_COUNT, epochs, seed)
    tiempo_w2v = time.perf_counter() - inicio_w2v

    palabras = obtener_palabras_trend_unicas(tokens_trends, modelo)
    if len(palabras) < 3:
        raise ValueError("No hay suficientes palabras de trends presentes en el vocabulario de entrenamiento.")

    vectores_raw = obtener_matriz_palabras(palabras, modelo)
    vectores_cluster = normalize(vectores_raw, norm="l2")

    tabla_k = evaluar_kmeans_objetivo(vectores_cluster, valores_k, seed)
    k_seleccionado = seleccionar_k_objetivo(tabla_k)
    kmeans, labels_1 = entrenar_kmeans_palabras(vectores_cluster, k_seleccionado, seed)

    cluster_politico, scores = seleccionar_cluster_politico(
        palabras,
        labels_1,
        vectores_raw,
        modelo,
        semillas,
    )
    centroides_1 = calcular_centroides(vectores_raw, labels_1, k_seleccionado)

    labels_2, palabras_anadidas, umbral_similitud = expandir_cluster_politico_top_n(
        palabras,
        vectores_raw,
        labels_1,
        cluster_politico,
        top_n,
    )
    centroides_2 = calcular_centroides(vectores_raw, labels_2, k_seleccionado)

    return {
        "modelo_word2vec": modelo,
        "tiempo_word2vec_s": float(tiempo_w2v),
        "palabras": palabras,
        "vectores_palabras_raw": vectores_raw,
        "vectores_palabras_cluster": vectores_cluster,
        "tabla_k": tabla_k,
        "k_seleccionado": int(k_seleccionado),
        "kmeans": kmeans,
        "labels_palabras_1": labels_1,
        "labels_palabras_2": labels_2,
        "cluster_politico": int(cluster_politico),
        "scores_cluster": scores,
        "centroides_1": centroides_1,
        "centroides_2": centroides_2,
        "palabras_anadidas": palabras_anadidas,
        "umbral_similitud": float(umbral_similitud),
        "top_n_solicitado": int(top_n),
    }

In [146]:
def ejecutar_holdout_sin_fuga_revision(tweets_desarrollo: pd.DataFrame, tokens_trends: Sequence[Sequence[str]], dimension: int, semillas: Sequence[str], top_n: int, test_size: float, epochs: int, seed: int) -> dict[str, Any]:
    """
    Ejecuta un holdout 80/20 donde la partición ocurre antes de todo aprendizaje semántico.

    Primero divide tweets por evento; después entrena Word2Vec y K-Means solo con el
    conjunto de entrenamiento. Los centroides y el léxico expandido se calculan también
    exclusivamente desde train. Finalmente se vectoriza y pseudoetiqueta el test sin
    reentrenar ni modificar el espacio semántico.

    Args:
    tweets_desarrollo (pd.DataFrame): Corpus disponible después de separar el holdout manual.
    tokens_trends (Sequence[Sequence[str]]): Trends tokenizados.
    dimension (int): Dimensión Word2Vec.
    semillas (Sequence[str]): Semillas políticas.
    top_n (int): Regla explícita de expansión.
    test_size (float): Proporción del holdout automático.
    epochs (int): Épocas Word2Vec.
    seed (int): Semilla reproducible.

    Returns:
    dict[str, Any]:
    Split, pipeline semántico, pseudoetiquetas, métricas y tablas de balance.

    Example:
    >>> resultado = ejecutar_holdout_sin_fuga_revision(tweets_desarrollo, tokens_trends, 100, semillas_politicas, 100, 0.2, 10, 5)
    """
    estrato = tweets_desarrollo["evento"].fillna("Sin evento").astype(str)
    train_df, test_df = train_test_split(
        tweets_desarrollo,
        test_size=test_size,
        random_state=seed,
        stratify=estrato,
    )
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    tokens_train = preparar_tokens_dataframe(train_df, "tokens_sin_stopwords")
    tokens_test = preparar_tokens_dataframe(test_df, "tokens_sin_stopwords")

    pipeline = construir_pipeline_semantico_train(
        tokens_train,
        tokens_trends,
        dimension,
        semillas,
        K_CANDIDATOS,
        top_n,
        epochs,
        seed,
    )

    modelo = pipeline["modelo_word2vec"]
    X_train = vectorizar_documentos(tokens_train, modelo, agregacion="suma")
    X_test = vectorizar_documentos(tokens_test, modelo, agregacion="suma")

    y1_train, _, _ = etiquetar_por_centroides(X_train, pipeline["centroides_1"], pipeline["cluster_politico"])
    y1_test, _, _ = etiquetar_por_centroides(X_test, pipeline["centroides_1"], pipeline["cluster_politico"])
    y2_train, _, _ = etiquetar_por_centroides(X_train, pipeline["centroides_2"], pipeline["cluster_politico"])
    y2_test, _, _ = etiquetar_por_centroides(X_test, pipeline["centroides_2"], pipeline["cluster_politico"])

    met1 = entrenar_evaluar_modelos_split(X_train, y1_train, X_test, y1_test, dimension, "Escenario 1", seed, KNN_VECINOS)
    met2 = entrenar_evaluar_modelos_split(X_train, y2_train, X_test, y2_test, dimension, "Escenario 2", seed, KNN_VECINOS)
    metricas = pd.concat([met1, met2], ignore_index=True)

    balance = pd.DataFrame([
        {"Conjunto": "Train", "Escenario": "Escenario 1", "N": len(y1_train), "Politicos": int(y1_train.sum()), "No_politicos": int((y1_train == 0).sum()), "Politicos_pct": float(y1_train.mean() * 100)},
        {"Conjunto": "Test", "Escenario": "Escenario 1", "N": len(y1_test), "Politicos": int(y1_test.sum()), "No_politicos": int((y1_test == 0).sum()), "Politicos_pct": float(y1_test.mean() * 100)},
        {"Conjunto": "Train", "Escenario": "Escenario 2", "N": len(y2_train), "Politicos": int(y2_train.sum()), "No_politicos": int((y2_train == 0).sum()), "Politicos_pct": float(y2_train.mean() * 100)},
        {"Conjunto": "Test", "Escenario": "Escenario 2", "N": len(y2_test), "Politicos": int(y2_test.sum()), "No_politicos": int((y2_test == 0).sum()), "Politicos_pct": float(y2_test.mean() * 100)},
    ])

    return {
        "dimension": int(dimension),
        "train_df": train_df,
        "test_df": test_df,
        "tokens_train": tokens_train,
        "tokens_test": tokens_test,
        "pipeline": pipeline,
        "X_train": X_train,
        "X_test": X_test,
        "y1_train": y1_train,
        "y1_test": y1_test,
        "y2_train": y2_train,
        "y2_test": y2_test,
        "metricas": metricas,
        "balance": balance,
    }

In [147]:
def resumir_holdouts_dimensiones_revision(resultados: Mapping[int, Mapping[str, Any]]) -> pd.DataFrame:
    """
    Resume selección de k, expansión y balance de clases para cada dimensión Word2Vec.

    Esta tabla responde de forma directa a la solicitud de reportar el número de términos
    añadidos en cada dimensión, el umbral de similitud resultante y la distribución de
    la clase política antes y después de la expansión.

    Args:
    resultados (Mapping[int, Mapping[str, Any]]): Resultados holdout sin fuga por dimensión.

    Returns:
    pd.DataFrame:
    Una fila por dimensión con k, clúster político, términos añadidos, umbral y balance.

    Example:
    >>> resumen = resumir_holdouts_dimensiones_revision({100: resultado_holdout})
    """
    filas: list[dict[str, Any]] = []
    for dimension, resultado in resultados.items():
        pipe = resultado["pipeline"]
        y1_test = np.asarray(resultado["y1_test"]).astype(int)
        y2_test = np.asarray(resultado["y2_test"]).astype(int)
        filas.append({
            "Dimension": int(dimension),
            "k_seleccionado": int(pipe["k_seleccionado"]),
            "Cluster_politico": int(pipe["cluster_politico"]),
            "Vocabulario_trends_en_train": int(len(pipe["palabras"])),
            "Top_N_solicitado": int(pipe.get("top_n_solicitado", len(pipe["palabras_anadidas"]))),
            "N_terminos_anadidos": int(len(pipe["palabras_anadidas"])),
            "Umbral_similitud": float(pipe["umbral_similitud"]),
            "Politicos_test_E1": int(y1_test.sum()),
            "Politicos_test_E1_pct": float(y1_test.mean() * 100.0),
            "Politicos_test_E2": int(y2_test.sum()),
            "Politicos_test_E2_pct": float(y2_test.mean() * 100.0),
            "Ganancia_politicos_test": int(y2_test.sum() - y1_test.sum()),
        })
    return pd.DataFrame(filas).sort_values("Dimension").reset_index(drop=True)

In [148]:
def comparar_kmeans_hdbscan_revision(resultado_holdout: Mapping[str, Any], min_cluster_size: int, min_samples: int) -> tuple[pd.DataFrame, np.ndarray]:
    """
    Compara el clustering K-Means seleccionado objetivamente con HDBSCAN.

    La comparación utiliza exactamente los mismos embeddings de palabras construidos
    solo con el conjunto de entrenamiento. Se reportan número de clústeres, porcentaje
    de ruido, métricas internas y tiempo.

    Args:
    resultado_holdout (Mapping[str, Any]): Salida de `ejecutar_holdout_sin_fuga_revision`.
    min_cluster_size (int): Tamaño mínimo de clúster HDBSCAN.
    min_samples (int): Parámetro de densidad HDBSCAN.

    Returns:
    tuple[pd.DataFrame, np.ndarray]:
    Tabla comparativa y etiquetas de HDBSCAN.

    Example:
    >>> comparacion, labels_hdb = comparar_kmeans_hdbscan_revision(resultado_holdout, 20, 5)
    """
    pipe = resultado_holdout["pipeline"]
    vectores = pipe["vectores_palabras_cluster"]
    labels_k = pipe["labels_palabras_1"]
    k = pipe["k_seleccionado"]
    fila_k = pd.DataFrame([{
        "Algoritmo": f"K-Means (k={k})",
        "Clusters": int(k),
        "Ruido_pct": 0.0,
        "Silhouette": float(silhouette_score(vectores, labels_k)),
        "Davies_Bouldin": float(davies_bouldin_score(vectores, labels_k)),
        "Calinski_Harabasz": float(calinski_harabasz_score(vectores, labels_k)),
        "Tiempo_s": float(pipe["tabla_k"].loc[pipe["tabla_k"]["k"] == k, "Tiempo_s"].iloc[0]),
        "Estado": "OK",
    }])
    labels_hdb, tabla_hdb = evaluar_hdbscan_revision(vectores, min_cluster_size, min_samples)
    return pd.concat([fila_k, tabla_hdb], ignore_index=True), labels_hdb

In [149]:
def evaluar_sensibilidad_top_n_revision(resultado_holdout: Mapping[str, Any], valores_top_n: Sequence[int], seed: int) -> pd.DataFrame:
    """
    Evalúa cómo cambia el rendimiento al variar explícitamente el número de términos añadidos.

    Reutiliza el mismo split y el mismo Word2Vec/clustering de entrenamiento. Para cada
    Top-N recalcula únicamente el léxico expandido, su centroide, el balance de la clase
    política y las métricas de los cuatro clasificadores.

    Args:
    resultado_holdout (Mapping[str, Any]): Resultado del holdout sin fuga.
    valores_top_n (Sequence[int]): Valores Top-N que se desean comparar.
    seed (int): Semilla de los clasificadores.

    Returns:
    pd.DataFrame:
    Métricas por Top-N, algoritmo, número real añadido y umbral de similitud.

    Example:
    >>> sensibilidad = evaluar_sensibilidad_top_n_revision(resultado_holdout, [25, 50, 100], 5)
    """
    pipe = resultado_holdout["pipeline"]
    palabras = pipe["palabras"]
    vectores = pipe["vectores_palabras_raw"]
    labels_1 = pipe["labels_palabras_1"]
    cluster_politico = pipe["cluster_politico"]
    X_train = resultado_holdout["X_train"]
    X_test = resultado_holdout["X_test"]
    dimension = resultado_holdout["dimension"]
    filas: list[pd.DataFrame] = []

    for top_n in valores_top_n:
        labels_2, detalle, umbral = expandir_cluster_politico_top_n(
            palabras,
            vectores,
            labels_1,
            cluster_politico,
            int(top_n),
        )
        centroides_2 = calcular_centroides(vectores, labels_2, pipe["k_seleccionado"])
        y_train, _, _ = etiquetar_por_centroides(X_train, centroides_2, cluster_politico)
        y_test, _, _ = etiquetar_por_centroides(X_test, centroides_2, cluster_politico)
        tabla = entrenar_evaluar_modelos_split(
            X_train,
            y_train,
            X_test,
            y_test,
            dimension,
            f"Top-{top_n}",
            seed,
            KNN_VECINOS,
        )
        tabla["Top_N_solicitado"] = int(top_n)
        tabla["N_terminos_anadidos"] = int(len(detalle))
        tabla["Umbral_similitud"] = float(umbral)
        tabla["Politicos_test"] = int(y_test.sum())
        tabla["Politicos_test_pct"] = float(y_test.mean() * 100.0)
        filas.append(tabla)

    return pd.concat(filas, ignore_index=True) if filas else pd.DataFrame()

In [259]:
def graficar_sensibilidad_top_n_revision(
    sensibilidad: pd.DataFrame,
    metrica: str = "F1-score"
) -> None:
    """
    Grafica la sensibilidad de una métrica respecto al Top-N de expansión.

    Cada línea corresponde a un clasificador y utiliza un color diferente.
    Esto permite verificar si las conclusiones dependen fuertemente de un
    único valor de expansión y facilita la comparación visual entre modelos.

    Args:
        sensibilidad (pd.DataFrame):
            Tabla producida por `evaluar_sensibilidad_top_n_revision`.

        metrica (str):
            Métrica que se desea visualizar.
            Valor por defecto: "F1-score".

    Returns:
        None:
            Muestra la gráfica Bokeh en el notebook.

    Example:
        >>> graficar_sensibilidad_top_n_revision(
        ...     sensibilidad,
        ...     "F1-score"
        ... )
    """
    if sensibilidad.empty:
        print("No hay resultados de sensibilidad para graficar.")
        return

    if metrica not in sensibilidad.columns:
        raise ValueError(
            f"La métrica '{metrica}' no existe en el DataFrame. "
            f"Columnas disponibles: {sensibilidad.columns.tolist()}"
        )

    algoritmos = list(sensibilidad["Algoritmo"].unique())

    if len(algoritmos) <= 10:
        paleta = Category10[10]
    else:
        paleta = Category20[20]

    colores_algoritmos = {
        algoritmo: paleta[i % len(paleta)]
        for i, algoritmo in enumerate(algoritmos)
    }

    p = figure(
        height=430,
        width=900,
        title=f"Sensibilidad de {metrica} frente a Top-N",
        x_axis_label="Top-N",
        y_axis_label=metrica
    )

    for algoritmo in algoritmos:

        sub = (
            sensibilidad[
                sensibilidad["Algoritmo"] == algoritmo
            ]
            .sort_values("Top_N_solicitado")
            .copy()
        )

        source = ColumnDataSource(sub)

        color = colores_algoritmos[algoritmo]

        p.line(
            "Top_N_solicitado",
            metrica,
            source=source,
            line_width=2.5,
            color=color,
            legend_label=str(algoritmo)
        )

        p.scatter(
            "Top_N_solicitado",
            metrica,
            source=source,
            size=8,
            color=color,
            legend_label=str(algoritmo)
        )

    p.add_tools(
        HoverTool(
            tooltips=[
                ("Algoritmo", "@Algoritmo"),
                ("Top-N", "@Top_N_solicitado"),
                (metrica, f"@{{{metrica}}}{{0.0000}}"),
                ("Términos añadidos", "@N_terminos_anadidos"),
                ("Umbral", "@Umbral_similitud{0.0000}")
            ]
        )
    )

    p.legend.location = "bottom_right"
    p.legend.click_policy = "hide"
    p.legend.title = "Clasificador"

    show(p)

In [264]:
def graficar_cobertura_umbral_top_n_revision(
    sensibilidad: pd.DataFrame,
    dimension: int = 100
) -> None:
    """
    Grafica el efecto de Top-N sobre:
    1. la cobertura de contenido político
    2. el umbral de similitud observado

    La cobertura corresponde al porcentaje de tweets del test
    pseudoetiquetados como políticos bajo cada configuración Top-N.

    Args:
        sensibilidad (pd.DataFrame):
            Tabla producida por
            `evaluar_sensibilidad_top_n_revision`.

            Debe contener:
            - Top_N_solicitado
            - Politicos_test_pct
            - Umbral_similitud

        dimension (int):
            Dimensión Word2Vec evaluada.

    Returns:
        None:
            Muestra dos gráficas Bokeh lado a lado.
    """

    if sensibilidad.empty:
        print("No hay resultados de sensibilidad para graficar.")
        return

    columnas_requeridas = {
        "Top_N_solicitado",
        "Politicos_test_pct",
        "Umbral_similitud",
    }

    faltantes = columnas_requeridas.difference(
        sensibilidad.columns
    )

    if faltantes:
        raise ValueError(
            f"Faltan columnas: {sorted(faltantes)}. "
            f"Columnas disponibles: "
            f"{sensibilidad.columns.tolist()}"
        )

    datos = (
        sensibilidad[
            [
                "Top_N_solicitado",
                "Politicos_test_pct",
                "Umbral_similitud",
            ]
        ]
        .drop_duplicates(subset="Top_N_solicitado")
        .sort_values("Top_N_solicitado")
        .copy()
    )

    source = ColumnDataSource(datos)

    p1 = figure(
        height=380,
        width=500,
        title=f"Political coverage vs Top-N — {dimension}D",
        x_axis_label="Top-N",
        y_axis_label="Political coverage (%)",
    )

    p1.line(
        "Top_N_solicitado",
        "Politicos_test_pct",
        source=source,
        line_width=2.5,
    )

    p1.scatter(
        "Top_N_solicitado",
        "Politicos_test_pct",
        source=source,
        size=9,
    )

    p1.add_tools(
        HoverTool(
            tooltips=[
                ("Top-N", "@Top_N_solicitado"),
                (
                    "Political coverage",
                    "@Politicos_test_pct{0.00}%"
                ),
            ]
        )
    )

    p2 = figure(
        height=380,
        width=500,
        title=f"Observed similarity cutoff vs Top-N — {dimension}D",
        x_axis_label="Top-N",
        y_axis_label="Cosine similarity cutoff",
    )

    p2.line(
        "Top_N_solicitado",
        "Umbral_similitud",
        source=source,
        line_width=2.5,
    )

    p2.scatter(
        "Top_N_solicitado",
        "Umbral_similitud",
        source=source,
        size=9,
    )

    p2.add_tools(
        HoverTool(
            tooltips=[
                ("Top-N", "@Top_N_solicitado"),
                (
                    "Similarity cutoff",
                    "@Umbral_similitud{0.0000}"
                ),
            ]
        )
    )

    show(row(p1, p2))

In [152]:
tokens_trends = preparar_tokens_dataframe(trends, "tokens_sin_stopwords")

In [153]:
semillas_tesis: list[str] = []
if "Terminos_Tesis" in resumen_preprocesamiento:
    semillas_tesis = (
        resumen_preprocesamiento["Terminos_Tesis"]
        .iloc[:, 0]
        .dropna()
        .astype(str)
        .str.lower()
        .str.strip()
        .tolist()
    )

In [154]:
semillas_politicas = list(dict.fromkeys(semillas_tesis + SEMILLAS_POLITICAS_EXTRA))
semillas_politicas_sin_hashtags = [
    s for s in semillas_politicas
    if s and not str(s).strip().startswith("#")
]

In [155]:
print(f"Semillas políticas totales: {len(semillas_politicas)}")

Semillas políticas totales: 20


In [156]:
print(semillas_politicas)

['ecu', 'lasso', 'guillermo', '@asambleaecuador', 'guerra', 'protesta', 'politica', 'politico', 'presidente', 'gobierno', 'asamblea', 'elecciones', 'paro', 'nacional', 'muerte', 'cruzada', 'correa', 'iza', '#paronacional', 'ecuador']


In [157]:
resultados_holdout: dict[int, dict[str, Any]] = {}

In [161]:
for dimension in DIMENSIONES:
    print("\n" + "=" * 100)
    print(f"HOLDOUT SIN FUGA — DIMENSIÓN {dimension}")
    print("=" * 100)

    resultados_holdout[int(dimension)] = ejecutar_holdout_sin_fuga_revision(
        tweets_desarrollo=tweets_desarrollo,
        tokens_trends=tokens_trends,
        dimension=int(dimension),
        semillas=semillas_politicas,
        top_n=EXPANSION_TOP_N_FINAL,
        test_size=TEST_SIZE,
        epochs=W2V_EPOCHS,
        seed=RANDOM_STATE,
    )


HOLDOUT SIN FUGA — DIMENSIÓN 50

HOLDOUT SIN FUGA — DIMENSIÓN 100

HOLDOUT SIN FUGA — DIMENSIÓN 200


In [162]:
resumen_holdout = resumir_holdouts_dimensiones_revision(resultados_holdout)

In [163]:
metricas_holdout = pd.concat(
    [resultado["metricas"] for resultado in resultados_holdout.values()],
    ignore_index=True,
)

In [164]:
mostrar_tabla_bokeh(
    resumen_holdout,
    "Selección de k, expansión y balance por dimensión",
    ancho=1350,
    alto=300,
)

In [165]:
mostrar_tabla_bokeh(
    metricas_holdout,
    "Holdout sin fuga — consistencia con pseudoetiquetado",
    ancho=1500,
    alto=480,
    max_filas=40,
)

In [266]:
tablas_top100_suplementarias = []

for dimension in sorted(resultados_holdout.keys()):
    resultado = resultados_holdout[dimension]
    detalle = (
        resultado["pipeline"]["palabras_anadidas"]
        .copy()
    )
    if detalle.empty:
        continue
    detalle.insert(
        0,
        "Dimension",
        int(dimension)
    )
    detalle = detalle[
        [
            "Dimension",
            "Rango",
            "Palabra",
            "Similitud_coseno_politica",
            "Distancia_coseno_politica",
        ]
    ].copy()
    tablas_top100_suplementarias.append(detalle)

In [267]:
tabla_top100_suplementaria = pd.concat(
    tablas_top100_suplementarias,
    ignore_index=True
)

In [268]:
tabla_top100_suplementaria = (
    tabla_top100_suplementaria
    .rename(
        columns={
            "Dimension": "Embedding_dimension",
            "Rango": "Ranking",
            "Palabra": "Term",
            "Similitud_coseno_politica": "Cosine_similarity",
            "Distancia_coseno_politica": "Cosine_distance",
        }
    )
    .sort_values(
        ["Embedding_dimension", "Ranking"]
    )
    .reset_index(drop=True)
)

In [274]:
verificacion_top100 = (
    tabla_top100_suplementaria
    .groupby("Embedding_dimension")
    .agg(
        N_terms=("Term", "size"),
        Minimum_similarity=("Cosine_similarity", "min"),
        Maximum_similarity=("Cosine_similarity", "max"),
    )
    .reset_index()
)

display(verificacion_top100)

,Embedding_dimension,N_terms,Minimum_similarity,Maximum_similarity
0,50,100,0.695287,0.885212
1,100,100,0.705781,0.900235
2,200,100,0.709782,0.903853


In [272]:
mostrar_tabla_bokeh(
    tabla_top100_suplementaria,
    "Top-100 expanded terms by embedding dimension",
    ancho=1450,
    alto=500,
    max_filas=300,
)

In [166]:
resultado_principal = resultados_holdout[DIMENSION_BENCHMARK]

In [167]:
pipe_principal = resultado_principal["pipeline"]

In [168]:
mostrar_tabla_bokeh(
    pipe_principal["tabla_k"],
    f"Selección objetiva de k — {DIMENSION_BENCHMARK}D",
    ancho=1000,
    alto=330,
)

In [258]:
graficar_validacion_k_revision(
    pipe_principal["tabla_k"],
    DIMENSION_BENCHMARK,
)

In [170]:
print(f"k seleccionado: {pipe_principal['k_seleccionado']}")
print(f"Clúster político: {pipe_principal['cluster_politico']}")
print(f"Top-N solicitado: {pipe_principal['top_n_solicitado']}")
print(f"Términos añadidos: {len(pipe_principal['palabras_anadidas']):,}")
print(f"Umbral coseno emergente: {pipe_principal['umbral_similitud']:.6f}")

k seleccionado: 2
Clúster político: 0
Top-N solicitado: 100
Términos añadidos: 100
Umbral coseno emergente: 0.705781


In [171]:
mostrar_tabla_bokeh(
    resumir_clusters(
        pipe_principal["palabras"],
        pipe_principal["labels_palabras_1"],
    ),
    "Clústeres iniciales del vocabulario de trends",
    ancho=1300,
    alto=320,
)

In [172]:
mostrar_tabla_bokeh(
    pipe_principal["palabras_anadidas"],
    f"Términos incorporados al Escenario 2 — Top-{EXPANSION_TOP_N_FINAL}",
    ancho=1250,
    alto=420,
    max_filas=60,
)

In [173]:
comparacion_clustering, labels_hdbscan = comparar_kmeans_hdbscan_revision(
    resultado_principal,
    HDBSCAN_MIN_CLUSTER_SIZE,
    HDBSCAN_MIN_SAMPLES,
)

In [174]:
mostrar_tabla_bokeh(
    comparacion_clustering,
    "K-Means vs HDBSCAN sobre embeddings de entrenamiento",
    ancho=1150,
    alto=230,
)

In [175]:
graficar_tsne_revision(
    pipe_principal["palabras"],
    pipe_principal["vectores_palabras_cluster"],
    pipe_principal["labels_palabras_1"],
    f"t-SNE — K-Means — {DIMENSION_BENCHMARK}D",
    RANDOM_STATE,
)

In [176]:
if np.any(labels_hdbscan != -1):
    graficar_tsne_revision(
        pipe_principal["palabras"],
        pipe_principal["vectores_palabras_cluster"],
        labels_hdbscan,
        f"t-SNE — HDBSCAN — {DIMENSION_BENCHMARK}D",
        RANDOM_STATE,
    )

In [177]:
sensibilidad_top_n = evaluar_sensibilidad_top_n_revision(
    resultado_principal,
    EXPANSION_TOP_N_CANDIDATOS,
    RANDOM_STATE,
)

In [178]:
mostrar_tabla_bokeh(
    sensibilidad_top_n,
    "Análisis de sensibilidad de la expansión Top-N",
    ancho=1500,
    alto=440,
    max_filas=50,
)

In [265]:
graficar_cobertura_umbral_top_n_revision(
    sensibilidad_top_n,
    dimension=DIMENSION_BENCHMARK
)

In [260]:
graficar_sensibilidad_top_n_revision(sensibilidad_top_n, "F1-score")

In [261]:
graficar_sensibilidad_top_n_revision(sensibilidad_top_n, "Recall")

# 7. Validación cruzada repetida y análisis estadístico

In [181]:
def ejecutar_cv_repetida_revision(tweets_desarrollo: pd.DataFrame, tokens_trends: Sequence[Sequence[str]], dimensiones: Sequence[int], semillas: Sequence[str], top_n: int, n_splits: int, n_repeats: int, epochs: int, seed: int) -> pd.DataFrame:
    """
    Ejecuta validación cruzada repetida reconstruyendo el pipeline dentro de cada fold.

    En cada fold se vuelve a entrenar Word2Vec, seleccionar k, ajustar K-Means, identificar
    el clúster político y expandir el léxico únicamente con los datos de entrenamiento.
    De este modo, ninguna observación del fold de prueba influye en las representaciones
    ni en las pseudoetiquetas de entrenamiento.

    Args:
    tweets_desarrollo (pd.DataFrame): Corpus de desarrollo sin el holdout manual.
    tokens_trends (Sequence[Sequence[str]]): Trends tokenizados.
    dimensiones (Sequence[int]): Dimensiones Word2Vec a evaluar.
    semillas (Sequence[str]): Términos políticos de referencia.
    top_n (int): Regla Top-N de expansión.
    n_splits (int): Número de folds por repetición.
    n_repeats (int): Número de repeticiones completas.
    epochs (int): Épocas Word2Vec dentro de cada fold.
    seed (int): Semilla reproducible.

    Returns:
    pd.DataFrame:
    Métricas crudas de todos los folds, escenarios, dimensiones y algoritmos.

    Example:
    >>> cv = ejecutar_cv_repetida_revision(tweets_desarrollo, tokens_trends, [100], semillas_politicas, 100, 5, 2, 10, 5)
    """
    if n_splits < 2 or n_repeats < 1:
        raise ValueError("Se requieren al menos 2 folds y 1 repetición.")

    estrato = tweets_desarrollo["evento"].fillna("Sin evento").astype(str).to_numpy()
    splitter = RepeatedStratifiedKFold(
        n_splits=int(n_splits),
        n_repeats=int(n_repeats),
        random_state=seed,
    )
    filas: list[pd.DataFrame] = []

    for dimension in dimensiones:
        print(f"\nCV repetida — dimensión {dimension}")
        for contador, (idx_train, idx_test) in enumerate(splitter.split(np.zeros(len(tweets_desarrollo)), estrato), start=1):
            repeticion = (contador - 1) // n_splits + 1
            fold = (contador - 1) % n_splits + 1
            print(f"  Repetición {repeticion}/{n_repeats}, fold {fold}/{n_splits}")

            train_df = tweets_desarrollo.iloc[idx_train].reset_index(drop=True)
            test_df = tweets_desarrollo.iloc[idx_test].reset_index(drop=True)
            tokens_train = preparar_tokens_dataframe(train_df, "tokens_sin_stopwords")
            tokens_test = preparar_tokens_dataframe(test_df, "tokens_sin_stopwords")

            pipe = construir_pipeline_semantico_train(
                tokens_train,
                tokens_trends,
                int(dimension),
                semillas,
                K_CANDIDATOS,
                int(top_n),
                int(epochs),
                seed + contador,
            )
            X_train = vectorizar_documentos(tokens_train, pipe["modelo_word2vec"], agregacion="suma")
            X_test = vectorizar_documentos(tokens_test, pipe["modelo_word2vec"], agregacion="suma")

            for escenario, centroides in [("Escenario 1", pipe["centroides_1"]), ("Escenario 2", pipe["centroides_2"])]:
                y_train, _, _ = etiquetar_por_centroides(X_train, centroides, pipe["cluster_politico"])
                y_test, _, _ = etiquetar_por_centroides(X_test, centroides, pipe["cluster_politico"])
                tabla = entrenar_evaluar_modelos_split(
                    X_train,
                    y_train,
                    X_test,
                    y_test,
                    int(dimension),
                    escenario,
                    seed + contador,
                    KNN_VECINOS,
                )
                tabla["Repeticion"] = int(repeticion)
                tabla["Fold"] = int(fold)
                tabla["k_seleccionado"] = int(pipe["k_seleccionado"])
                tabla["N_terminos_anadidos"] = int(len(pipe["palabras_anadidas"])) if escenario == "Escenario 2" else 0
                tabla["Umbral_similitud"] = float(pipe["umbral_similitud"]) if escenario == "Escenario 2" else np.nan
                tabla["Politicos_test_pct"] = float(y_test.mean() * 100.0)
                filas.append(tabla)

    return pd.concat(filas, ignore_index=True) if filas else pd.DataFrame()

In [182]:
def resumir_cv_intervalos_revision(cv_resultados: pd.DataFrame, metricas: Sequence[str]) -> pd.DataFrame:
    """
    Resume validación cruzada con media, desviación estándar e intervalo de confianza del 95%.

    El intervalo se calcula con la distribución t de Student sobre las observaciones
    de los folds repetidos para cada combinación de dimensión, escenario y algoritmo.

    Args:
    cv_resultados (pd.DataFrame): Resultados crudos de validación cruzada.
    metricas (Sequence[str]): Nombres de las métricas que se desean resumir.

    Returns:
    pd.DataFrame:
    Tabla larga con N, media, desviación estándar y límites IC95%.

    Example:
    >>> resumen_cv = resumir_cv_intervalos_revision(cv, ["F1-score", "Recall"])
    """
    filas: list[dict[str, Any]] = []
    grupos = cv_resultados.groupby(["Dimension", "Escenario", "Algoritmo"], dropna=False)

    for claves, sub in grupos:
        dim, escenario, algoritmo = claves
        for metrica in metricas:
            valores = pd.to_numeric(sub[metrica], errors="coerce").dropna().to_numpy(dtype=float)
            n = len(valores)
            if n == 0:
                continue
            media = float(np.mean(valores))
            std = float(np.std(valores, ddof=1)) if n > 1 else 0.0
            if n > 1:
                margen = float(student_t.ppf(0.975, df=n - 1) * std / math.sqrt(n))
            else:
                margen = 0.0
            limite_inf = max(0.0, media - margen)
            limite_sup = min(1.0, media + margen)
            filas.append({
                "Dimension": int(dim),
                "Escenario": str(escenario),
                "Algoritmo": str(algoritmo),
                "Metrica": str(metrica),
                "N": int(n),
                "Media": media,
                "Desv_std": std,
                "IC95_inf": limite_inf,
                "IC95_sup": limite_sup,
            })

    return pd.DataFrame(filas)

In [183]:
def ajustar_p_holm(p_values: Sequence[float]) -> np.ndarray:
    """
    Ajusta una colección de p-valores con el procedimiento secuencial de Holm.

    El ajuste controla el error familiar cuando se ejecutan varias comparaciones
    pareadas entre escenarios.

    Args:
    p_values (Sequence[float]): P-valores sin ajustar.

    Returns:
    np.ndarray:
    P-valores ajustados en el mismo orden de entrada.

    Example:
    >>> ajustar_p_holm([0.01, 0.04, 0.20])
    array([0.03, 0.08, 0.2 ])
    """
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    if m == 0:
        return np.array([], dtype=float)

    orden = np.argsort(p)
    ajustados_orden = np.empty(m, dtype=float)
    max_prev = 0.0
    for rango, idx in enumerate(orden):
        valor = min(1.0, (m - rango) * p[idx])
        max_prev = max(max_prev, valor)
        ajustados_orden[rango] = max_prev

    salida = np.empty(m, dtype=float)
    for rango, idx in enumerate(orden):
        salida[idx] = ajustados_orden[rango]
    return salida

In [184]:
def pruebas_wilcoxon_escenarios_revision(cv_resultados: pd.DataFrame, metricas: Sequence[str]) -> pd.DataFrame:
    """
    Compara Escenario 1 y Escenario 2 con Wilcoxon pareado sobre los mismos folds.

    Para cada dimensión, algoritmo y métrica se emparejan resultados por repetición
    y fold. Se reporta el estadístico, p-valor sin ajustar, diferencia media y un
    p-valor ajustado por Holm para múltiples comparaciones.

    Args:
    cv_resultados (pd.DataFrame): Resultados crudos de validación cruzada repetida.
    metricas (Sequence[str]): Métricas que se desean comparar.

    Returns:
    pd.DataFrame:
    Tabla de pruebas pareadas y p-valores ajustados.

    Example:
    >>> pruebas = pruebas_wilcoxon_escenarios_revision(cv, ["F1-score", "Recall"])
    """
    filas: list[dict[str, Any]] = []
    for (dimension, algoritmo), sub in cv_resultados.groupby(["Dimension", "Algoritmo"]):
        for metrica in metricas:
            piv = sub.pivot_table(
                index=["Repeticion", "Fold"],
                columns="Escenario",
                values=metrica,
                aggfunc="first",
            ).dropna()
            if "Escenario 1" not in piv.columns or "Escenario 2" not in piv.columns or len(piv) < 2:
                continue

            a = piv["Escenario 1"].to_numpy(dtype=float)
            b = piv["Escenario 2"].to_numpy(dtype=float)
            if np.allclose(a, b):
                estadistico, p_valor = 0.0, 1.0
            else:
                prueba = wilcoxon(b, a, alternative="two-sided", zero_method="wilcox")
                estadistico, p_valor = float(prueba.statistic), float(prueba.pvalue)

            filas.append({
                "Dimension": int(dimension),
                "Algoritmo": str(algoritmo),
                "Metrica": str(metrica),
                "N_pares": int(len(piv)),
                "Media_E1": float(np.mean(a)),
                "Media_E2": float(np.mean(b)),
                "Diferencia_media_E2_menos_E1": float(np.mean(b - a)),
                "Wilcoxon_W": estadistico,
                "p_valor": p_valor,
            })

    tabla = pd.DataFrame(filas)
    if not tabla.empty:
        tabla["p_ajustado_Holm"] = ajustar_p_holm(tabla["p_valor"].to_numpy())
        tabla["Significativo_0_05"] = tabla["p_ajustado_Holm"] < 0.05
    return tabla

In [185]:
def graficar_cv_intervalos_revision(resumen_cv: pd.DataFrame, metrica: str = "F1-score") -> None:
    """
    Grafica medias e intervalos de confianza del 95% para una métrica de validación cruzada.

    La visualización usa Bokeh y presenta cada algoritmo/escenario como una categoría
    independiente, facilitando comparar estabilidad además del valor promedio.

    Args:
    resumen_cv (pd.DataFrame): Resumen producido por `resumir_cv_intervalos_revision`.
    metrica (str): Métrica que se desea visualizar.
    *Valor por defecto: "F1-score".*

    Returns:
    None:
    Muestra la gráfica directamente en el notebook.

    Example:
    >>> graficar_cv_intervalos_revision(resumen_cv, "F1-score")
    """
    datos = resumen_cv[resumen_cv["Metrica"] == metrica].copy()
    if datos.empty:
        print(f"No existen datos para la métrica {metrica}.")
        return

    datos["Categoria"] = (
        datos["Dimension"].astype(str)
        + "D | "
        + datos["Algoritmo"].astype(str)
        + " | "
        + datos["Escenario"].astype(str).str.replace("Escenario ", "E", regex=False)
    )
    datos = datos.sort_values(["Dimension", "Algoritmo", "Escenario"]).reset_index(drop=True)
    source = ColumnDataSource(datos)
    p = figure(
        x_range=datos["Categoria"].tolist(),
        height=480,
        width=1100,
        title=f"{metrica}: media e IC95% — validación cruzada repetida",
        y_axis_label=metrica,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    p.scatter(x="Categoria", y="Media", source=source, size=9)
    p.segment(x0="Categoria", y0="IC95_inf", x1="Categoria", y1="IC95_sup", source=source, line_width=2)
    p.xaxis.major_label_orientation = 1.1
    p.add_tools(HoverTool(tooltips=[
        ("Configuración", "@Categoria"),
        ("Media", "@Media{0.0000}"),
        ("Std", "@Desv_std{0.0000}"),
        ("IC95", "[@IC95_inf{0.0000}, @IC95_sup{0.0000}]"),
        ("N", "@N"),
    ]))
    show(p)

In [186]:
cv_resultados = ejecutar_cv_repetida_revision(
    tweets_desarrollo=tweets_desarrollo,
    tokens_trends=tokens_trends,
    dimensiones=DIMENSIONES,
    semillas=semillas_politicas,
    top_n=EXPANSION_TOP_N_FINAL,
    n_splits=CV_FOLDS,
    n_repeats=CV_REPEATS,
    epochs=CV_EPOCHS,
    seed=RANDOM_STATE,
)

cv_resumen = resumir_cv_intervalos_revision(
    cv_resultados,
    list(METRICAS_ESTADISTICAS),
)

wilcoxon_escenarios = pruebas_wilcoxon_escenarios_revision(
    cv_resultados,
    ["Precisión", "Recall", "F1-score", "Valor F2"],
)

mostrar_tabla_bokeh(
    cv_resumen,
    "Validación cruzada repetida — media, desviación e IC95%",
    ancho=1450,
    alto=500,
    max_filas=70,
)
mostrar_tabla_bokeh(
    wilcoxon_escenarios,
    "Wilcoxon pareado Escenario 2 vs Escenario 1 + Holm",
    ancho=1400,
    alto=430,
    max_filas=60,
)
graficar_cv_intervalos_revision(cv_resumen, "F1-score")


CV repetida — dimensión 50
  Repetición 1/2, fold 1/5
  Repetición 1/2, fold 2/5
  Repetición 1/2, fold 3/5
  Repetición 1/2, fold 4/5
  Repetición 1/2, fold 5/5
  Repetición 2/2, fold 1/5
  Repetición 2/2, fold 2/5
  Repetición 2/2, fold 3/5
  Repetición 2/2, fold 4/5
  Repetición 2/2, fold 5/5

CV repetida — dimensión 100
  Repetición 1/2, fold 1/5
  Repetición 1/2, fold 2/5
  Repetición 1/2, fold 3/5
  Repetición 1/2, fold 4/5
  Repetición 1/2, fold 5/5
  Repetición 2/2, fold 1/5
  Repetición 2/2, fold 2/5
  Repetición 2/2, fold 3/5
  Repetición 2/2, fold 4/5
  Repetición 2/2, fold 5/5

CV repetida — dimensión 200
  Repetición 1/2, fold 1/5
  Repetición 1/2, fold 2/5
  Repetición 1/2, fold 3/5
  Repetición 1/2, fold 4/5
  Repetición 1/2, fold 5/5
  Repetición 2/2, fold 1/5
  Repetición 2/2, fold 2/5
  Repetición 2/2, fold 3/5
  Repetición 2/2, fold 4/5
  Repetición 2/2, fold 5/5


In [289]:
metricas_apendice_b = (
    wilcoxon_escenarios["Metrica"]
    .drop_duplicates()
    .tolist()
)

In [290]:
print(
    "Métricas incluidas en Appendix B:",
    metricas_apendice_b
)

Métricas incluidas en Appendix B: ['Precisión', 'Recall', 'F1-score', 'Valor F2']


In [291]:
cv_apendice_b = cv_resumen[
    cv_resumen["Metrica"].isin(
        metricas_apendice_b
    )
].copy()

In [292]:
cv_e1 = (
    cv_apendice_b[
        cv_apendice_b["Escenario"] == "Escenario 1"
    ][
        [
            "Dimension",
            "Algoritmo",
            "Metrica",
            "N",
            "Media",
            "Desv_std",
            "IC95_inf",
            "IC95_sup",
        ]
    ]
    .copy()
    .rename(
        columns={
            "N": "N_E1",
            "Media": "E1_Mean",
            "Desv_std": "E1_SD",
            "IC95_inf": "E1_CI95_Lower",
            "IC95_sup": "E1_CI95_Upper",
        }
    )
)

In [293]:
cv_e2 = (
    cv_apendice_b[
        cv_apendice_b["Escenario"] == "Escenario 2"
    ][
        [
            "Dimension",
            "Algoritmo",
            "Metrica",
            "N",
            "Media",
            "Desv_std",
            "IC95_inf",
            "IC95_sup",
        ]
    ]
    .copy()
    .rename(
        columns={
            "N": "N_E2",
            "Media": "E2_Mean",
            "Desv_std": "E2_SD",
            "IC95_inf": "E2_CI95_Lower",
            "IC95_sup": "E2_CI95_Upper",
        }
    )
)

In [294]:
tabla_appendix_b_cv = cv_e1.merge(
    cv_e2,
    on=[
        "Dimension",
        "Algoritmo",
        "Metrica",
    ],
    how="inner",
    validate="one_to_one",
)

In [295]:
wilcoxon_apendice_b = wilcoxon_escenarios[
    [
        "Dimension",
        "Algoritmo",
        "Metrica",
        "N_pares",
        "Media_E1",
        "Media_E2",
        "Diferencia_media_E2_menos_E1",
        "Wilcoxon_W",
        "p_valor",
        "p_ajustado_Holm",
        "Significativo_0_05",
    ]
].copy()

In [296]:
tabla_appendix_b_cv = tabla_appendix_b_cv.merge(
    wilcoxon_apendice_b,
    on=[
        "Dimension",
        "Algoritmo",
        "Metrica",
    ],
    how="inner",
    validate="one_to_one",
)

In [297]:
if not np.allclose(
    tabla_appendix_b_cv["E1_Mean"],
    tabla_appendix_b_cv["Media_E1"],
    equal_nan=True,
):
    raise ValueError(
        "Las medias de Scenario 1 de cv_resumen y "
        "wilcoxon_escenarios no coinciden."
    )

In [298]:
if not np.allclose(
    tabla_appendix_b_cv["E2_Mean"],
    tabla_appendix_b_cv["Media_E2"],
    equal_nan=True,
):
    raise ValueError(
        "Las medias de Scenario 2 de cv_resumen y "
        "wilcoxon_escenarios no coinciden."
    )

In [299]:
tabla_appendix_b_cv = (
    tabla_appendix_b_cv
    .drop(
        columns=[
            "Media_E1",
            "Media_E2",
        ]
    )
)

In [300]:
tabla_appendix_b_cv = (
    tabla_appendix_b_cv
    .rename(
        columns={
            "Dimension": "Embedding_dimension",
            "Algoritmo": "Model",
            "Metrica": "Metric",
            "N_pares": "N_pairs",
            "Diferencia_media_E2_menos_E1":
                "Mean_difference_E2_minus_E1",
            "p_valor": "p_value",
            "p_ajustado_Holm": "Holm_adjusted_p",
            "Significativo_0_05":
                "Significant_after_Holm",
        }
    )
)

In [301]:
mapa_modelos = {
    "Árbol de Decisión": "Decision Tree",
    "KNN": "KNN",
    "Naïve Bayes": "Naive Bayes",
    "SVM": "SVM",
}

mapa_metricas = {
    "Precisión": "Precision",
    "Recall": "Recall",
    "F1-score": "F1-score",
    "Valor F2": "F2-score",
}

In [302]:
tabla_appendix_b_cv["Model"] = (
    tabla_appendix_b_cv["Model"]
    .replace(mapa_modelos)
)

tabla_appendix_b_cv["Metric"] = (
    tabla_appendix_b_cv["Metric"]
    .replace(mapa_metricas)
)

In [303]:
orden_modelos = {
    "Decision Tree": 0,
    "KNN": 1,
    "Naive Bayes": 2,
    "SVM": 3,
}

orden_metricas = {
    "Precision": 0,
    "Recall": 1,
    "F1-score": 2,
    "F2-score": 3,
}

In [304]:
tabla_appendix_b_cv["_orden_modelo"] = (
    tabla_appendix_b_cv["Model"]
    .map(orden_modelos)
)

tabla_appendix_b_cv["_orden_metrica"] = (
    tabla_appendix_b_cv["Metric"]
    .map(orden_metricas)
)

tabla_appendix_b_cv = (
    tabla_appendix_b_cv
    .sort_values(
        [
            "Embedding_dimension",
            "_orden_modelo",
            "_orden_metrica",
        ]
    )
    .drop(
        columns=[
            "_orden_modelo",
            "_orden_metrica",
        ]
    )
    .reset_index(drop=True)
)

In [305]:
n_esperado = len(
    wilcoxon_escenarios
)

In [306]:
if len(tabla_appendix_b_cv) != n_esperado:
    raise ValueError(
        "El número de filas del Appendix B no coincide "
        f"con Wilcoxon. Esperadas: {n_esperado}. "
        f"Obtenidas: {len(tabla_appendix_b_cv)}."
    )

In [307]:
columnas_criticas = [
    "E1_Mean",
    "E1_SD",
    "E1_CI95_Lower",
    "E1_CI95_Upper",
    "E2_Mean",
    "E2_SD",
    "E2_CI95_Lower",
    "E2_CI95_Upper",
    "Wilcoxon_W",
    "p_value",
    "Holm_adjusted_p",
]

if tabla_appendix_b_cv[
    columnas_criticas
].isna().any().any():
    raise ValueError(
        "Appendix B contiene valores faltantes "
        "en columnas estadísticas obligatorias."
    )

In [308]:
tabla_appendix_b_cv_table = (
    tabla_appendix_b_cv[
        [
            "Embedding_dimension",
            "Model",
            "Metric",
            "E1_Mean",
            "E1_SD",
            "E1_CI95_Lower",
            "E1_CI95_Upper",
            "E2_Mean",
            "E2_SD",
            "E2_CI95_Lower",
            "E2_CI95_Upper",
            "Wilcoxon_W",
            "p_value",
            "Holm_adjusted_p",
            "Significant_after_Holm",
        ]
    ]
    .copy()
)

In [309]:
tabla_appendix_b_cv_table["Scenario_1_Mean_SD"] = (
    tabla_appendix_b_cv_table.apply(
        lambda fila:
        f"{fila['E1_Mean']:.4f} ± "
        f"{fila['E1_SD']:.4f}",
        axis=1,
    )
)

In [310]:
tabla_appendix_b_cv_table["Scenario_1_CI95"] = (
    tabla_appendix_b_cv_table.apply(
        lambda fila:
        f"[{fila['E1_CI95_Lower']:.4f}, "
        f"{fila['E1_CI95_Upper']:.4f}]",
        axis=1,
    )
)

In [311]:
tabla_appendix_b_cv_table["Scenario_2_Mean_SD"] = (
    tabla_appendix_b_cv_table.apply(
        lambda fila:
        f"{fila['E2_Mean']:.4f} ± "
        f"{fila['E2_SD']:.4f}",
        axis=1,
    )
)

In [312]:
tabla_appendix_b_cv_table["Scenario_2_CI95"] = (
    tabla_appendix_b_cv_table.apply(
        lambda fila:
        f"[{fila['E2_CI95_Lower']:.4f}, "
        f"{fila['E2_CI95_Upper']:.4f}]",
        axis=1,
    )
)

In [313]:
tabla_appendix_b_cv_table = (
    tabla_appendix_b_cv_table[
        [
            "Embedding_dimension",
            "Model",
            "Metric",
            "Scenario_1_Mean_SD",
            "Scenario_1_CI95",
            "Scenario_2_Mean_SD",
            "Scenario_2_CI95",
            "Wilcoxon_W",
            "p_value",
            "Holm_adjusted_p",
            "Significant_after_Holm",
        ]
    ]
)

In [314]:
mostrar_tabla_bokeh(
    tabla_appendix_b_cv_table,
    "Complete repeated cross-validation and statistical results",
    ancho=1750,
    alto=520,
    max_filas=60,
)

In [315]:
print(
    "Número total de filas:",
    len(tabla_appendix_b_cv)
)

print(
    "Dimensiones:",
    sorted(
        tabla_appendix_b_cv[
            "Embedding_dimension"
        ].unique().tolist()
    )
)

print(
    "Modelos:",
    tabla_appendix_b_cv[
        "Model"
    ].unique().tolist()
)

print(
    "Métricas:",
    tabla_appendix_b_cv[
        "Metric"
    ].unique().tolist()
)

print(
    "Comparaciones significativas después de Holm:",
    int(
        tabla_appendix_b_cv[
            "Significant_after_Holm"
        ].sum()
    )
)

Número total de filas: 48
Dimensiones: [50, 100, 200]
Modelos: ['Decision Tree', 'KNN', 'Naive Bayes', 'SVM']
Métricas: ['Precision', 'Recall', 'F1-score', 'F2-score']
Comparaciones significativas después de Holm: 0


# 8. Validación externa hashtag-blind

In [187]:
def evaluar_validacion_hashtags_revision(tweets_desarrollo: pd.DataFrame, holdout_silver: pd.DataFrame, trends_ciegos: pd.DataFrame, dimension: int, semillas: Sequence[str], top_n: int, epochs: int, seed: int) -> dict[str, Any]:
    """
    Evalúa pseudoetiquetado y clasificadores contra un silver standard basado en hashtags.

    Word2Vec, K-Means, selección del clúster político y expansión se construyen solo con
    tweets de desarrollo que no pertenecen al holdout hashtag. Además, todos los hashtags
    se retiran de las features. Las etiquetas hashtag se usan únicamente al final para
    medir concordancia externa de los dos escenarios y de los cuatro clasificadores.

    Args:
    tweets_desarrollo (pd.DataFrame): Corpus ciego que excluye el holdout silver.
    holdout_silver (pd.DataFrame): Tweets ciegos con `etiqueta_silver_hashtag`.
    trends_ciegos (pd.DataFrame): Trends con hashtags retirados para evitar señal directa.
    dimension (int): Dimensión Word2Vec del experimento.
    semillas (Sequence[str]): Semillas políticas sin hashtags.
    top_n (int): Número explícito de términos añadidos al léxico político.
    epochs (int): Épocas de entrenamiento Word2Vec.
    seed (int): Semilla reproducible.

    Returns:
    dict[str, Any]:
    Pipeline, métricas, predicciones, McNemar, métricas por evento y balance del silver standard.

    Example:
    >>> resultado = evaluar_validacion_hashtags_revision(dev_hash, holdout_hash, trends_hash, 100, semillas_sin_hash, 100, 10, 5)
    """
    if "etiqueta_silver_hashtag" not in holdout_silver.columns:
        raise KeyError("El holdout debe contener `etiqueta_silver_hashtag`.")

    y_silver = holdout_silver["etiqueta_silver_hashtag"].astype(int).to_numpy()
    if len(np.unique(y_silver)) < 2:
        raise ValueError("El silver standard debe contener ambas clases 0 y 1.")

    tokens_train = preparar_tokens_dataframe(tweets_desarrollo, "tokens_sin_stopwords")
    tokens_holdout = preparar_tokens_dataframe(holdout_silver, "tokens_sin_stopwords")
    tokens_trends = preparar_tokens_dataframe(trends_ciegos, "tokens_sin_stopwords")

    pipe = construir_pipeline_semantico_train(
        tokens_train,
        tokens_trends,
        int(dimension),
        semillas,
        K_CANDIDATOS,
        int(top_n),
        int(epochs),
        int(seed),
    )

    X_train = vectorizar_documentos(tokens_train, pipe["modelo_word2vec"], agregacion="suma")
    X_holdout = vectorizar_documentos(tokens_holdout, pipe["modelo_word2vec"], agregacion="suma")

    y1_train, _, _ = etiquetar_por_centroides(X_train, pipe["centroides_1"], pipe["cluster_politico"])
    y2_train, _, _ = etiquetar_por_centroides(X_train, pipe["centroides_2"], pipe["cluster_politico"])
    y1_holdout, _, _ = etiquetar_por_centroides(X_holdout, pipe["centroides_1"], pipe["cluster_politico"])
    y2_holdout, _, _ = etiquetar_por_centroides(X_holdout, pipe["centroides_2"], pipe["cluster_politico"])

    filas: list[dict[str, Any]] = []
    predicciones: dict[str, np.ndarray] = {
        "Pseudo_E1": y1_holdout.astype(int),
        "Pseudo_E2": y2_holdout.astype(int),
    }

    for escenario, y_train, y_pseudo in [
        ("Escenario 1", y1_train, y1_holdout),
        ("Escenario 2", y2_train, y2_holdout),
    ]:
        fila_pseudo: dict[str, Any] = {
            "Dimension": int(dimension),
            "Escenario": escenario,
            "Modelo": "Pseudoetiquetado directo",
            "Referencia": "Silver standard de hashtags — features sin hashtags",
            "N_holdout": int(len(y_silver)),
        }
        fila_pseudo.update(calcular_metricas_silver_hashtags(y_silver, y_pseudo))
        filas.append(fila_pseudo)

        for nombre, modelo in crear_modelos_clasificacion(KNN_VECINOS, seed).items():
            modelo.fit(X_train, y_train)
            pred = np.asarray(modelo.predict(X_holdout)).astype(int)
            predicciones[f"{nombre}_{'E1' if escenario == 'Escenario 1' else 'E2'}"] = pred
            fila: dict[str, Any] = {
                "Dimension": int(dimension),
                "Escenario": escenario,
                "Modelo": nombre,
                "Referencia": "Silver standard de hashtags — features sin hashtags",
                "N_holdout": int(len(y_silver)),
            }
            fila.update(calcular_metricas_silver_hashtags(y_silver, pred))
            filas.append(fila)

    # Resultado natural y resultado balanceado del pseudoetiquetado directo.
    rng = np.random.default_rng(seed)
    idx0 = np.flatnonzero(y_silver == 0)
    idx1 = np.flatnonzero(y_silver == 1)
    n_balance = int(min(len(idx0), len(idx1)))
    idx_balance = np.concatenate([
        rng.choice(idx0, size=n_balance, replace=False),
        rng.choice(idx1, size=n_balance, replace=False),
    ]) if n_balance > 0 else np.arange(len(y_silver))
    rng.shuffle(idx_balance)

    balance_filas: list[dict[str, Any]] = []
    for nombre_pred in ["Pseudo_E1", "Pseudo_E2"]:
        fila = {
            "Prediccion": nombre_pred,
            "Muestra": "Natural",
            "N": int(len(y_silver)),
            "Positivos": int(y_silver.sum()),
            "Negativos": int((y_silver == 0).sum()),
        }
        fila.update(calcular_metricas_silver_hashtags(y_silver, predicciones[nombre_pred]))
        balance_filas.append(fila)
        fila_bal = {
            "Prediccion": nombre_pred,
            "Muestra": "Balanceada 1:1",
            "N": int(len(idx_balance)),
            "Positivos": int(y_silver[idx_balance].sum()),
            "Negativos": int((y_silver[idx_balance] == 0).sum()),
        }
        fila_bal.update(calcular_metricas_silver_hashtags(y_silver[idx_balance], predicciones[nombre_pred][idx_balance]))
        balance_filas.append(fila_bal)

    por_evento: list[dict[str, Any]] = []
    if "evento" in holdout_silver.columns:
        for evento, indices in holdout_silver.groupby("evento").groups.items():
            idx = np.asarray(list(indices), dtype=int)
            if len(np.unique(y_silver[idx])) < 2:
                continue
            for nombre_pred in ["Pseudo_E1", "Pseudo_E2"]:
                fila_evento: dict[str, Any] = {
                    "Dimension": int(dimension),
                    "Evento": str(evento),
                    "Prediccion": nombre_pred,
                    "N": int(len(idx)),
                }
                fila_evento.update(calcular_metricas_silver_hashtags(y_silver[idx], predicciones[nombre_pred][idx]))
                por_evento.append(fila_evento)

    mcnemar = prueba_mcnemar_exacta(y_silver, y1_holdout, y2_holdout)
    mcnemar.insert(0, "Dimension", int(dimension))
    mcnemar.insert(1, "Comparacion", "Pseudo E1 vs Pseudo E2")

    return {
        "dimension": int(dimension),
        "pipeline": pipe,
        "metricas": pd.DataFrame(filas),
        "metricas_balance": pd.DataFrame(balance_filas),
        "metricas_evento": pd.DataFrame(por_evento),
        "mcnemar": mcnemar,
        "y_silver": y_silver,
        "predicciones": predicciones,
        "holdout": holdout_silver.reset_index(drop=True),
    }

In [188]:
def evaluar_auditoria_humana_hashtag(
    anotaciones: pd.DataFrame,
    resultado_silver: Mapping[str, Any],
) -> dict[str, pd.DataFrame]:
    """
    Evalúa silver, pseudoetiquetas y clasificadores contra la segunda anotación humana.

    Une por `id_registro` la muestra anotada antes del modelado con las predicciones
    obtenidas posteriormente. De esta forma el archivo
    `anotacion_desacuerdos_hashtag.xlsx` participa realmente en la evaluación final.

    Args:
    anotaciones (pd.DataFrame): Segunda plantilla completa con `etiqueta_humana`.
    resultado_silver (Mapping[str, Any]): Resultado completo de validación hashtag-blind.

    Returns:
    dict[str, pd.DataFrame]:
    Tablas `metricas`, `mcnemar` y `detalle`.

    Example:
    >>> auditoria = evaluar_auditoria_humana_hashtag(
    ...     ANOTACIONES_HASHTAG_PREVALIDADAS, resultados_silver[100]
    ... )
    """
    holdout = resultado_silver["holdout"].copy().reset_index(drop=True)
    if "id_registro" not in holdout.columns:
        raise KeyError("El holdout silver no contiene `id_registro`.")

    detalle = holdout[
        [c for c in [
            "id_registro",
            "evento",
            "texto_para_anotacion",
            "texto_limpio",
            "etiqueta_silver_hashtag",
        ] if c in holdout.columns]
    ].copy()

    predicciones = resultado_silver["predicciones"]
    for nombre, valores in predicciones.items():
        detalle[nombre] = np.asarray(valores).astype(int)

    humanas = anotaciones[["id_registro", "etiqueta_humana"]].copy()
    humanas["id_registro"] = humanas["id_registro"].astype(str)
    detalle["id_registro"] = detalle["id_registro"].astype(str)

    detalle = humanas.merge(
        detalle,
        on="id_registro",
        how="inner",
        validate="one_to_one",
    )

    if len(detalle) != len(humanas):
        raise RuntimeError(
            "No todas las anotaciones hashtag pudieron unirse con el holdout silver."
        )

    y_humano = detalle["etiqueta_humana"].astype(int).to_numpy()
    dimension = int(resultado_silver["dimension"])

    columnas_pred = ["etiqueta_silver_hashtag"] + list(predicciones.keys())
    filas: list[dict[str, Any]] = []

    for columna in columnas_pred:
        y_pred = detalle[columna].astype(int).to_numpy()
        fila: dict[str, Any] = {
            "Dimension": dimension,
            "Prediccion": columna,
            "Referencia": "Anotación humana ciega del silver standard",
            "N": int(len(detalle)),
        }
        fila.update(calcular_metricas_silver_hashtags(y_humano, y_pred))
        filas.append(fila)

    comparaciones: list[pd.DataFrame] = []

    if "Pseudo_E2" in detalle.columns:
        prueba = prueba_mcnemar_exacta(
            y_humano,
            detalle["etiqueta_silver_hashtag"].astype(int).to_numpy(),
            detalle["Pseudo_E2"].astype(int).to_numpy(),
        )
        prueba.insert(0, "Dimension", dimension)
        prueba.insert(1, "Comparacion", "Silver hashtag vs Pseudo E2")
        comparaciones.append(prueba)

    if {"Pseudo_E1", "Pseudo_E2"}.issubset(detalle.columns):
        prueba = prueba_mcnemar_exacta(
            y_humano,
            detalle["Pseudo_E1"].astype(int).to_numpy(),
            detalle["Pseudo_E2"].astype(int).to_numpy(),
        )
        prueba.insert(0, "Dimension", dimension)
        prueba.insert(1, "Comparacion", "Pseudo E1 vs Pseudo E2")
        comparaciones.append(prueba)

    detalle["acierto_silver"] = (
        detalle["etiqueta_silver_hashtag"].astype(int)
        == detalle["etiqueta_humana"].astype(int)
    )
    if "Pseudo_E1" in detalle.columns:
        detalle["acierto_pseudo_E1"] = (
            detalle["Pseudo_E1"].astype(int)
            == detalle["etiqueta_humana"].astype(int)
        )
    if "Pseudo_E2" in detalle.columns:
        detalle["acierto_pseudo_E2"] = (
            detalle["Pseudo_E2"].astype(int)
            == detalle["etiqueta_humana"].astype(int)
        )
        detalle["desacuerdo_silver_pseudoE2"] = (
            detalle["etiqueta_silver_hashtag"].astype(int)
            != detalle["Pseudo_E2"].astype(int)
        )

    mcnemar = (
        pd.concat(comparaciones, ignore_index=True)
        if comparaciones
        else pd.DataFrame()
    )

    return {
        "metricas": pd.DataFrame(filas),
        "mcnemar": mcnemar,
        "detalle": detalle,
    }

In [189]:
def actualizar_auditoria_modelo_hashtag(
    ruta: Path,
    detalle: pd.DataFrame,
) -> Path:
    """
    Agrega al segundo Excel una hoja posterior con etiquetas y predicciones del modelo.

    La hoja `Anotacion` se preserva. `Auditoria_Modelo` solo se crea después de que la
    anotación humana ya fue completada y validada, evitando sesgo durante el etiquetado.

    Args:
    ruta (Path): Archivo `anotacion_desacuerdos_hashtag.xlsx`.
    detalle (pd.DataFrame): Tabla unida con humano, silver y predicciones.

    Returns:
    Path:
    Ruta del mismo archivo actualizado.

    Example:
    >>> actualizar_auditoria_modelo_hashtag(
    ...     RUTA_ANOTACION_DESACUERDOS_HASHTAG, auditoria_hash["detalle"]
    ... )
    """
    if not ruta.exists():
        raise FileNotFoundError(f"No existe el archivo: {ruta}")

    with pd.ExcelWriter(
        ruta,
        engine="openpyxl",
        mode="a",
        if_sheet_exists="replace",
    ) as writer:
        detalle.to_excel(
            writer,
            sheet_name="Auditoria_Modelo",
            index=False,
        )

    return ruta

In [190]:
resultados_silver: dict[int, dict[str, Any]] = {}

In [191]:
for dimension in DIMENSIONES:
    print("\n" + "=" * 100)
    print(f"VALIDACIÓN HASHTAG-BLIND — DIMENSIÓN {dimension}")
    print("=" * 100)

    resultados_silver[int(dimension)] = evaluar_validacion_hashtags_revision(
        tweets_desarrollo=tweets_desarrollo_ciego_hashtags,
        holdout_silver=silver_hashtags_ciego,
        trends_ciegos=trends_ciegos_hashtags,
        dimension=int(dimension),
        semillas=semillas_politicas_sin_hashtags,
        top_n=EXPANSION_TOP_N_FINAL,
        epochs=W2V_EPOCHS,
        seed=RANDOM_STATE,
    )


VALIDACIÓN HASHTAG-BLIND — DIMENSIÓN 50

VALIDACIÓN HASHTAG-BLIND — DIMENSIÓN 100

VALIDACIÓN HASHTAG-BLIND — DIMENSIÓN 200


In [192]:
metricas_silver = pd.concat(
    [r["metricas"] for r in resultados_silver.values()],
    ignore_index=True,
)

In [193]:
metricas_silver_balance = pd.concat(
    [r["metricas_balance"] for r in resultados_silver.values()],
    ignore_index=True,
)

In [194]:
mcnemar_silver = pd.concat(
    [r["mcnemar"] for r in resultados_silver.values()],
    ignore_index=True,
)

In [195]:
metricas_silver_evento = pd.concat(
    [
        r["metricas_evento"]
        for r in resultados_silver.values()
        if not r["metricas_evento"].empty
    ],
    ignore_index=True,
) if any(not r["metricas_evento"].empty for r in resultados_silver.values()) else pd.DataFrame()

In [196]:
mostrar_tabla_bokeh(
    metricas_silver,
    "Concordancia con silver standard de hashtags",
    ancho=1650,
    alto=520,
    max_filas=60,
)

In [197]:
mostrar_tabla_bokeh(
    metricas_silver_balance,
    "Silver standard — distribución natural y evaluación balanceada 1:1",
    ancho=1550,
    alto=340,
    max_filas=40,
)

In [198]:
mostrar_tabla_bokeh(
    mcnemar_silver,
    "McNemar exacto — Escenario 1 vs Escenario 2",
    ancho=1050,
    alto=240,
)

In [199]:
if not metricas_silver_evento.empty:
    mostrar_tabla_bokeh(
        metricas_silver_evento,
        "Concordancia silver por evento",
        ancho=1550,
        alto=360,
        max_filas=40,
    )

In [200]:
resultado_silver_principal = resultados_silver[DIMENSION_BENCHMARK]

In [201]:
graficar_matriz_confusion_hashtags_revision(
    resultado_silver_principal["y_silver"],
    resultado_silver_principal["predicciones"]["Pseudo_E1"],
    f"Silver hashtags vs Pseudo E1 — {DIMENSION_BENCHMARK}D",
)

In [202]:
graficar_matriz_confusion_hashtags_revision(
    resultado_silver_principal["y_silver"],
    resultado_silver_principal["predicciones"]["Pseudo_E2"],
    f"Silver hashtags vs Pseudo E2 — {DIMENSION_BENCHMARK}D",
)

In [203]:
AUDITORIAS_HUMANAS_HASHTAG: dict[int, dict[str, pd.DataFrame]] = {}
for dimension, resultado in resultados_silver.items():
    AUDITORIAS_HUMANAS_HASHTAG[int(dimension)] = evaluar_auditoria_humana_hashtag(
        ANOTACIONES_HASHTAG_PREVALIDADAS,
        resultado,
    )

In [204]:
METRICAS_AUDITORIA_HUMANA_HASHTAG = pd.concat(
    [
        auditoria["metricas"]
        for auditoria in AUDITORIAS_HUMANAS_HASHTAG.values()
    ],
    ignore_index=True,
)

In [205]:
MCNEMAR_AUDITORIA_HUMANA_HASHTAG = pd.concat(
    [
        auditoria["mcnemar"]
        for auditoria in AUDITORIAS_HUMANAS_HASHTAG.values()
        if not auditoria["mcnemar"].empty
    ],
    ignore_index=True,
)

In [206]:
DETALLE_AUDITORIA_HUMANA_HASHTAG = (
    AUDITORIAS_HUMANAS_HASHTAG[DIMENSION_BENCHMARK]["detalle"]
)

In [207]:
mostrar_tabla_bokeh(
    METRICAS_AUDITORIA_HUMANA_HASHTAG,
    "Validación humana del silver standard y del pseudoetiquetado",
    ancho=1750,
    alto=520,
    max_filas=80,
)

In [208]:
mostrar_tabla_bokeh(
    MCNEMAR_AUDITORIA_HUMANA_HASHTAG,
    "McNemar usando etiqueta humana: silver vs pseudoetiquetado",
    ancho=1200,
    alto=280,
    max_filas=30,
)

In [209]:
ruta_auditoria_actualizada = actualizar_auditoria_modelo_hashtag(
    RUTA_ANOTACION_DESACUERDOS_HASHTAG,
    DETALLE_AUDITORIA_HUMANA_HASHTAG,
)

In [210]:
print(f"Segundo archivo utilizado y actualizado con Auditoria_Modelo: {ruta_auditoria_actualizada}")

Segundo archivo utilizado y actualizado con Auditoria_Modelo: C:\Users\LABIA\Documents\Tesis cleo EPN\Codigo\data\anotacion_desacuerdos_hashtag.xlsx


# 9. Benchmarks contemporáneos y coste computacional

In [211]:
def entrenar_fasttext_revision(corpus: Sequence[Sequence[str]], dimension: int, window: int, min_count: int, epochs: int, seed: int) -> FastText:
    """
    Entrena FastText con subpalabras sobre el mismo corpus de entrenamiento.

    FastText se añade como baseline de representación estática enriquecida por
    subpalabras. Este experimento complementa Word2Vec y permite observar si términos
    raros, variantes y hashtags se benefician de información morfológica.

    Args:
    corpus (Sequence[Sequence[str]]): Documentos tokenizados de entrenamiento.
    dimension (int): Dimensión de los embeddings.
    window (int): Ventana contextual.
    min_count (int): Frecuencia mínima.
    epochs (int): Épocas de entrenamiento.
    seed (int): Semilla reproducible.

    Returns:
    FastText:
    Modelo FastText entrenado.

    Example:
    >>> ft = entrenar_fasttext_revision(tokens_train, 100, 5, 1, 10, 5)
    """
    corpus_util = [list(doc) for doc in corpus if len(doc) > 0]
    if not corpus_util:
        raise ValueError("El corpus está vacío.")
    modelo = FastText(
        vector_size=int(dimension),
        window=int(window),
        min_count=int(min_count),
        sg=1,
        workers=min(8, max(1, (os.cpu_count() or 2) - 1)),
        seed=int(seed),
    )
    modelo.build_vocab(corpus_iterable=corpus_util)
    modelo.train(corpus_iterable=corpus_util, total_examples=len(corpus_util), epochs=int(epochs))
    return modelo

In [212]:
def vectorizar_documentos_fasttext(tokens_documentos: Sequence[Sequence[str]], modelo: FastText, agregacion: str = "media") -> np.ndarray:
    """
    Convierte documentos en vectores usando embeddings FastText.

    FastText puede obtener vectores para términos fuera del vocabulario explícito a
    partir de subpalabras. Los vectores de cada tweet se agregan mediante media o suma.

    Args:
    tokens_documentos (Sequence[Sequence[str]]): Tweets tokenizados.
    modelo (FastText): Modelo FastText entrenado.
    agregacion (str): Estrategia `media` o `suma`.
    *Valor por defecto: "media".*

    Returns:
    np.ndarray:
    Matriz de documentos vectorizados.

    Example:
    >>> X_ft = vectorizar_documentos_fasttext(tokens_train, ft)
    """
    if agregacion not in {"media", "suma"}:
        raise ValueError("agregacion debe ser 'media' o 'suma'.")

    dim = int(modelo.vector_size)
    salida = np.zeros((len(tokens_documentos), dim), dtype=np.float32)
    for i, doc in enumerate(tokens_documentos):
        vectores = [modelo.wv.get_vector(str(tok)) for tok in doc if str(tok).strip()]
        if not vectores:
            continue
        matriz = np.asarray(vectores, dtype=np.float32)
        salida[i] = matriz.mean(axis=0) if agregacion == "media" else matriz.sum(axis=0)
    return salida

In [213]:
def generar_embeddings_transformer_revision(
    textos: Sequence[str],
    nombre_modelo: str,
    batch_size: int,
    max_length: int,
) -> tuple[np.ndarray, float]:
    """
    Genera embeddings contextuales congelados con un Transformer preentrenado.

    Aplica mean pooling sobre la última capa oculta usando la máscara de atención.
    El modelo se ejecuta automáticamente en CUDA, Apple MPS o CPU de acuerdo con
    `DISPOSITIVO_TORCH`. Los vectores finales se devuelven en CPU como float32.

    Args:
    textos (Sequence[str]): Textos que se desean codificar.
    nombre_modelo (str): Identificador Hugging Face del modelo preentrenado.
    batch_size (int): Tamaño de lote de inferencia.
    max_length (int): Longitud máxima tokenizada.

    Returns:
    tuple[np.ndarray, float]:
    Embeddings de documentos y tiempo de codificación en segundos.

    Example:
    >>> X_bert, segundos = generar_embeddings_transformer_revision(
    ...     ["texto de prueba"],
    ...     "dccuchile/bert-base-spanish-wwm-uncased",
    ...     8,
    ...     128,
    ... )
    """
    if len(textos) == 0:
        raise ValueError("La lista de textos está vacía.")

    dispositivo = DISPOSITIVO_TORCH
    tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
    modelo = AutoModel.from_pretrained(nombre_modelo).to(dispositivo)
    modelo.eval()

    vectores: list[np.ndarray] = []
    inicio_tiempo = time.perf_counter()

    with torch.no_grad():
        for inicio_lote in range(0, len(textos), int(batch_size)):
            lote = list(textos[inicio_lote:inicio_lote + int(batch_size)])
            entradas = tokenizer(
                lote,
                padding=True,
                truncation=True,
                max_length=int(max_length),
                return_tensors="pt",
            )
            entradas = {
                clave: valor.to(dispositivo)
                for clave, valor in entradas.items()
            }

            salida = modelo(**entradas).last_hidden_state
            mascara = (
                entradas["attention_mask"]
                .unsqueeze(-1)
                .expand(salida.size())
                .float()
            )
            suma = torch.sum(salida * mascara, dim=1)
            divisor = torch.clamp(mascara.sum(dim=1), min=1e-9)
            pooled = suma / divisor
            vectores.append(
                pooled.detach().cpu().numpy().astype(np.float32)
            )

    tiempo = time.perf_counter() - inicio_tiempo

    del modelo
    gc.collect()
    if dispositivo.type == "cuda":
        torch.cuda.empty_cache()

    return np.vstack(vectores), float(tiempo)

In [214]:
def muestrear_indices_estratificados(y: np.ndarray, max_n: int | None, seed: int) -> np.ndarray:
    """
    Selecciona índices de forma estratificada o conserva el conjunto completo.

    Si `max_n` es `None` o es mayor o igual que el tamaño disponible, la función
    devuelve todos los índices. Cuando se especifica un límite menor, selecciona
    una muestra estratificada que conserva la distribución binaria de clases.

    Args:
    y (np.ndarray): Etiquetas binarias.
    max_n (int | None): Máximo número de ejemplos a conservar. `None` usa todos.
    seed (int): Semilla reproducible.

    Returns:
    np.ndarray:
    Índices seleccionados en orden ascendente.

    Example:
    >>> idx = muestrear_indices_estratificados(np.array([0, 0, 1, 1]), None, 5)
    >>> len(idx)
    4
    """
    y = np.asarray(y)

    if max_n is None or int(max_n) >= len(y):
        return np.arange(len(y), dtype=int)

    if int(max_n) <= 0:
        raise ValueError("max_n debe ser None o un entero positivo.")

    indices = np.arange(len(y), dtype=int)
    seleccion, _ = train_test_split(
        indices,
        train_size=int(max_n),
        random_state=seed,
        stratify=y,
    )
    return np.sort(seleccion.astype(int))

In [215]:
def evaluar_representacion_linearsvc_revision(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray, nombre: str, tiempo_representacion_s: float, seed: int, referencia: str) -> dict[str, Any]:
    """
    Ajusta LinearSVC sobre una representación y devuelve métricas junto con el coste temporal.

    El mismo clasificador lineal se emplea para Word2Vec, FastText y Transformers con
    el fin de aislar, en lo posible, el efecto de la representación del texto.

    Args:
    X_train (np.ndarray): Características del entrenamiento.
    y_train (np.ndarray): Etiquetas del entrenamiento.
    X_test (np.ndarray): Características de prueba.
    y_test (np.ndarray): Etiquetas de referencia.
    nombre (str): Nombre del benchmark.
    tiempo_representacion_s (float): Tiempo invertido en generar características.
    seed (int): Semilla de LinearSVC.
    referencia (str): Tipo de ground truth usado en la evaluación.

    Returns:
    dict[str, Any]:
    Métricas, tiempos y dimensionalidad.

    Example:
    >>> fila = evaluar_representacion_linearsvc_revision(Xtr, ytr, Xte, yte, "Word2Vec", 1.2, 5, "Pseudoetiquetas")
    """
    inicio = time.perf_counter()
    modelo = LinearSVC(random_state=seed, max_iter=5000, dual="auto")
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    tiempo_clasificador = time.perf_counter() - inicio

    fila: dict[str, Any] = {
        "Representacion": nombre,
        "Referencia": referencia,
        "N_train": int(len(y_train)),
        "N_test": int(len(y_test)),
        "Features": int(X_train.shape[1]),
        "Tiempo_representacion_s": float(tiempo_representacion_s),
        "Tiempo_clasificador_s": float(tiempo_clasificador),
        "Tiempo_total_s": float(tiempo_representacion_s + tiempo_clasificador),
    }
    fila.update(calcular_metricas_tesis(y_test, pred))
    return fila

In [216]:
def graficar_precision_costo_revision(benchmark: pd.DataFrame, metrica: str = "F1-score") -> None:
    """
    Grafica el compromiso entre rendimiento y tiempo total para los benchmarks.

    El eje horizontal representa segundos de representación más clasificación y el
    vertical la métrica elegida. Solo se muestran filas ejecutadas correctamente.

    Args:
    benchmark (pd.DataFrame): Tabla de benchmarks de representaciones.
    metrica (str): Métrica mostrada en el eje vertical.
    *Valor por defecto: "F1-score".*

    Returns:
    None:
    Muestra la gráfica Bokeh dentro del notebook.

    Example:
    >>> graficar_precision_costo_revision(benchmark, "F1-score")
    """
    datos = benchmark.dropna(subset=["Tiempo_total_s", metrica]).copy()
    if datos.empty:
        print("No existen benchmarks ejecutados para graficar.")
        return

    source = ColumnDataSource(datos)
    p = figure(
        height=460,
        width=900,
        title=f"Compromiso rendimiento–coste ({metrica})",
        x_axis_label="Tiempo total (s)",
        y_axis_label=metrica,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    p.scatter("Tiempo_total_s", metrica, source=source, size=11)
    p.add_tools(HoverTool(tooltips=[
        ("Representación", "@Representacion"),
        ("Tiempo", "@Tiempo_total_s{0.00}s"),
        (metrica, f"@{{{metrica}}}{{0.0000}}"),
        ("Features", "@Features"),
    ]))
    show(p)

In [217]:
def ejecutar_benchmark_silver_revision(
    tweets_desarrollo: pd.DataFrame,
    holdout_silver: pd.DataFrame,
    tokens_trends: Sequence[Sequence[str]],
    dimension: int,
    semillas: Sequence[str],
    top_n: int,
    epochs: int,
    fasttext_epochs: int,
    modelos_transformer: Mapping[str, str],
    seed: int,
) -> pd.DataFrame:
    """
    Compara representaciones clásicas y contextuales contra el silver standard externo.

    El entrenamiento usa exclusivamente el corpus de desarrollo hashtag-blind.
    Las pseudoetiquetas de entrenamiento se generan con el Escenario 2 construido
    únicamente desde desarrollo. El holdout silver solo se utiliza al final para
    evaluar concordancia externa.

    Args:
    tweets_desarrollo (pd.DataFrame): Corpus de desarrollo sin hashtags y sin holdout silver.
    holdout_silver (pd.DataFrame): Holdout silver hashtag-blind con etiquetas débiles.
    tokens_trends (Sequence[Sequence[str]]): Trends sin hashtags.
    dimension (int): Dimensión de Word2Vec y FastText.
    semillas (Sequence[str]): Semillas políticas sin hashtags.
    top_n (int): Regla explícita de expansión léxica.
    epochs (int): Épocas de Word2Vec.
    fasttext_epochs (int): Épocas de FastText.
    modelos_transformer (Mapping[str, str]): Modelos Hugging Face a comparar.
    seed (int): Semilla reproducible.

    Returns:
    pd.DataFrame:
    Métricas y tiempos de representación/clasificación contra el silver standard.

    Example:
    >>> tabla = ejecutar_benchmark_silver_revision(
    ...     dev, silver, tokens_trends, 100, semillas, 100, 15, 15, MODELOS_TRANSFORMER, 5
    ... )
    """
    y_test = holdout_silver["etiqueta_silver_hashtag"].astype(int).to_numpy()
    if len(np.unique(y_test)) < 2:
        raise ValueError("El silver standard debe contener ambas clases.")

    tokens_train = preparar_tokens_dataframe(tweets_desarrollo, "tokens_sin_stopwords")
    tokens_test = preparar_tokens_dataframe(holdout_silver, "tokens_sin_stopwords")

    pipe = construir_pipeline_semantico_train(
        tokens_train,
        tokens_trends,
        int(dimension),
        semillas,
        K_CANDIDATOS,
        int(top_n),
        int(epochs),
        seed,
    )

    X_train_pipe = vectorizar_documentos(
        tokens_train,
        pipe["modelo_word2vec"],
        agregacion="suma",
    )
    y_train, _, _ = etiquetar_por_centroides(
        X_train_pipe,
        pipe["centroides_2"],
        pipe["cluster_politico"],
    )

    textos_train = tweets_desarrollo["texto_limpio"].fillna("").astype(str).tolist()
    textos_test = holdout_silver["texto_limpio"].fillna("").astype(str).tolist()
    referencia = "Silver standard de hashtags — evaluación externa hashtag-blind"

    filas: list[dict[str, Any]] = []

    inicio = time.perf_counter()
    tfidf = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=50000,
        sublinear_tf=True,
    )
    Xtr = tfidf.fit_transform(textos_train)
    Xte = tfidf.transform(textos_test)
    t_repr = time.perf_counter() - inicio
    filas.append(
        evaluar_representacion_linearsvc_revision(
            Xtr, y_train, Xte, y_test,
            "TF-IDF (1-2gram) + LinearSVC",
            t_repr, seed, referencia,
        )
    )

    inicio = time.perf_counter()
    w2v = entrenar_word2vec(
        tokens_train,
        int(dimension),
        W2V_WINDOW,
        W2V_MIN_COUNT,
        int(epochs),
        seed,
    )
    Xtr = vectorizar_documentos(tokens_train, w2v, agregacion="media")
    Xte = vectorizar_documentos(tokens_test, w2v, agregacion="media")
    t_repr = time.perf_counter() - inicio
    filas.append(
        evaluar_representacion_linearsvc_revision(
            Xtr, y_train, Xte, y_test,
            "Word2Vec + LinearSVC",
            t_repr, seed, referencia,
        )
    )

    inicio = time.perf_counter()
    ft = entrenar_fasttext_revision(
        tokens_train,
        int(dimension),
        W2V_WINDOW,
        W2V_MIN_COUNT,
        int(fasttext_epochs),
        seed,
    )
    Xtr = vectorizar_documentos_fasttext(tokens_train, ft, agregacion="media")
    Xte = vectorizar_documentos_fasttext(tokens_test, ft, agregacion="media")
    t_repr = time.perf_counter() - inicio
    filas.append(
        evaluar_representacion_linearsvc_revision(
            Xtr, y_train, Xte, y_test,
            "FastText + LinearSVC",
            t_repr, seed, referencia,
        )
    )

    for nombre_visible, nombre_modelo in modelos_transformer.items():
        try:
            Xtr, t_train = generar_embeddings_transformer_revision(
                textos_train,
                nombre_modelo,
                TRANSFORMER_BATCH_SIZE,
                TRANSFORMER_MAX_LENGTH,
            )
            Xte, t_test = generar_embeddings_transformer_revision(
                textos_test,
                nombre_modelo,
                TRANSFORMER_BATCH_SIZE,
                TRANSFORMER_MAX_LENGTH,
            )
            filas.append(
                evaluar_representacion_linearsvc_revision(
                    Xtr,
                    y_train,
                    Xte,
                    y_test,
                    f"{nombre_visible} congelado + LinearSVC",
                    t_train + t_test,
                    seed,
                    referencia,
                )
            )
            del Xtr, Xte
            gc.collect()
            if DISPOSITIVO_TORCH.type == "cuda":
                torch.cuda.empty_cache()
        except Exception as exc:
            filas.append({
                "Representacion": f"{nombre_visible} congelado + LinearSVC",
                "Referencia": referencia,
                "N_train": int(len(y_train)),
                "N_test": int(len(y_test)),
                "Features": np.nan,
                "Tiempo_representacion_s": np.nan,
                "Tiempo_clasificador_s": np.nan,
                "Tiempo_total_s": np.nan,
                "Exactitud": np.nan,
                "Precisión": np.nan,
                "Recall": np.nan,
                "F1-score": np.nan,
                "Sensibilidad": np.nan,
                "Especificidad": np.nan,
                "Precisión positiva": np.nan,
                "Tasa de falsos positivos": np.nan,
                "Valor F2": np.nan,
                "Valor F0.5": np.nan,
                "TP": np.nan,
                "TN": np.nan,
                "FP": np.nan,
                "FN": np.nan,
                "Error": str(exc),
            })

    return pd.DataFrame(filas)

In [218]:
def fine_tuning_transformer_silver(
    textos_train: Sequence[str],
    y_train: np.ndarray,
    textos_test: Sequence[str],
    y_test: np.ndarray,
    nombre_visible: str,
    nombre_modelo: str,
    epochs: int,
    batch_size: int,
    max_length: int,
    learning_rate: float,
    seed: int,
) -> pd.DataFrame:
    """
    Realiza fine-tuning binario de un Transformer y evalúa contra el silver standard.

    El modelo se entrena con pseudoetiquetas del corpus de desarrollo y se evalúa
    únicamente sobre el holdout silver hashtag-blind. La función utiliza CUDA o MPS
    cuando están disponibles. Para CPU también es técnicamente compatible, aunque el
    notebook decide no ejecutarla por defecto debido al alto coste computacional.

    Args:
    textos_train (Sequence[str]): Textos de desarrollo usados para entrenar.
    y_train (np.ndarray): Pseudoetiquetas de entrenamiento.
    textos_test (Sequence[str]): Textos del holdout silver.
    y_test (np.ndarray): Etiquetas silver del holdout.
    nombre_visible (str): Nombre mostrado en las tablas.
    nombre_modelo (str): Identificador del modelo Hugging Face.
    epochs (int): Número de épocas de fine-tuning.
    batch_size (int): Tamaño de lote.
    max_length (int): Longitud máxima tokenizada.
    learning_rate (float): Tasa de aprendizaje.
    seed (int): Semilla reproducible.

    Returns:
    pd.DataFrame:
    Una fila con métricas, tiempo total y dispositivo utilizado.

    Example:
    >>> tabla = fine_tuning_transformer_silver(
    ...     textos_train, y_train, textos_test, y_test,
    ...     "BETO", "dccuchile/bert-base-spanish-wwm-uncased",
    ...     2, 16, 128, 2e-5, 5
    ... )
    """
    if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
        raise ValueError("Train y test deben contener ambas clases.")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
    modelo = AutoModelForSequenceClassification.from_pretrained(
        nombre_modelo,
        num_labels=2,
    ).to(DISPOSITIVO_TORCH)

    optimizador = torch.optim.AdamW(
        modelo.parameters(),
        lr=float(learning_rate),
    )

    y_train = np.asarray(y_train).astype(np.int64)
    y_test = np.asarray(y_test).astype(np.int64)
    rng = np.random.default_rng(seed)

    inicio = time.perf_counter()
    modelo.train()

    for _ in range(int(epochs)):
        orden = rng.permutation(len(textos_train))
        for inicio_lote in range(0, len(orden), int(batch_size)):
            idx = orden[inicio_lote:inicio_lote + int(batch_size)]
            lote_textos = [str(textos_train[i]) for i in idx]
            lote_y = torch.tensor(
                y_train[idx],
                dtype=torch.long,
                device=DISPOSITIVO_TORCH,
            )

            entradas = tokenizer(
                lote_textos,
                padding=True,
                truncation=True,
                max_length=int(max_length),
                return_tensors="pt",
            )
            entradas = {
                k: v.to(DISPOSITIVO_TORCH)
                for k, v in entradas.items()
            }

            optimizador.zero_grad(set_to_none=True)
            salida = modelo(**entradas, labels=lote_y)
            salida.loss.backward()
            optimizador.step()

    modelo.eval()
    predicciones: list[int] = []

    with torch.no_grad():
        for inicio_lote in range(0, len(textos_test), int(batch_size)):
            lote = list(textos_test[inicio_lote:inicio_lote + int(batch_size)])
            entradas = tokenizer(
                lote,
                padding=True,
                truncation=True,
                max_length=int(max_length),
                return_tensors="pt",
            )
            entradas = {
                k: v.to(DISPOSITIVO_TORCH)
                for k, v in entradas.items()
            }
            logits = modelo(**entradas).logits
            predicciones.extend(
                torch.argmax(logits, dim=1).detach().cpu().numpy().astype(int).tolist()
            )

    tiempo = time.perf_counter() - inicio
    metricas = calcular_metricas_tesis(
        y_test,
        np.asarray(predicciones, dtype=int),
    )

    fila: dict[str, Any] = {
        "Representacion": f"{nombre_visible} fine-tuned",
        "Referencia": "Silver standard de hashtags — evaluación externa hashtag-blind",
        "Dispositivo": str(DISPOSITIVO_TORCH),
        "Epochs": int(epochs),
        "Batch_size": int(batch_size),
        "N_train": int(len(y_train)),
        "N_test": int(len(y_test)),
        "Tiempo_total_s": float(tiempo),
    }
    fila.update(metricas)

    del modelo
    gc.collect()
    if DISPOSITIVO_TORCH.type == "cuda":
        torch.cuda.empty_cache()

    return pd.DataFrame([fila])

In [219]:
benchmark_silver = ejecutar_benchmark_silver_revision(
    tweets_desarrollo=tweets_desarrollo_ciego_hashtags,
    holdout_silver=silver_hashtags_ciego,
    tokens_trends=preparar_tokens_dataframe(trends_ciegos_hashtags, "tokens_sin_stopwords"),
    dimension=DIMENSION_BENCHMARK,
    semillas=semillas_politicas_sin_hashtags,
    top_n=EXPANSION_TOP_N_FINAL,
    epochs=W2V_EPOCHS,
    fasttext_epochs=FASTTEXT_EPOCHS,
    modelos_transformer=MODELOS_TRANSFORMER,
    seed=RANDOM_STATE,
)

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/310 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/486k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [220]:
mostrar_tabla_bokeh(
    benchmark_silver,
    "Benchmark contemporáneo contra silver standard: rendimiento y coste",
    ancho=1700,
    alto=400,
    max_filas=20,
)

In [221]:
graficar_precision_costo_revision(benchmark_silver, "F1-score")

In [222]:
fine_tuning_silver = pd.DataFrame()

In [223]:
if EJECUTAR_FINE_TUNING_TRANSFORMER:
    tokens_dev_ciego = preparar_tokens_dataframe(
        tweets_desarrollo_ciego_hashtags,
        "tokens_sin_stopwords",
    )
    tokens_trends_ciegos = preparar_tokens_dataframe(
        trends_ciegos_hashtags,
        "tokens_sin_stopwords",
    )

    pipe_ft = construir_pipeline_semantico_train(
        tokens_dev_ciego,
        tokens_trends_ciegos,
        DIMENSION_BENCHMARK,
        semillas_politicas_sin_hashtags,
        K_CANDIDATOS,
        EXPANSION_TOP_N_FINAL,
        W2V_EPOCHS,
        RANDOM_STATE,
    )

    X_dev_ft = vectorizar_documentos(
        tokens_dev_ciego,
        pipe_ft["modelo_word2vec"],
        agregacion="suma",
    )
    y_dev_ft, _, _ = etiquetar_por_centroides(
        X_dev_ft,
        pipe_ft["centroides_2"],
        pipe_ft["cluster_politico"],
    )

    textos_dev = tweets_desarrollo_ciego_hashtags["texto_limpio"].fillna("").astype(str).tolist()
    textos_silver = silver_hashtags_ciego["texto_limpio"].fillna("").astype(str).tolist()
    y_silver = silver_hashtags_ciego["etiqueta_silver_hashtag"].astype(int).to_numpy()

    filas_ft: list[pd.DataFrame] = []
    for nombre_visible, nombre_modelo in MODELOS_TRANSFORMER.items():
        print(f"Fine-tuning: {nombre_visible} en {DISPOSITIVO_TORCH}")
        filas_ft.append(
            fine_tuning_transformer_silver(
                textos_train=textos_dev,
                y_train=y_dev_ft,
                textos_test=textos_silver,
                y_test=y_silver,
                nombre_visible=nombre_visible,
                nombre_modelo=nombre_modelo,
                epochs=TRANSFORMER_FINE_TUNE_EPOCHS,
                batch_size=TRANSFORMER_BATCH_SIZE,
                max_length=TRANSFORMER_MAX_LENGTH,
                learning_rate=TRANSFORMER_FINE_TUNE_LR,
                seed=RANDOM_STATE,
            )
        )

    fine_tuning_silver = pd.concat(filas_ft, ignore_index=True)
    mostrar_tabla_bokeh(
        fine_tuning_silver,
        "Fine-tuning Transformer contra silver standard",
        ancho=1600,
        alto=300,
        max_filas=10,
    )
else:
    print(
        "Fine-tuning Transformer omitido automáticamente porque no se detectó GPU/MPS. "
        "Los benchmarks congelados sí se ejecutan sobre el corpus completo."
    )

Fine-tuning Transformer omitido automáticamente porque no se detectó GPU/MPS. Los benchmarks congelados sí se ejecutan sobre el corpus completo.


# 10. Validación humana independiente

In [224]:
def evaluar_validacion_manual_revision(tweets_desarrollo: pd.DataFrame, anotaciones: pd.DataFrame, tokens_trends: Sequence[Sequence[str]], dimension: int, semillas: Sequence[str], top_n: int, epochs: int, seed: int) -> pd.DataFrame:
    """
    Evalúa pseudoetiquetas y clasificadores contra un ground truth humano independiente.

    Word2Vec, clustering y expansión se construyen con `tweets_desarrollo`, que excluye
    previamente el holdout manual. Luego las filas con `etiqueta_consenso` se vectorizan
    sin reentrenar el modelo. Se comparan tanto las pseudoetiquetas directas como las
    predicciones de los clasificadores con las etiquetas humanas.

    Args:
    tweets_desarrollo (pd.DataFrame): Corpus de desarrollo sin el holdout manual.
    anotaciones (pd.DataFrame): Plantilla completada con `etiqueta_consenso`.
    tokens_trends (Sequence[Sequence[str]]): Trends tokenizados.
    dimension (int): Dimensión Word2Vec.
    semillas (Sequence[str]): Semillas políticas.
    top_n (int): Número de términos de expansión.
    epochs (int): Épocas Word2Vec.
    seed (int): Semilla reproducible.

    Returns:
    pd.DataFrame:
    Métricas frente a ground truth humano para pseudoetiquetas y clasificadores.

    Example:
    >>> manual = evaluar_validacion_manual_revision(tweets_desarrollo, anotaciones, tokens_trends, 100, semillas_politicas, 100, 10, 5)
    """
    validas = anotaciones.dropna(subset=["etiqueta_consenso"]).copy()
    if len(validas) < 20:
        raise ValueError("Se requieren al menos 20 anotaciones con consenso para ejecutar la validación manual.")

    y_humano = validas["etiqueta_consenso"].astype(int).to_numpy()
    if len(np.unique(y_humano)) < 2:
        raise ValueError("Las anotaciones con consenso deben contener ambas clases 0 y 1.")

    tokens_train = preparar_tokens_dataframe(tweets_desarrollo, "tokens_sin_stopwords")
    tokens_manual = [convertir_tokens(x) for x in validas["texto_limpio"].fillna("").astype(str).tolist()]

    pipe = construir_pipeline_semantico_train(
        tokens_train,
        tokens_trends,
        dimension,
        semillas,
        K_CANDIDATOS,
        top_n,
        epochs,
        seed,
    )
    X_train = vectorizar_documentos(tokens_train, pipe["modelo_word2vec"], agregacion="suma")
    X_manual = vectorizar_documentos(tokens_manual, pipe["modelo_word2vec"], agregacion="suma")

    y1_train, _, _ = etiquetar_por_centroides(X_train, pipe["centroides_1"], pipe["cluster_politico"])
    y2_train, _, _ = etiquetar_por_centroides(X_train, pipe["centroides_2"], pipe["cluster_politico"])
    y1_manual, _, _ = etiquetar_por_centroides(X_manual, pipe["centroides_1"], pipe["cluster_politico"])
    y2_manual, _, _ = etiquetar_por_centroides(X_manual, pipe["centroides_2"], pipe["cluster_politico"])

    filas: list[dict[str, Any]] = []
    for escenario, y_train, y_pseudo in [
        ("Escenario 1", y1_train, y1_manual),
        ("Escenario 2", y2_train, y2_manual),
    ]:
        fila_pseudo: dict[str, Any] = {
            "Dimension": int(dimension),
            "Escenario": escenario,
            "Modelo": "Pseudoetiquetado directo",
            "Referencia": "Ground truth humano",
        }
        fila_pseudo.update(calcular_metricas_tesis(y_humano, y_pseudo))
        filas.append(fila_pseudo)

        for nombre, modelo in crear_modelos_clasificacion(KNN_VECINOS, seed).items():
            modelo.fit(X_train, y_train)
            pred = modelo.predict(X_manual)
            fila: dict[str, Any] = {
                "Dimension": int(dimension),
                "Escenario": escenario,
                "Modelo": nombre,
                "Referencia": "Ground truth humano",
            }
            fila.update(calcular_metricas_tesis(y_humano, pred))
            filas.append(fila)

    return pd.DataFrame(filas)

In [225]:
def ejecutar_benchmark_manual_revision(tweets_desarrollo: pd.DataFrame, anotaciones: pd.DataFrame, max_train: int | None, max_test: int | None, dimension: int, semillas: Sequence[str], tokens_trends: Sequence[Sequence[str]], top_n: int, epochs: int, fasttext_epochs: int, modelos_transformer: Mapping[str, str], ejecutar_transformers: bool, seed: int) -> pd.DataFrame:
    """
    Compara representaciones modernas usando etiquetas humanas como referencia de test.

    El entrenamiento continúa usando pseudoetiquetas del Escenario 2 obtenidas
    exclusivamente desde el corpus de desarrollo. El conjunto manual nunca participa
    en Word2Vec, clustering, expansión ni ajuste de clasificadores. Todos los métodos
    se evalúan contra `etiqueta_consenso`, por lo que este benchmark sí estima desempeño
    respecto a un ground truth humano independiente.

    Args:
    tweets_desarrollo (pd.DataFrame): Corpus de desarrollo sin holdout manual.
    anotaciones (pd.DataFrame): Anotaciones con etiqueta de consenso.
    max_train (int | None): Máximo de ejemplos pseudoetiquetados; `None` usa todos.
    max_test (int | None): Máximo de ejemplos humanos de test; `None` usa todos.
    dimension (int): Dimensión para Word2Vec y FastText.
    semillas (Sequence[str]): Semillas políticas.
    tokens_trends (Sequence[Sequence[str]]): Trends tokenizados.
    top_n (int): Regla de expansión léxica.
    epochs (int): Épocas Word2Vec.
    fasttext_epochs (int): Épocas FastText.
    modelos_transformer (Mapping[str, str]): Modelos Hugging Face a evaluar.
    ejecutar_transformers (bool): Indica si se ejecutan Transformers.
    seed (int): Semilla reproducible.

    Returns:
    pd.DataFrame:
    Métricas y tiempos de TF-IDF, Word2Vec, FastText y Transformers frente a humanos.

    Example:
    >>> bench_manual = ejecutar_benchmark_manual_revision(tweets_desarrollo, anotaciones, 5000, 600, 100, semillas_politicas, tokens_trends, 100, 10, 10, MODELOS_TRANSFORMER, True, 5)
    """
    validas = anotaciones.dropna(subset=["etiqueta_consenso"]).copy()
    if len(validas) < 20:
        raise ValueError("Se necesitan al menos 20 etiquetas humanas de consenso.")
    y_test_full = validas["etiqueta_consenso"].astype(int).to_numpy()
    if len(np.unique(y_test_full)) < 2:
        raise ValueError("El ground truth humano debe contener las clases 0 y 1.")

    tokens_dev = preparar_tokens_dataframe(tweets_desarrollo, "tokens_sin_stopwords")
    pipe = construir_pipeline_semantico_train(
        tokens_dev,
        tokens_trends,
        int(dimension),
        semillas,
        K_CANDIDATOS,
        int(top_n),
        int(epochs),
        seed,
    )
    X_dev_w2v = vectorizar_documentos(tokens_dev, pipe["modelo_word2vec"], agregacion="suma")
    y_train_full, _, _ = etiquetar_por_centroides(
        X_dev_w2v,
        pipe["centroides_2"],
        pipe["cluster_politico"],
    )

    idx_train = muestrear_indices_estratificados(y_train_full, max_train, seed)
    idx_test = muestrear_indices_estratificados(y_test_full, max_test, seed)

    train_df = tweets_desarrollo.iloc[idx_train].reset_index(drop=True)
    test_df = validas.iloc[idx_test].reset_index(drop=True)
    y_train = y_train_full[idx_train]
    y_test = y_test_full[idx_test]

    textos_train = train_df["texto_limpio"].fillna("").astype(str).tolist()
    textos_test = test_df["texto_limpio"].fillna("").astype(str).tolist()
    tokens_train = preparar_tokens_dataframe(train_df, "tokens_sin_stopwords")
    tokens_test = [convertir_tokens(x) for x in test_df["texto_limpio"].fillna("").astype(str).tolist()]
    referencia = "Ground truth humano independiente"

    filas: list[dict[str, Any]] = []

    inicio = time.perf_counter()
    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)
    Xtr_tfidf = tfidf.fit_transform(textos_train)
    Xte_tfidf = tfidf.transform(textos_test)
    tiempo = time.perf_counter() - inicio
    filas.append(evaluar_representacion_linearsvc_revision(
        Xtr_tfidf, y_train, Xte_tfidf, y_test, "TF-IDF (1-2gram) + LinearSVC", tiempo, seed, referencia
    ))

    inicio = time.perf_counter()
    w2v = entrenar_word2vec(tokens_train, int(dimension), W2V_WINDOW, W2V_MIN_COUNT, int(epochs), seed)
    Xtr_w2v = vectorizar_documentos(tokens_train, w2v, agregacion="media")
    Xte_w2v = vectorizar_documentos(tokens_test, w2v, agregacion="media")
    tiempo = time.perf_counter() - inicio
    filas.append(evaluar_representacion_linearsvc_revision(
        Xtr_w2v, y_train, Xte_w2v, y_test, "Word2Vec + LinearSVC", tiempo, seed, referencia
    ))

    inicio = time.perf_counter()
    ft = entrenar_fasttext_revision(tokens_train, int(dimension), W2V_WINDOW, W2V_MIN_COUNT, int(fasttext_epochs), seed)
    Xtr_ft = vectorizar_documentos_fasttext(tokens_train, ft, agregacion="media")
    Xte_ft = vectorizar_documentos_fasttext(tokens_test, ft, agregacion="media")
    tiempo = time.perf_counter() - inicio
    filas.append(evaluar_representacion_linearsvc_revision(
        Xtr_ft, y_train, Xte_ft, y_test, "FastText + LinearSVC", tiempo, seed, referencia
    ))

    if ejecutar_transformers:
        for nombre_visible, nombre_modelo in modelos_transformer.items():
            try:
                Xtr, t_train = generar_embeddings_transformer_revision(
                    textos_train, nombre_modelo, TRANSFORMER_BATCH_SIZE, TRANSFORMER_MAX_LENGTH
                )
                Xte, t_test = generar_embeddings_transformer_revision(
                    textos_test, nombre_modelo, TRANSFORMER_BATCH_SIZE, TRANSFORMER_MAX_LENGTH
                )
                filas.append(evaluar_representacion_linearsvc_revision(
                    Xtr, y_train, Xte, y_test, nombre_visible, t_train + t_test, seed, referencia
                ))
            except Exception as exc:
                filas.append({
                    "Representacion": nombre_visible,
                    "Referencia": referencia,
                    "N_train": int(len(y_train)),
                    "N_test": int(len(y_test)),
                    "Features": np.nan,
                    "Tiempo_representacion_s": np.nan,
                    "Tiempo_clasificador_s": np.nan,
                    "Tiempo_total_s": np.nan,
                    "Exactitud": np.nan,
                    "Precisión": np.nan,
                    "Recall": np.nan,
                    "F1-score": np.nan,
                    "Sensibilidad": np.nan,
                    "Especificidad": np.nan,
                    "Precisión positiva": np.nan,
                    "Tasa de falsos positivos": np.nan,
                    "Valor F2": np.nan,
                    "Valor F0.5": np.nan,
                    "TP": np.nan,
                    "TN": np.nan,
                    "FP": np.nan,
                    "FN": np.nan,
                    "Error": str(exc),
                })

    return pd.DataFrame(filas)

In [287]:
def graficar_matrices_confusion_ground_truth_revision(
    validacion_manual: pd.DataFrame
) -> None:
    """
    Grafica las matrices de confusión de Scenario 1 y Scenario 2
    contra el ground truth humano independiente utilizando Bokeh.

    Los valores TP, TN, FP y FN se recuperan directamente de
    `validacion_manual`.

    Las dos matrices utilizan la misma escala de color y tienen
    habilitadas las herramientas interactivas de Bokeh.

    Args:
        validacion_manual (pd.DataFrame):
            Tabla generada por
            `evaluar_validacion_manual_revision`.

    Returns:
        None
    """
    requeridas = {
        "Escenario",
        "Modelo",
        "TP",
        "TN",
        "FP",
        "FN",
    }

    faltantes = requeridas.difference(
        validacion_manual.columns
    )

    if faltantes:
        raise ValueError(
            f"Faltan columnas requeridas: {sorted(faltantes)}. "
            f"Columnas disponibles: "
            f"{validacion_manual.columns.tolist()}"
        )

    datos = validacion_manual[
        validacion_manual["Modelo"]
        == "Pseudoetiquetado directo"
    ].copy()

    matrices = {}

    for escenario in [
        "Escenario 1",
        "Escenario 2",
    ]:

        fila = datos[
            datos["Escenario"] == escenario
        ]

        if fila.empty:
            raise ValueError(
                f"No se encontró el pseudoetiquetado directo "
                f"para {escenario}."
            )

        fila = fila.iloc[0]

        matrices[escenario] = {
            "TN": int(fila["TN"]),
            "FP": int(fila["FP"]),
            "FN": int(fila["FN"]),
            "TP": int(fila["TP"]),
        }

    vmax = max(
        valor
        for matriz in matrices.values()
        for valor in matriz.values()
    )

    color_mapper = LinearColorMapper(
        palette=Blues9,
        low=0,
        high=vmax,
    )

    def crear_matriz_bokeh(
        escenario: str,
        valores: dict,
    ):

        source = ColumnDataSource(
            data={
                "x": [
                    "Non-political",
                    "Political",
                    "Non-political",
                    "Political",
                ],
                "y": [
                    "Non-political",
                    "Non-political",
                    "Political",
                    "Political",
                ],
                "valor": [
                    valores["TN"],
                    valores["FP"],
                    valores["FN"],
                    valores["TP"],
                ],
                "tipo": [
                    "True Negative",
                    "False Positive",
                    "False Negative",
                    "True Positive",
                ],
                "abreviatura": [
                    "TN",
                    "FP",
                    "FN",
                    "TP",
                ],
            }
        )

        titulo = escenario.replace(
            "Escenario",
            "Scenario"
        )

        pan_tool = PanTool()
        wheel_zoom_tool = WheelZoomTool()
        box_zoom_tool = BoxZoomTool()
        reset_tool = ResetTool()
        save_tool = SaveTool()
        help_tool = HelpTool()

        p = figure(
            width=500,
            height=430,
            title=titulo,
            x_range=[
                "Non-political",
                "Political",
            ],
            y_range=[
                "Political",
                "Non-political",
            ],
            x_axis_label="Predicted label",
            y_axis_label="Human ground truth",
            tools=[
                pan_tool,
                wheel_zoom_tool,
                box_zoom_tool,
                reset_tool,
                save_tool,
                help_tool,
            ],
            toolbar_location="right",
        )

        p.toolbar.active_scroll = wheel_zoom_tool

        renderer = p.rect(
            x="x",
            y="y",
            width=1,
            height=1,
            source=source,
            fill_color={
                "field": "valor",
                "transform": color_mapper,
            },
            line_color="white",
            line_width=2,
        )

        etiquetas = LabelSet(
            x="x",
            y="y",
            text="valor",
            source=source,
            text_align="center",
            text_baseline="middle",
            text_font_size="15px",
            text_font_style="bold",
        )

        p.add_layout(etiquetas)

        hover = HoverTool(
            renderers=[renderer],
            tooltips=[
                ("Type", "@tipo"),
                ("Code", "@abreviatura"),
                ("Human label", "@y"),
                ("Predicted label", "@x"),
                ("Count", "@valor"),
            ],
        )

        p.add_tools(hover)

        p.xgrid.grid_line_color = None
        p.ygrid.grid_line_color = None

        return p

    p1 = crear_matriz_bokeh(
        "Escenario 1",
        matrices["Escenario 1"],
    )

    p2 = crear_matriz_bokeh(
        "Escenario 2",
        matrices["Escenario 2"],
    )

    color_bar = ColorBar(
        color_mapper=color_mapper,
        ticker=BasicTicker(),
        label_standoff=8,
        title="Count",
    )

    p2.add_layout(
        color_bar,
        "right"
    )

    show(
        row(
            p1,
            p2,
        )
    )

In [226]:
anotaciones_manual = cargar_anotaciones_manuales(RUTA_ANOTACION_MANUAL)
acuerdo_anotadores = calcular_acuerdo_anotadores(anotaciones_manual)

In [227]:
mostrar_tabla_bokeh(
    acuerdo_anotadores,
    "Acuerdo entre anotadores — ground truth humano",
    alto=180,
)

In [228]:
validacion_manual = evaluar_validacion_manual_revision(
    tweets_desarrollo=tweets_desarrollo,
    anotaciones=anotaciones_manual,
    tokens_trends=tokens_trends,
    dimension=DIMENSION_BENCHMARK,
    semillas=semillas_politicas,
    top_n=EXPANSION_TOP_N_FINAL,
    epochs=W2V_EPOCHS,
    seed=RANDOM_STATE,
)

In [229]:
benchmark_manual = ejecutar_benchmark_manual_revision(
    tweets_desarrollo=tweets_desarrollo,
    anotaciones=anotaciones_manual,
    max_train=None,
    max_test=None,
    dimension=DIMENSION_BENCHMARK,
    semillas=semillas_politicas,
    tokens_trends=tokens_trends,
    top_n=EXPANSION_TOP_N_FINAL,
    epochs=W2V_EPOCHS,
    fasttext_epochs=FASTTEXT_EPOCHS,
    modelos_transformer=MODELOS_TRANSFORMER,
    ejecutar_transformers=True,
    seed=RANDOM_STATE,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [230]:
mostrar_tabla_bokeh(
    validacion_manual,
    "Validación frente a ground truth humano independiente",
    ancho=1550,
    alto=430,
    max_filas=30,
)

In [288]:
graficar_matrices_confusion_ground_truth_revision(
    validacion_manual
)

In [231]:
mostrar_tabla_bokeh(
    benchmark_manual,
    "Benchmark contemporáneo frente a ground truth humano",
    ancho=1700,
    alto=400,
    max_filas=20,
)

In [232]:
graficar_precision_costo_revision(benchmark_manual, "F1-score")

In [322]:
def graficar_tradeoff_representaciones_revision(
    benchmark_manual: pd.DataFrame
) -> None:
    """
    Grafica el trade-off observado entre tiempo total de ejecución
    y Balanced Accuracy contra el ground truth humano independiente.

    La función utiliza exclusivamente resultados ya calculados en
    `benchmark_manual`.

    Eje X:
        Tiempo total de ejecución en segundos, en escala logarítmica.

    Eje Y:
        Balanced Accuracy, calculada como:

            (Sensibilidad + Especificidad) / 2

    Representaciones:
        TF-IDF, Word2Vec, FastText, BETO y XLM-RoBERTa.

    No se identifica ni resalta ningún modelo como superior.

    Args:
        benchmark_manual (pd.DataFrame):
            Resultados del benchmark contra el ground truth humano.

    Returns:
        None
    """
    columnas_requeridas = {
        "Representacion",
        "Tiempo_total_s",
        "Sensibilidad",
        "Especificidad",
    }

    faltantes = columnas_requeridas.difference(
        benchmark_manual.columns
    )

    if faltantes:
        raise ValueError(
            f"Faltan columnas requeridas: {sorted(faltantes)}. "
            f"Columnas disponibles: "
            f"{benchmark_manual.columns.tolist()}"
        )

    datos = benchmark_manual.copy()
    datos["Balanced_Accuracy"] = (
        pd.to_numeric(
            datos["Sensibilidad"],
            errors="coerce",
        )
        +
        pd.to_numeric(
            datos["Especificidad"],
            errors="coerce",
        )
    ) / 2.0

    datos["Runtime_seconds"] = pd.to_numeric(
        datos["Tiempo_total_s"],
        errors="coerce",
    )

    def normalizar_representacion(valor):
        texto = str(valor).lower()

        if "tf-idf" in texto or "tfidf" in texto:
            return "TF-IDF"

        if "word2vec" in texto:
            return "Word2Vec"

        if "fasttext" in texto:
            return "FastText"

        if "beto" in texto:
            return "BETO"

        if (
            "xlm-r" in texto
            or "xlm_r" in texto
            or "xlm" in texto
            or "roberta" in texto
        ):
            return "XLM-RoBERTa"

        return None

    datos["Representation_plot"] = (
        datos["Representacion"]
        .apply(normalizar_representacion)
    )

    datos = datos[
        datos["Representation_plot"].notna()
    ].copy()

    datos = datos.dropna(
        subset=[
            "Runtime_seconds",
            "Balanced_Accuracy",
        ]
    ).copy()

    datos = datos[
        datos["Runtime_seconds"] > 0
    ].copy()

    representaciones_esperadas = {
        "TF-IDF",
        "Word2Vec",
        "FastText",
        "BETO",
        "XLM-RoBERTa",
    }

    representaciones_encontradas = set(
        datos["Representation_plot"]
        .unique()
        .tolist()
    )

    faltantes_modelos = (
        representaciones_esperadas
        - representaciones_encontradas
    )

    if faltantes_modelos:
        raise ValueError(
            "Faltan representaciones para construir la figura: "
            f"{sorted(faltantes_modelos)}. "
            f"Encontradas: "
            f"{sorted(representaciones_encontradas)}"
        )

    datos = (
        datos
        .drop_duplicates(
            subset="Representation_plot",
            keep="first",
        )
        .copy()
    )

    orden_representaciones = [
        "TF-IDF",
        "Word2Vec",
        "FastText",
        "BETO",
        "XLM-RoBERTa",
    ]

    orden = {
        nombre: i
        for i, nombre
        in enumerate(orden_representaciones)
    }

    datos["_orden"] = (
        datos["Representation_plot"]
        .map(orden)
    )

    datos = (
        datos
        .sort_values("_orden")
        .drop(columns="_orden")
        .reset_index(drop=True)
    )

    tabla_figura = datos[
        [
            "Representation_plot",
            "Runtime_seconds",
            "Sensibilidad",
            "Especificidad",
            "Balanced_Accuracy",
        ]
    ].copy()

    print(
        "Datos utilizados para la figura "
        "performance-cost:"
    )

    display(tabla_figura)

    source = ColumnDataSource(
        data={
            "representation":
                datos["Representation_plot"].tolist(),

            "runtime":
                datos["Runtime_seconds"].tolist(),

            "balanced":
                datos["Balanced_Accuracy"].tolist(),

            "sensitivity":
                datos["Sensibilidad"].tolist(),

            "specificity":
                datos["Especificidad"].tolist(),
        }
    )

    pan_tool = PanTool()
    wheel_zoom_tool = WheelZoomTool()
    box_zoom_tool = BoxZoomTool()
    reset_tool = ResetTool()
    save_tool = SaveTool()
    help_tool = HelpTool()

    p = figure(
        width=850,
        height=520,
        title=(
            "Balanced Accuracy vs Computational Time"
        ),
        x_axis_type="log",
        x_axis_label=(
            "Total execution time (s, log scale)"
        ),
        y_axis_label="Balanced Accuracy",
        tools=[
            pan_tool,
            wheel_zoom_tool,
            box_zoom_tool,
            reset_tool,
            save_tool,
            help_tool,
        ],
        toolbar_location="right",
    )
    p.toolbar.active_scroll = wheel_zoom_tool

    renderer = p.scatter(
        x="runtime",
        y="balanced",
        source=source,
        size=14,
        alpha=0.85,
    )

    labels = LabelSet(
        x="runtime",
        y="balanced",
        text="representation",
        source=source,
        x_offset=8,
        y_offset=6,
        text_font_size="10pt",
    )

    p.add_layout(labels)

    hover = HoverTool(
        renderers=[renderer],
        tooltips=[
            (
                "Representation",
                "@representation",
            ),
            (
                "Total execution time",
                "@runtime{0.00} s",
            ),
            (
                "Balanced Accuracy",
                "@balanced{0.0000}",
            ),
            (
                "Sensitivity",
                "@sensitivity{0.0000}",
            ),
            (
                "Specificity",
                "@specificity{0.0000}",
            ),
        ],
    )

    p.add_tools(hover)
    p.grid.grid_line_alpha = 0.25
    p.outline_line_alpha = 0.5
    show(p)

In [323]:
graficar_tradeoff_representaciones_revision(
    benchmark_manual
)

Datos utilizados para la figura performance-cost:


,Representation_plot,Runtime_seconds,Sensibilidad,Especificidad,Balanced_Accuracy
0,TF-IDF,7.093269,0.993007,0.302548,0.647777
1,Word2Vec,117.527423,0.996503,0.334395,0.665449
2,FastText,95.348073,1.000000,0.315287,0.657643
3,BETO,"7,278.705757",0.996503,0.191083,0.593793
4,XLM-RoBERTa,"8,045.138230",1.000000,0.194268,0.597134


# 11. Análisis descriptivo posterior a la evaluación

In [233]:
def normalizar_provincia(valor: Any) -> str:
    """
    Convierte una ubicación textual de Twitter en una provincia ecuatoriana aproximada.

    La ubicación de Twitter es texto libre. La función normaliza tildes y signos y
    aplica un diccionario de provincias, capitales y ciudades frecuentes para obtener
    una categoría provincial reproducible. Si no existe coincidencia devuelve
    `No identificada`; si el valor está vacío devuelve `Sin ubicación`.

    Args:
    valor (Any): Texto original del campo de ubicación.

    Returns:
    str:
    Nombre normalizado de la provincia o una categoría de ausencia/no identificación.

    Example:
    >>> normalizar_provincia("Guayaquil, Ecuador")
    'Guayas'
    """
    if pd.isna(valor) or not str(valor).strip():
        return "Sin ubicación"

    texto = str(valor).lower().strip()
    reemplazos = str.maketrans("áéíóúüñ", "aeiouun")
    texto = texto.translate(reemplazos)
    texto = re.sub(r"[^a-z0-9#@\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()

    alias = {
        "Azuay": ["azuay", "cuenca", "gualaceo"],
        "Bolívar": ["bolivar", "guaranda"],
        "Cañar": ["canar", "azogues", "la troncal"],
        "Carchi": ["carchi", "tulcan"],
        "Chimborazo": ["chimborazo", "riobamba"],
        "Cotopaxi": ["cotopaxi", "latacunga"],
        "El Oro": ["el oro", "machala", "pasaje"],
        "Esmeraldas": ["esmeraldas", "atacames"],
        "Galápagos": ["galapagos", "puerto ayora", "san cristobal"],
        "Guayas": ["guayas", "guayaquil", "duran", "samborondon", "daule", "milagro"],
        "Imbabura": ["imbabura", "ibarra", "otavalo"],
        "Loja": ["loja", "catamayo"],
        "Los Ríos": ["los rios", "babahoyo", "quevedo", "vinces"],
        "Manabí": ["manabi", "manta", "portoviejo", "chone", "jipijapa"],
        "Morona Santiago": ["morona santiago", "macas"],
        "Napo": ["napo", "tena"],
        "Orellana": ["orellana", "coca", "francisco de orellana"],
        "Pastaza": ["pastaza", "puyo"],
        "Pichincha": ["pichincha", "quito", "cayambe", "sangolqui", "ruminahui"],
        "Santa Elena": ["santa elena", "salinas", "la libertad"],
        "Santo Domingo de los Tsáchilas": ["santo domingo", "tsachilas"],
        "Sucumbíos": ["sucumbios", "lago agrio", "nueva loja"],
        "Tungurahua": ["tungurahua", "ambato", "banos"],
        "Zamora Chinchipe": ["zamora chinchipe", "zamora"],
    }

    for provincia, variantes in alias.items():
        for variante in variantes:
            if re.search(rf"\b{re.escape(variante)}\b", texto):
                return provincia
    return "No identificada"

In [234]:
def contar_por_provincia(tweets: pd.DataFrame, etiquetas: np.ndarray, columna_ubicacion: str = "location") -> pd.DataFrame:
    """
    Cuenta los tweets políticos por provincia para un conjunto de etiquetas binarias.

    Mantiene únicamente los tweets etiquetados con 1, normaliza la ubicación textual
    y genera la frecuencia provincial utilizada para comparar los dos escenarios.

    Args:
    tweets (pd.DataFrame): Dataset de tweets alineado con las etiquetas.
    etiquetas (np.ndarray): Vector binario donde 1 representa contenido político.
    columna_ubicacion (str): Nombre del campo de ubicación.
    *Valor por defecto: "location".*
    
    Returns:
    pd.DataFrame:
    Tabla con provincia y número de tweets políticos.

    Example:
    >>> contar_por_provincia(tweets, etiquetas).head()
    """
    if len(tweets) != len(etiquetas):
        raise ValueError("La cantidad de tweets y etiquetas debe coincidir.")
    df = tweets[[columna_ubicacion]].copy()
    df["label"] = etiquetas.astype(int)
    df = df[df["label"] == 1].copy()
    df["Provincia"] = df[columna_ubicacion].map(normalizar_provincia)
    return df["Provincia"].value_counts().rename_axis("Provincia").reset_index(name="Tweets_politicos")

In [251]:
def graficar_provincias_escenarios(
    provincias_1: pd.DataFrame,
    provincias_2: pd.DataFrame,
    dimension: int,
    top_n: int = 15
) -> None:
    """
    Compara la cantidad de tweets políticos por provincia entre los dos escenarios.

    Une los conteos provinciales del modelo inicial y del modelo propuesto, selecciona
    las provincias con mayor volumen combinado y las muestra en barras agrupadas.

    Args:
    provincias_1 (pd.DataFrame): Conteos provinciales del Escenario 1.
    provincias_2 (pd.DataFrame): Conteos provinciales del Escenario 2.
    dimension (int): Dimensión de Word2Vec evaluada.
    top_n (int): Número máximo de provincias mostradas.
    *Valor por defecto: 15.*

    Returns:
    None:
    Muestra un gráfico Bokeh comparativo.

    Example:
    >>> graficar_provincias_escenarios(p1, p2, 200)
    """

    a = provincias_1.rename(
        columns={"Tweets_politicos": "Escenario_1"}
    )

    b = provincias_2.rename(
        columns={"Tweets_politicos": "Escenario_2"}
    )

    df = a.merge(
        b,
        on="Provincia",
        how="outer"
    ).fillna(0)

    df["Total"] = (
        df["Escenario_1"] +
        df["Escenario_2"]
    )

    df = (
        df.sort_values("Total", ascending=False)
        .head(top_n)
    )

    factores = df["Provincia"].astype(str).tolist()

    source = ColumnDataSource(df)

    p = figure(
        x_range=factores,
        width=1100,
        height=450,
        title=f"Tweets políticos por provincia — dimensión {dimension}"
    )

    p.vbar(
        x=dodge("Provincia", -0.18, range=p.x_range),
        top="Escenario_1",
        width=0.34,
        source=source,
        color="#1f77b4",
        legend_label="Escenario 1"
    )

    p.vbar(
        x=dodge("Provincia", 0.18, range=p.x_range),
        top="Escenario_2",
        width=0.34,
        source=source,
        color="#ff7f0e",
        legend_label="Escenario 2"
    )

    p.xaxis.major_label_orientation = math.pi / 3
    p.yaxis.axis_label = "Tweets políticos"

    p.legend.location = "top_right"

    p.add_tools(
        HoverTool(
            tooltips=[
                ("Provincia", "@Provincia"),
                ("Escenario 1", "@Escenario_1{0,0}"),
                ("Escenario 2", "@Escenario_2{0,0}")
            ]
        )
    )

    show(p)

In [236]:
tokens_desarrollo_total = preparar_tokens_dataframe(
    tweets_desarrollo,
    "tokens_sin_stopwords",
)

In [237]:
pipeline_descriptivo = construir_pipeline_semantico_train(
    tokens_desarrollo_total,
    tokens_trends,
    DIMENSION_BENCHMARK,
    semillas_politicas,
    K_CANDIDATOS,
    EXPANSION_TOP_N_FINAL,
    W2V_EPOCHS,
    RANDOM_STATE,
)

In [238]:
X_descriptivo = vectorizar_documentos(
    tokens_desarrollo_total,
    pipeline_descriptivo["modelo_word2vec"],
    agregacion="suma",
)

In [239]:
y_desc_e1, _, _ = etiquetar_por_centroides(
    X_descriptivo,
    pipeline_descriptivo["centroides_1"],
    pipeline_descriptivo["cluster_politico"],
)

In [240]:
y_desc_e2, _, _ = etiquetar_por_centroides(
    X_descriptivo,
    pipeline_descriptivo["centroides_2"],
    pipeline_descriptivo["cluster_politico"],
)

In [241]:
cobertura_descriptiva = pd.DataFrame([
    {
        "Escenario": "Escenario 1",
        "N": len(y_desc_e1),
        "Politicos": int(y_desc_e1.sum()),
        "No_politicos": int((y_desc_e1 == 0).sum()),
        "Politicos_pct": float(y_desc_e1.mean() * 100.0),
    },
    {
        "Escenario": "Escenario 2",
        "N": len(y_desc_e2),
        "Politicos": int(y_desc_e2.sum()),
        "No_politicos": int((y_desc_e2 == 0).sum()),
        "Politicos_pct": float(y_desc_e2.mean() * 100.0),
    },
])

In [242]:
mostrar_tabla_bokeh(
    cobertura_descriptiva,
    "Cobertura descriptiva del contenido político",
    alto=190,
)

In [243]:
provincias_e1 = contar_por_provincia(tweets_desarrollo, y_desc_e1)

In [244]:
provincias_e2 = contar_por_provincia(tweets_desarrollo, y_desc_e2)

In [252]:
graficar_provincias_escenarios(
    provincias_e1,
    provincias_e2,
    DIMENSION_BENCHMARK,
    top_n=15,
)

# 12. Guardar resultados finales

In [246]:
def guardar_resultados_revision(tablas: Mapping[str, pd.DataFrame], output_dir: Path) -> Path:
    """
    Guarda las tablas de respuesta experimental a revisores en un único Excel.

    Cada DataFrame se exporta en una hoja con nombre seguro y, adicionalmente, se crea
    un CSV independiente para facilitar análisis posterior y trazabilidad.

    Args:
    tablas (Mapping[str, pd.DataFrame]): Nombre lógico y tabla a exportar.
    output_dir (Path): Directorio de salida.

    Returns:
    Path:
    Ruta al libro Excel consolidado.

    Example:
    >>> ruta = guardar_resultados_revision({"auditoria": tabla}, Path("outputs_bias_aware/revisores"))
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    ruta_excel = output_dir / "resultados_experimentos_revisores.xlsx"

    with pd.ExcelWriter(ruta_excel, engine="openpyxl") as writer:
        for nombre, tabla in tablas.items():
            if tabla is None or not isinstance(tabla, pd.DataFrame):
                continue
            hoja = re.sub(r"[^A-Za-z0-9_]", "_", str(nombre))[:31] or "Hoja"
            tabla.to_excel(writer, sheet_name=hoja, index=False)
            tabla.to_csv(output_dir / f"{hoja}.csv", index=False, encoding="utf-8-sig")

    return ruta_excel

In [316]:
tablas_finales = {
    "auditoria_librerias": AUDITORIA_LIBRERIAS,
    "limpieza": AUDITORIA_LIMPIEZA,
    "auditoria_hardware": AUDITORIA_HARDWARE,
    "auditoria_dataset": AUDITORIA_DATASET["resumen"],
    "auditoria_eventos": AUDITORIA_DATASET["eventos"],
    "auditoria_faltantes": AUDITORIA_DATASET["faltantes"],

    "silver_balance": resumen_silver,

    "holdout_resumen": resumen_holdout,
    "holdout_metricas": metricas_holdout,

    "k_objetivo": pipe_principal["tabla_k"],

    "lexico_topn": pipe_principal["palabras_anadidas"],

    "supplementary_top100": tabla_top100_suplementaria,
    "supplementary_top100_summary": verificacion_top100,

    "clustering_comp": comparacion_clustering,
    "sensibilidad_topn": sensibilidad_top_n,

    "cv_resultados": cv_resultados,
    "cv_resumen": cv_resumen,
    "wilcoxon_holm": wilcoxon_escenarios,
    
    "appendix_b_cv_complete": tabla_appendix_b_cv,
    "appendix_b_cv_table": tabla_appendix_b_cv_table,
    
    "silver_metricas": metricas_silver,
    "silver_balance_eval": metricas_silver_balance,
    "silver_evento": metricas_silver_evento,
    "silver_mcnemar": mcnemar_silver,

    "hash_humano_metricas": METRICAS_AUDITORIA_HUMANA_HASHTAG,
    "hash_humano_mcnemar": MCNEMAR_AUDITORIA_HUMANA_HASHTAG,
    "hash_humano_detalle": DETALLE_AUDITORIA_HUMANA_HASHTAG,

    "benchmark_silver": benchmark_silver,
    "finetuning_silver": fine_tuning_silver,

    "acuerdo_anotadores": acuerdo_anotadores,
    "validacion_manual": validacion_manual,
    "benchmark_manual": benchmark_manual,

    "cobertura_descriptiva": cobertura_descriptiva,
    "provincias_e1": provincias_e1,
    "provincias_e2": provincias_e2,
}

In [317]:
ruta_resultados_finales = guardar_resultados_revision(
    tablas_finales,
    REVISION_OUTPUT_DIR,
)

In [318]:
print(f"Resultados consolidados: {ruta_resultados_finales}")

Resultados consolidados: C:\Users\LABIA\Documents\Tesis cleo EPN\Codigo\outputs_bias_aware\revisores_final\resultados_experimentos_revisores.xlsx


In [319]:
print(f"Auditoría de versiones: {RUTA_REQUIREMENTS_RUNTIME}")

Auditoría de versiones: C:\Users\LABIA\Documents\Tesis cleo EPN\Codigo\outputs_bias_aware\revisores_final\requirements_runtime.txt
